# Notebook for decoding and analyzing GeoHEIF file from local file and HTTP server

## Headers

### Standard C++ headers

In [ ]:
#include <iostream>
#include <fstream>
#include <vector>
#include <string>
#include <stdexcept>
#include <curl/curl.h>
#include <cstdint>
#include <sstream>
#include <cstring>
#include <iomanip>
#include <algorithm>
#include <type_traits>
#include <map>           // Already used in meta parser
#include <set>           // Already used in meta parser
#include <optional>      // Already used in meta parser
#include <functional>    // ADD THIS LINE - needed for std::function

### Headers for non-standard libraries

##### Environment detection

In [ ]:
#ifndef HEIF_NOTEBOOK_GLOBALS_H
#define HEIF_NOTEBOOK_GLOBALS_H

#include <iostream>
#include <filesystem>
#include <cstdlib>
#include <string>
#include <vector>

namespace fs = std::filesystem;

// Detect operating system
#if defined(_WIN32) || defined(_WIN64)
    #define HEIF_OS_WINDOWS 1
    #define HEIF_LIB_PREFIX ""
    #define HEIF_LIB_EXT ".dll"
#elif defined(__APPLE__) || defined(__MACH__)
    #define HEIF_OS_MACOS 1
    #define HEIF_LIB_PREFIX "lib"
    #define HEIF_LIB_EXT ".dylib"
#elif defined(__linux__)
    #define HEIF_OS_LINUX 1
    #define HEIF_LIB_PREFIX "lib"
    #define HEIF_LIB_EXT ".so"
#else
    #define HEIF_OS_UNKNOWN 1
    #define HEIF_LIB_PREFIX "lib"
    #define HEIF_LIB_EXT ".so"
#endif

// Global namespace for persistent environment configuration
namespace heif_notebook {

// Environment configuration structure
struct EnvironmentConfig {
    fs::path conda_prefix;
    fs::path include_path;
    fs::path lib_path;
    std::string os_name;
    std::string lib_extension;
    std::string lib_prefix;
    bool is_conda_env;
    
    EnvironmentConfig() : is_conda_env(false) {
        // Detect OS
#ifdef HEIF_OS_WINDOWS
        os_name = "Windows";
#elif defined(HEIF_OS_MACOS)
        os_name = "macOS";
#elif defined(HEIF_OS_LINUX)
        os_name = "Linux";
#else
        os_name = "Unknown";
#endif
        
        lib_extension = HEIF_LIB_EXT;
        lib_prefix = HEIF_LIB_PREFIX;
        
        // Get CONDA_PREFIX
        const char* conda_prefix_env = std::getenv("CONDA_PREFIX");
        if (conda_prefix_env) {
            conda_prefix = fs::path(conda_prefix_env);
            is_conda_env = true;
            include_path = conda_prefix / "include";
            lib_path = conda_prefix / "lib";
        } else {
            // Fallback to system paths
            is_conda_env = false;
#ifdef HEIF_OS_WINDOWS
            include_path = "C:/Program Files/include";
            lib_path = "C:/Program Files/lib";
#else
            include_path = "/usr/local/include";
            lib_path = "/usr/local/lib";
#endif
        }
    }
    
    void print_info() const {
        std::cout << "\n=== Environment Information ===" << std::endl;
        std::cout << "Operating System: " << os_name << std::endl;
        std::cout << "Library Prefix: " << lib_prefix << std::endl;
        std::cout << "Library Extension: " << lib_extension << std::endl;
        std::cout << "Conda Environment: " << (is_conda_env ? "Yes" : "No") << std::endl;
        
        if (is_conda_env) {
            std::cout << "CONDA_PREFIX: " << conda_prefix << std::endl;
            std::cout << "Include Path: " << include_path << std::endl;
            std::cout << "Library Path: " << lib_path << std::endl;
        }
        std::cout << "\n✓ Environment detected" << std::endl;
    }
};

// Global configuration instance - singleton pattern
inline EnvironmentConfig& get_env_config() {
    static EnvironmentConfig instance;
    return instance;
}

} // namespace heif_notebook

// Initialize and print environment info
heif_notebook::get_env_config().print_info();

#endif // HEIF_NOTEBOOK_GLOBALS_H

##### Library load utility

In [ ]:
#ifndef HEIF_NOTEBOOK_DLOPEN_H
#define HEIF_NOTEBOOK_DLOPEN_H

#include <iostream>
#include <filesystem>
#include <string>
#include <vector>
#include <map>

// Platform-specific includes for dlopen
#ifdef HEIF_OS_WINDOWS
    #include <windows.h>
    #define RTLD_NOW 0
    #define RTLD_GLOBAL 0
    typedef HMODULE dl_handle_t;
    
    void* dlopen(const char* filename, int flags) {
        return (void*)LoadLibraryA(filename);
    }
    
    const char* dlerror() {
        static char error_msg[256];
        FormatMessageA(FORMAT_MESSAGE_FROM_SYSTEM, NULL, GetLastError(),
                      MAKELANGID(LANG_NEUTRAL, SUBLANG_DEFAULT),
                      error_msg, sizeof(error_msg), NULL);
        return error_msg;
    }
    
    int dlclose(void* handle) {
        return FreeLibrary((HMODULE)handle) ? 0 : -1;
    }
#else
    #include <dlfcn.h>
    typedef void* dl_handle_t;
#endif

namespace heif_notebook {

// Library handle manager
class LibraryLoader {
private:
    std::map<std::string, dl_handle_t> loaded_libraries;
    
public:
    // Construct full library path and find existing library
    std::string get_library_path(const std::string& lib_name) {
        namespace fs = std::filesystem;
        
        // Get environment config
        auto& env = get_env_config();
        
        // Try with different version suffix patterns
        std::vector<std::string> try_patterns = {
            env.lib_prefix + lib_name + env.lib_extension,
        };
        
        // Add versioned patterns for Unix-like systems
#ifndef HEIF_OS_WINDOWS
        // Try .dylib.X or .so.X patterns
        for (int ver = 1; ver <= 10; ver++) {
            try_patterns.push_back(env.lib_prefix + lib_name + env.lib_extension + "." + std::to_string(ver));
        }
        // Try .X.Y.Z patterns
        for (int major = 1; major <= 5; major++) {
            for (int minor = 0; minor <= 9; minor++) {
                try_patterns.push_back(env.lib_prefix + lib_name + env.lib_extension + "." + 
                                      std::to_string(major) + "." + std::to_string(minor));
            }
        }
        // Try .X.dylib patterns (alternate macOS versioning)
        for (int ver = 1; ver <= 10; ver++) {
            try_patterns.push_back(env.lib_prefix + lib_name + "." + std::to_string(ver) + env.lib_extension);
        }
#endif
        
        // Search for existing library file
        for (const auto& pattern : try_patterns) {
            fs::path lib_path = env.lib_path / pattern;
            if (fs::exists(lib_path)) {
                return lib_path.string();
            }
        }
        
        // Return default path (for error reporting)
        fs::path default_path = env.lib_path / (env.lib_prefix + lib_name + env.lib_extension);
        return default_path.string();
    }
    
    // Check if library exists
    bool library_exists(const std::string& lib_name) {
        namespace fs = std::filesystem;
        std::string lib_path = get_library_path(lib_name);
        return fs::exists(lib_path);
    }
    
    // Load a library
    bool load_library(const std::string& lib_name, bool required = true) {
        // Check if already loaded
        if (loaded_libraries.find(lib_name) != loaded_libraries.end()) {
            std::cout << "  ℹ " << lib_name << " already loaded" << std::endl;
            return true;
        }
        
        std::cout << "  Loading: " << lib_name << std::endl;
        
        std::string lib_path = get_library_path(lib_name);
        std::cout << "    Path: " << lib_path << std::endl;
        
        namespace fs = std::filesystem;
        if (!fs::exists(lib_path)) {
            if (required) {
                std::cout << "    ✗ Not found (Required)" << std::endl;
            } else {
                std::cout << "    ⚠ Not found (Optional)" << std::endl;
            }
            return false;
        }
        
        dl_handle_t handle = (dl_handle_t)dlopen(lib_path.c_str(), RTLD_NOW | RTLD_GLOBAL);
        
        if (!handle) {
            const char* error = dlerror();
            std::cout << "    ✗ Failed to load: " << (error ? error : "Unknown error") << std::endl;
            return false;
        }
        
        loaded_libraries[lib_name] = handle;
        std::cout << "    ✓ Loaded successfully" << std::endl;
        return true;
    }
    
    // Load multiple libraries in order
    void load_libraries(const std::vector<std::pair<std::string, bool>>& libraries) {
        std::cout << "\n=== Loading Libraries ===" << std::endl;
        for (const auto& [lib_name, required] : libraries) {
            load_library(lib_name, required);
        }
        std::cout << "\n✓ Library loading complete" << std::endl;
    }
    
    // Get count of loaded libraries
    size_t loaded_count() const {
        return loaded_libraries.size();
    }
    
    // Unload all libraries
    ~LibraryLoader() {
        for (auto& [name, handle] : loaded_libraries) {
            if (handle) {
                dlclose(handle);
            }
        }
    }
};

// Global library loader instance - singleton pattern
inline LibraryLoader& get_library_loader() {
    static LibraryLoader instance;
    return instance;
}

} // namespace heif_notebook

std::cout << "\n✓ Library loader initialized" << std::endl;

#endif // HEIF_NOTEBOOK_DLOPEN_H

##### Load libraries

In [ ]:
#ifndef HEIF_NOTEBOOK_LOAD_ALL_H
#define HEIF_NOTEBOOK_LOAD_ALL_H

// Define library loading order (dependencies first)
std::vector<std::pair<std::string, bool>> libraries_to_load = {
    // Core compression libraries (no dependencies)
    {"z", true},              // zlib - required by many libraries
    {"bz2", false},           // bzip2 - optional
    {"lzma", false},          // LZMA - optional
    {"zstd", false},          // Zstandard - optional
    {"brotlicommon", false},  // Brotli common - optional
    {"brotlidec", false},     // Brotli decoder - optional
    {"brotlienc", false},     // Brotli encoder - optional
    
    // Image format libraries
    {"jpeg", false},          // JPEG - used by TIFF/GDAL
    {"png", false},           // PNG - used by GDAL
    {"png16", false},         // PNG 1.6 - alternative name
    {"webp", false},          // WebP - optional
    
    // TIFF library (depends on zlib, jpeg, etc.)
    {"tiff", true},           // libtiff - required
    
    // NOTE: We skip loading libgeotiff separately since GDAL includes GeoTIFF support
    // Loading both causes header conflicts
    
    // Geospatial libraries
    {"proj", false},          // PROJ - used by GDAL
    {"sqlite3", false},       // SQLite - used by GDAL
    {"curl", false},          // cURL - used by GDAL
    {"expat", false},         // Expat XML parser - used by GDAL
    {"xml2", false},          // libxml2 - used by GDAL
    
    // GDAL (includes GeoTIFF support, depends on many of the above)
    {"gdal", true},           // GDAL - required
    
    // Video codec libraries (for HEIF)
    {"x265", false},          // x265 HEVC encoder - optional
    {"de265", false},         // de265 HEVC decoder - optional
    {"aom", false},           // AOM AV1 codec - optional
    {"dav1d", false},         // dav1d AV1 decoder - optional
    {"rav1e", false},         // rav1e AV1 encoder - optional
    {"SvtAv1Enc", false},     // SVT-AV1 encoder - optional
    {"SvtAv1Dec", false},     // SVT-AV1 decoder - optional
    
    // HEIF library (depends on codecs)
    {"heif", true},           // libheif - required
};

// Load all libraries
heif_notebook::get_library_loader().load_libraries(libraries_to_load);

// Report summary
std::cout << "\nSuccessfully loaded " << heif_notebook::get_library_loader().loaded_count() 
          << " libraries" << std::endl;

#endif // HEIF_NOTEBOOK_LOAD_ALL_H

##### All library headers

In [ ]:
#ifndef HEIF_NOTEBOOK_ALL_HEADERS_H
#define HEIF_NOTEBOOK_ALL_HEADERS_H

// ============================================================================
// Load all library headers in correct order (dependencies first)
// NOTE: Do NOT include libgeotiff headers - GDAL includes GeoTIFF support
// ============================================================================

// 1. zlib (no dependencies)
#ifndef HEIF_ZLIB_LOADED
#define HEIF_ZLIB_LOADED
#include <zlib.h>
#endif

// 2. libtiff (depends on zlib)
#ifndef HEIF_TIFF_LOADED
#define HEIF_TIFF_LOADED
#include <tiffio.h>
#include <tiffio.hxx>
#endif

// 3. GDAL (depends on TIFF, includes GeoTIFF support internally)
#ifndef HEIF_GDAL_LOADED
#define HEIF_GDAL_LOADED
#ifdef __has_include
#if __has_include(<gdal.h>)
    #include <gdal.h>
    #include <gdal_priv.h>
    #include <cpl_conv.h>
    #include <cpl_string.h>
    #include <ogr_spatialref.h>
    #define HEIF_GDAL_INCLUDE_PREFIX ""
#elif __has_include(<gdal/gdal.h>)
    #include <gdal/gdal.h>
    #include <gdal/gdal_priv.h>
    #include <gdal/cpl_conv.h>
    #include <gdal/cpl_string.h>
    #include <gdal/ogr_spatialref.h>
    #define HEIF_GDAL_INCLUDE_PREFIX "gdal/"
#else
    #error "GDAL headers not found. Check installation."
#endif
#else
    #include <gdal.h>
    #include <gdal_priv.h>
    #include <cpl_conv.h>
    #include <cpl_string.h>
    #include <ogr_spatialref.h>
    #define HEIF_GDAL_INCLUDE_PREFIX ""
#endif
#endif // HEIF_GDAL_LOADED

// 4. libheif (may depend on codecs)
#ifndef HEIF_LIBHEIF_LOADED
#define HEIF_LIBHEIF_LOADED

// Enable experimental features
#ifndef HEIF_ENABLE_EXPERIMENTAL_FEATURES
#define HEIF_ENABLE_EXPERIMENTAL_FEATURES 1
#endif

#ifndef WITH_UNCOMPRESSED_CODEC
#define WITH_UNCOMPRESSED_CODEC 1
#endif

#ifdef __has_include
#if __has_include(<libheif/heif.h>)
    #include <libheif/heif.h>
    #include <libheif/heif_properties.h>
    #include <libheif/heif_items.h>
    #include <libheif/heif_experimental.h>
    #include <libheif/heif_uncompressed.h>
    #define HEIF_INCLUDE_PREFIX "libheif/"
#elif __has_include(<heif.h>)
    #include <heif.h>
    #include <heif_properties.h>
    #include <heif_items.h>
    #include <heif_experimental.h>
    #include <heif_uncompressed.h>
    #define HEIF_INCLUDE_PREFIX ""
#else
    #error "libheif headers not found. Check installation."
#endif
#else
    #include <libheif/heif.h>
    #include <libheif/heif_properties.h>
    #include <libheif/heif_items.h>
    #include <libheif/heif_experimental.h>
    #include <libheif/heif_uncompressed.h>
    #define HEIF_INCLUDE_PREFIX "libheif/"
#endif
#endif // HEIF_LIBHEIF_LOADED

// 5. Optional compression libraries
#ifndef HEIF_OPTIONAL_LIBS_LOADED
#define HEIF_OPTIONAL_LIBS_LOADED

#ifdef __has_include
#if __has_include(<brotli/encode.h>)
    #ifndef HEIF_HAVE_BROTLI
    #define HEIF_HAVE_BROTLI 1
    #include <brotli/encode.h>
    #include <brotli/decode.h>
    #endif
#endif

#if __has_include(<zstd.h>)
    #ifndef HEIF_HAVE_ZSTD
    #define HEIF_HAVE_ZSTD 1
    #include <zstd.h>
    #endif
#endif
#endif

#endif // HEIF_OPTIONAL_LIBS_LOADED

// ============================================================================
// Check and report loaded headers
// ============================================================================

std::cout << "\n=== Library Headers Loaded ===" << std::endl;
std::cout << "✓ zlib" << std::endl;
std::cout << "✓ libtiff" << std::endl;
std::cout << "✓ GDAL (include prefix: " << HEIF_GDAL_INCLUDE_PREFIX << ")" << std::endl;
std::cout << "  (GeoTIFF support included via GDAL)" << std::endl;
std::cout << "✓ libheif (include prefix: " << HEIF_INCLUDE_PREFIX << ")" << std::endl;

#ifdef HEIF_HAVE_BROTLI
std::cout << "✓ Brotli" << std::endl;
#else
std::cout << "⚠ Brotli (not available)" << std::endl;
#endif

#ifdef HEIF_HAVE_ZSTD
std::cout << "✓ ZSTD" << std::endl;
#else
std::cout << "⚠ ZSTD (not available)" << std::endl;
#endif

std::cout << "\n✓ All library headers loaded successfully" << std::endl;

#endif // HEIF_NOTEBOOK_ALL_HEADERS_H

## Utility Functions

### General Helper Uitlities

In [ ]:

#ifndef UTILS_COMMON_DEFINED
#define UTILS_COMMON_DEFINED

// Generic safe min function that handles different integral types
template<typename T1, typename T2>
constexpr auto safe_min(T1 a, T2 b) {
    using CommonType = std::common_type_t<T1, T2>;
    return std::min(static_cast<CommonType>(a), static_cast<CommonType>(b));
}

// Generic safe max function that handles different integral types
template<typename T1, typename T2>
constexpr auto safe_max(T1 a, T2 b) {
    using CommonType = std::common_type_t<T1, T2>;
    return std::max(static_cast<CommonType>(a), static_cast<CommonType>(b));
}

#endif // UTILS_COMMON_DEFINED

### File Range Reader Function

A utility function for reading data with offset and byte count from file systems

In [ ]:
#ifndef UTILS_READ_FILE_RANGE_DEFINED
#define UTILS_READ_FILE_RANGE_DEFINED

// Read a range of bytes from a file
// offset: starting position in bytes
// count: number of bytes to read
// Returns a vector containing the requested bytes
std::vector<uint8_t> readFileRange(const std::string& filepath, size_t offset, size_t count) {
    std::ifstream file(filepath, std::ios::binary);
    
    if (!file.is_open()) {
        throw std::runtime_error("Failed to open file: " + filepath);
    }
    
    // Seek to the offset
    file.seekg(offset, std::ios::beg);
    if (!file.good()) {
        throw std::runtime_error("Failed to seek to offset " + std::to_string(offset));
    }
    
    // Read the data
    std::vector<uint8_t> buffer(count);
    file.read(reinterpret_cast<char*>(buffer.data()), count);
    
    // Check how many bytes were actually read
    std::streamsize bytesRead = file.gcount();
    if (bytesRead < static_cast<std::streamsize>(count)) {
        buffer.resize(bytesRead);
        std::cerr << "Warning: Only read " << bytesRead << " bytes out of " << count << " requested\n";
    }
    
    file.close();
    return buffer;
}

#endif // UTILS_READ_FILE_RANGE_DEFINED

### HTTP Range Reader Function

A utility function for reading data with offset and byte count from HTTP.

In [ ]:
#ifndef READ_HTTP_RANGE_DEFINED
#define READ_HTTP_RANGE_DEFINED

// Callback function for libcurl to write data
static size_t writeCallback(void* contents, size_t size, size_t nmemb, void* userp) {
    size_t totalSize = size * nmemb;
    std::vector<uint8_t>* buffer = static_cast<std::vector<uint8_t>*>(userp);
    
    uint8_t* data = static_cast<uint8_t*>(contents);
    buffer->insert(buffer->end(), data, data + totalSize);
    
    return totalSize;
}

// Read a range of bytes from an HTTP/HTTPS URL using Range GET
// url: the HTTP/HTTPS URL
// offset: starting position in bytes
// count: number of bytes to read
// Returns a vector containing the requested bytes
std::vector<uint8_t> readHttpRange(const std::string& url, size_t offset, size_t count) {
    std::vector<uint8_t> buffer;
    
    CURL* curl = curl_easy_init();
    if (!curl) {
        throw std::runtime_error("Failed to initialize CURL");
    }
    
    // Set up the Range header
    std::string rangeHeader = "Range: bytes=" + std::to_string(offset) + "-" + 
                              std::to_string(offset + count - 1);
    
    struct curl_slist* headers = nullptr;
    headers = curl_slist_append(headers, rangeHeader.c_str());
    
    // Configure CURL
    curl_easy_setopt(curl, CURLOPT_URL, url.c_str());
    curl_easy_setopt(curl, CURLOPT_HTTPHEADER, headers);
    curl_easy_setopt(curl, CURLOPT_WRITEFUNCTION, writeCallback);
    curl_easy_setopt(curl, CURLOPT_WRITEDATA, &buffer);
    curl_easy_setopt(curl, CURLOPT_FOLLOWLOCATION, 1L);
    curl_easy_setopt(curl, CURLOPT_SSL_VERIFYPEER, 1L);
    curl_easy_setopt(curl, CURLOPT_SSL_VERIFYHOST, 2L);
    
    // Perform the request
    CURLcode res = curl_easy_perform(curl);
    
    // Check response code
    long responseCode = 0;
    curl_easy_getinfo(curl, CURLINFO_RESPONSE_CODE, &responseCode);
    
    // Cleanup
    curl_slist_free_all(headers);
    curl_easy_cleanup(curl);
    
    if (res != CURLE_OK) {
        throw std::runtime_error(std::string("CURL request failed: ") + curl_easy_strerror(res));
    }
    
    // 206 Partial Content is the expected response for range requests
    // 200 OK might be returned if server doesn't support ranges (returns full content)
    if (responseCode != 206 && responseCode != 200) {
        throw std::runtime_error("HTTP request failed with code: " + std::to_string(responseCode));
    }
    
    if (responseCode == 200) {
        std::cerr << "Warning: Server returned 200 OK instead of 206 Partial Content. "
                  << "Range requests may not be supported.\n";
    }
    
    return buffer;
}

#endif // READ_HTTP_RANGE_DEFINED

### Test/Example Usage

In [ ]:
#ifndef TEST_RANGE_READERS_DEFINED
#define TEST_RANGE_READERS_DEFINED

enum class ReadSourceType {
    FILE,
    HTTP_RANGE
};

void testRangeReaders(const std::string& source, ReadSourceType type, 
                      size_t offset = 0, size_t byteCount = 100) {
    try {
        std::cout << "=== Testing Range Readers ===\n\n";
        
        std::vector<uint8_t> data;
        
        if (type == ReadSourceType::FILE) {
            std::cout << "Source: FILE\n";
            std::cout << "Path: " << source << "\n";
            std::cout << "Offset: " << offset << " bytes\n";
            std::cout << "Byte Count: " << byteCount << " bytes\n\n";
            
            data = readFileRange(source, offset, byteCount);
            std::cout << "✓ Successfully read " << data.size() << " bytes from file\n";
            
        } else if (type == ReadSourceType::HTTP_RANGE) {
            std::cout << "Source: HTTP/HTTPS\n";
            std::cout << "URL: " << source << "\n";
            std::cout << "Offset: " << offset << " bytes\n";
            std::cout << "Byte Count: " << byteCount << " bytes\n\n";
            
            data = readHttpRange(source, offset, byteCount);
            std::cout << "✓ Successfully read " << data.size() << " bytes from HTTP\n";
        }
        
        // Display data preview
        if (!data.empty()) {
            size_t previewSize = safe_min(size_t(64), data.size());
            
            std::cout << "\n=== Data Preview ===\n";
            std::cout << "First " << previewSize << " bytes (hex):\n";
            
            for (size_t i = 0; i < previewSize; ++i) {
                if (i > 0 && i % 16 == 0) {
                    std::cout << "\n";
                }
                printf("%02x ", data[i]);
            }
            std::cout << "\n";
            
            // Try to show as ASCII (printable characters only)
            std::cout << "\nFirst " << previewSize << " bytes (ASCII, '.' for non-printable):\n";
            for (size_t i = 0; i < previewSize; ++i) {
                if (i > 0 && i % 64 == 0) {
                    std::cout << "\n";
                }
                char c = static_cast<char>(data[i]);
                if (c >= 32 && c <= 126) {
                    std::cout << c;
                } else {
                    std::cout << '.';
                }
            }
            std::cout << "\n";
        }
        
        std::cout << "\n✓ Test completed successfully!\n";
        
    } catch (const std::exception& e) {
        std::cerr << "\n✗ Error: " << e.what() << "\n";
    }
}

// Convenience overload that takes string for type
void testRangeReaders(const std::string& source, const std::string& typeStr, 
                      size_t offset = 0, size_t byteCount = 100) {
    ReadSourceType type;
    if (typeStr == "file" || typeStr == "FILE") {
        type = ReadSourceType::FILE;
    } else if (typeStr == "http" || typeStr == "HTTP" || 
               typeStr == "http-range" || typeStr == "HTTP-RANGE" ||
               typeStr == "https" || typeStr == "HTTPS") {
        type = ReadSourceType::HTTP_RANGE;
    } else {
        std::cerr << "Error: Invalid type '" << typeStr << "'. Use 'file' or 'http-range'.\n";
        return;
    }
    testRangeReaders(source, type, offset, byteCount);
}

// Test both file and HTTP with the same parameters (useful for comparing)
void testRangeReadersBoth(const std::string& filePath, const std::string& httpUrl,
                          size_t offset = 0, size_t byteCount = 100) {
    std::cout << "╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Testing FILE Source                               ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n";
    testRangeReaders(filePath, ReadSourceType::FILE, offset, byteCount);
    
    std::cout << "\n\n";
    std::cout << "╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Testing HTTP Source                               ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n";
    testRangeReaders(httpUrl, ReadSourceType::HTTP_RANGE, offset, byteCount);
}

#endif // TEST_RANGE_READERS_DEFINED


### Usage Example

In [ ]:
// Example 1: Test file reading with enum
// testRangeReaders("benchmark_output/heif_NONE_t256_p3.heic", ReadSourceType::FILE, 0, 100);

// Example 2: Test file reading with string type
// testRangeReaders("sample.heif", "file", 0, 100);

// Example 3: Test HTTP reading with enum
// testRangeReaders("https://example.com/sample.heif", ReadSourceType::HTTP_RANGE, 0, 100);

// Example 4: Test HTTP reading with string type
// testRangeReaders("https://example.com/sample.heif", "http", 0, 100);

// Example 5: Read from specific offset
// testRangeReaders("sample.heif", "file", 1024, 256);

// Example 6: Read larger chunk
// testRangeReaders("https://example.com/sample.heif", "https", 0, 1024);

// Example 7: Test both file and HTTP with same parameters
// testRangeReadersBoth("local_sample.heif", "https://example.com/sample.heif", 0, 256);

## Header Parser

### HEIF ftyp box

#### Parser for ftyp box

 HEIF ftyp Box Parser and Meta Box Locator

In [ ]:
#ifndef HEIF_FTYP_PARSER_DEFINED
#define HEIF_FTYP_PARSER_DEFINED


// Structure to represent a parsed box header
struct BoxHeader {
    uint64_t size;        // Total size of the box including header
    uint32_t type;        // Four-character code as uint32_t
    size_t headerSize;    // Size of the header itself (8 or 16 bytes)
    
    std::string typeString() const {
        char str[5] = {0};
        str[0] = (type >> 24) & 0xFF;
        str[1] = (type >> 16) & 0xFF;
        str[2] = (type >> 8) & 0xFF;
        str[3] = type & 0xFF;
        return std::string(str);
    }
};

// Structure to hold ftyp box information
struct FtypBox {
    uint32_t majorBrand;
    uint32_t minorVersion;
    std::vector<uint32_t> compatibleBrands;
    uint64_t boxSize;     // Total size of ftyp box
    
    std::string majorBrandString() const {
        char str[5] = {0};
        str[0] = (majorBrand >> 24) & 0xFF;
        str[1] = (majorBrand >> 16) & 0xFF;
        str[2] = (majorBrand >> 8) & 0xFF;
        str[3] = majorBrand & 0xFF;
        return std::string(str);
    }
    
    std::vector<std::string> compatibleBrandsStrings() const {
        std::vector<std::string> brands;
        for (uint32_t brand : compatibleBrands) {
            char str[5] = {0};
            str[0] = (brand >> 24) & 0xFF;
            str[1] = (brand >> 16) & 0xFF;
            str[2] = (brand >> 8) & 0xFF;
            str[3] = brand & 0xFF;
            brands.push_back(std::string(str));
        }
        return brands;
    }
};

// Structure to represent meta box location
struct MetaBoxLocation {
    size_t offset;        // Offset from start of file
    uint64_t size;        // Size of meta box (0 if not yet known)
    bool found;           // Whether meta box was found in the data
};

// Helper function to read big-endian uint32_t
inline uint32_t readBE32(const uint8_t* data) {
    return (static_cast<uint32_t>(data[0]) << 24) |
           (static_cast<uint32_t>(data[1]) << 16) |
           (static_cast<uint32_t>(data[2]) << 8) |
           static_cast<uint32_t>(data[3]);
}

// Helper function to read big-endian uint64_t
inline uint64_t readBE64(const uint8_t* data) {
    return (static_cast<uint64_t>(data[0]) << 56) |
           (static_cast<uint64_t>(data[1]) << 48) |
           (static_cast<uint64_t>(data[2]) << 40) |
           (static_cast<uint64_t>(data[3]) << 32) |
           (static_cast<uint64_t>(data[4]) << 24) |
           (static_cast<uint64_t>(data[5]) << 16) |
           (static_cast<uint64_t>(data[6]) << 8) |
           static_cast<uint64_t>(data[7]);
}

// Parse box header from data
BoxHeader parseBoxHeader(const uint8_t* data, size_t dataSize, size_t offset = 0) {
    if (offset + 8 > dataSize) {
        throw std::runtime_error("Not enough data to parse box header");
    }
    
    BoxHeader header;
    uint32_t size32 = readBE32(data + offset);
    header.type = readBE32(data + offset + 4);
    
    if (size32 == 1) {
        // 64-bit size
        if (offset + 16 > dataSize) {
            throw std::runtime_error("Not enough data to parse extended box header");
        }
        header.size = readBE64(data + offset + 8);
        header.headerSize = 16;
    } else if (size32 == 0) {
        // Box extends to end of file
        header.size = 0; // Special case
        header.headerSize = 8;
    } else {
        header.size = size32;
        header.headerSize = 8;
    }
    
    return header;
}

// Parse ftyp box and locate meta box
// Returns: pair of FtypBox and MetaBoxLocation
std::pair<FtypBox, MetaBoxLocation> parseFtypAndLocateMeta(
    const std::vector<uint8_t>& data, 
    size_t fileOffset = 0) 
{
    FtypBox ftyp;
    MetaBoxLocation metaLoc = {0, 0, false};
    
    if (data.size() < 8) {
        throw std::runtime_error("Data too small to contain any box");
    }
    
    size_t currentOffset = 0;
    bool ftypParsed = false;
    
    // Parse boxes in the data
    while (currentOffset < data.size()) {
        // Check if we have enough data for a box header
        if (currentOffset + 8 > data.size()) {
            break;
        }
        
        BoxHeader header;
        try {
            header = parseBoxHeader(data.data(), data.size(), currentOffset);
        } catch (const std::exception& e) {
            // Not enough data for this box
            break;
        }
        
        std::cout << "Found box: '" << header.typeString() 
                  << "' at offset " << (fileOffset + currentOffset)
                  << ", size: " << header.size << " bytes\n";
        
        // Check for ftyp box
        if (header.type == 0x66747970 && !ftypParsed) { // 'ftyp'
            if (currentOffset + header.headerSize + 8 > data.size()) {
                throw std::runtime_error("Incomplete ftyp box in data");
            }
            
            // Parse ftyp content
            size_t ftypDataOffset = currentOffset + header.headerSize;
            ftyp.majorBrand = readBE32(data.data() + ftypDataOffset);
            ftyp.minorVersion = readBE32(data.data() + ftypDataOffset + 4);
            ftyp.boxSize = header.size;
            
            // Parse compatible brands
            size_t brandsOffset = ftypDataOffset + 8;
            size_t brandsEnd = safe_min(currentOffset + header.size, data.size());
            
            while (brandsOffset + 4 <= brandsEnd) {
                uint32_t brand = readBE32(data.data() + brandsOffset);
                ftyp.compatibleBrands.push_back(brand);
                brandsOffset += 4;
            }
            
            ftypParsed = true;
            
            std::cout << "  ftyp - Major Brand: " << ftyp.majorBrandString()
                      << ", Minor Version: " << ftyp.minorVersion << "\n";
            std::cout << "  Compatible Brands: ";
            for (const auto& brand : ftyp.compatibleBrandsStrings()) {
                std::cout << brand << " ";
            }
            std::cout << "\n";
        }
        
        // Check for meta box
        if (header.type == 0x6D657461) { // 'meta'
            
            std::cout << "DEBUG: Meta box header parsing:\n";
            std::cout << "  Raw bytes at offset " << currentOffset << ": ";
            for (int i = 0; i < 16; ++i) {
                printf("%02X ", data[currentOffset + i]);
            }
            std::cout << "\n";
            std::cout << "  Parsed size: " << header.size << "\n";
            std::cout << "  Header size: " << header.headerSize << "\n";            

            metaLoc.offset = fileOffset + currentOffset;
            metaLoc.size = header.size;
            metaLoc.found = true;
            
            std::cout << "  meta box found at offset " << metaLoc.offset 
                      << ", size: " << metaLoc.size << " bytes\n";
        }
        
        // Move to next box
        if (header.size == 0) {
            // Box extends to end of file, can't continue parsing
            break;
        }
        
        currentOffset += header.size;
    }
    
    if (!ftypParsed) {
        throw std::runtime_error("No valid ftyp box found in data");
    }
    
    return {ftyp, metaLoc};
}

// Convenience function: Calculate offset and byte count for meta box retrieval
struct MetaBoxRequest {
    size_t offset;
    size_t byteCount;
    bool needsRetrieval;
    
    void print() const {
        if (needsRetrieval) {
            std::cout << "\nMeta box request parameters:\n";
            std::cout << "  Offset: " << offset << " bytes\n";
            std::cout << "  Byte Count: " << byteCount << " bytes\n";
        } else {
            std::cout << "\nMeta box not found in provided data.\n";
            std::cout << "Need to read more data from file.\n";
        }
    }
};

MetaBoxRequest calculateMetaBoxRequest(const MetaBoxLocation& metaLoc) {
    MetaBoxRequest request;
    
    if (metaLoc.found) {
        request.offset = metaLoc.offset;
        request.byteCount = static_cast<size_t>(metaLoc.size);
        request.needsRetrieval = true;
    } else {
        request.offset = 0;
        request.byteCount = 0;
        request.needsRetrieval = false;
    }
    
    return request;
}

#endif // HEIF_FTYP_PARSER_DEFINED

#### Test/Example Usage

In [ ]:
#ifndef TEST_HEIFFTYP_PARSER_DEFINED
#define TEST_HEIFFTYP_PARSER_DEFINED

enum class SourceType {
    FILE,
    HTTP_RANGE
};

void testFtypParser(const std::string& source, SourceType type, size_t initialBytes = 256) {
    try {
        std::cout << "=== Testing ftyp Parser ===\n\n";
        
        // Read initial bytes based on source type
        std::vector<uint8_t> data;
        
        if (type == SourceType::FILE) {
            std::cout << "Reading from file: " << source << "\n";
            data = readFileRange(source, 0, initialBytes);
        } else if (type == SourceType::HTTP_RANGE) {
            std::cout << "Reading from HTTP: " << source << "\n";
            data = readHttpRange(source, 0, initialBytes);
        }
        
        std::cout << "Read " << data.size() << " bytes\n\n";
        
        // Parse ftyp and locate meta box
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(data, 0);
        
        std::cout << "\n=== ftyp Box Summary ===\n";
        std::cout << "Major Brand: " << ftyp.majorBrandString() << "\n";
        std::cout << "Minor Version: " << ftyp.minorVersion << "\n";
        std::cout << "Compatible Brands: ";
        for (const auto& brand : ftyp.compatibleBrandsStrings()) {
            std::cout << brand << " ";
        }
        std::cout << "\n";
        std::cout << "ftyp Box Size: " << ftyp.boxSize << " bytes\n";
        
        std::cout << "\n=== meta Box Location ===\n";
        if (metaLoc.found) {
            std::cout << "Found: Yes\n";
            std::cout << "Offset: " << metaLoc.offset << " bytes\n";
            std::cout << "Size: " << metaLoc.size << " bytes\n";
            
            // Calculate request parameters
            auto request = calculateMetaBoxRequest(metaLoc);
            request.print();
            
            std::cout << "\n=== Next Step: Retrieve meta Box ===\n";
            if (type == SourceType::FILE) {
                std::cout << "Use: readFileRange(\"" << source << "\", " 
                          << request.offset << ", " << request.byteCount << ")\n";
            } else {
                std::cout << "Use: readHttpRange(\"" << source << "\", " 
                          << request.offset << ", " << request.byteCount << ")\n";
            }
        } else {
            std::cout << "Found: No\n";
            std::cout << "The meta box is beyond the first " << initialBytes << " bytes.\n";
            std::cout << "Consider reading more data (e.g., 1024 or 4096 bytes).\n";
            std::cout << "\nRetry with more bytes:\n";
            if (type == SourceType::FILE) {
                std::cout << "  testFtypParser(\"" << source << "\", SourceType::FILE, 4096);\n";
            } else {
                std::cout << "  testFtypParser(\"" << source << "\", SourceType::HTTP_RANGE, 4096);\n";
            }
        }
        
    } catch (const std::exception& e) {
        std::cerr << "Error: " << e.what() << "\n";
    }
}

// Convenience overload that takes string for type
void testFtypParser(const std::string& source, const std::string& typeStr, size_t initialBytes = 256) {
    SourceType type;
    if (typeStr == "file" || typeStr == "FILE") {
        type = SourceType::FILE;
    } else if (typeStr == "http" || typeStr == "HTTP" || typeStr == "http-range" || typeStr == "HTTP-RANGE") {
        type = SourceType::HTTP_RANGE;
    } else {
        std::cerr << "Error: Invalid type '" << typeStr << "'. Use 'file' or 'http-range'.\n";
        return;
    }
    testFtypParser(source, type, initialBytes);
}

#endif // TEST_HEIF_FTYP_PARSER_DEFINED


#### Example Usage

In [ ]:
// Example 1: Test with a local file
//testFtypParser("benchmark_output/heif_NONE_t256_p3.heic", SourceType::FILE);
// testFtypParser("benchmark_output/output_tiled_pyramid_5levels_grid_256_deflate.heif", SourceType::FILE);

// Example 2: Test with a local file using string type
// testFtypParser("sample.heif", "file");

// Example 3: Test with HTTP URL
// testFtypParser("https://example.com/sample.heif", SourceType::HTTP_RANGE);

// Example 4: Test with HTTP URL using string type
// testFtypParser("https://example.com/sample.heif", "http-range");

// Example 5: Read more bytes if meta box not found in first 256 bytes
// testFtypParser("sample.heif", SourceType::FILE, 4096);

// Example 6: Test with HTTP and more bytes
// testFtypParser("https://example.com/sample.heif", "http", 1024);

// Example 7: test the real data
//testFtypParser("benchmark_output/output_tiled_pyramid_5levels_grid_256_deflate.heif", "file", 4096)

## Metadata Parser

### HEIF meta box

#### HEIF Meta Box Parser

#### Main

In [ ]:
#ifndef HEIF_META_BOX_PARSER_DEFINED
#define HEIF_META_BOX_PARSER_DEFINED

#include <map>
#include <set>
#include <optional>

// Uncompressed frame configuration (uncC)
// Tili-specific structures
struct TiliConfiguration {
    uint32_t numTileColumns;
    uint32_t numTileRows;
    uint32_t tileWidth;
    uint32_t tileHeight;
    uint32_t flags;  // Add this line
    bool hasExplicitOffsets;
    std::vector<uint64_t> tileOffsets;  // If provided in tili box

    // ADD THESE:
    uint8_t offsetFieldBytes;   // 4, 5, 6, or 8
    uint8_t sizeFieldBytes;     // 0, 3, 4, or 8
    bool sequentialHint;        // tiles_are_in_mdat_order flag
    
    void print() const {
        std::cout << "Tili Grid: " << numTileColumns << "×" << numTileRows;
        std::cout << ", Tile size: " << tileWidth << "×" << tileHeight;
        std::cout << ", Offset: " << (int)offsetFieldBytes << "B";
        std::cout << ", Size: " << (int)sizeFieldBytes << "B";
        std::cout << ", Sequential: " << (sequentialHint ? "Y" : "N");
        if (hasExplicitOffsets) {
            std::cout << ", " << tileOffsets.size() << " explicit offsets";
        }
    }
};


// First, update the UncompressedConfig print() method to show tile grid:
struct UncompressedConfig {
    uint32_t profile;
    std::vector<uint8_t> componentFormat;
    uint8_t samplingType;
    uint8_t interlaceMode;
    uint8_t blockSize;
    bool componentsPadded;
    uint32_t blockPadLsb;
    uint32_t blockBitDepthLsb;
    uint32_t blockReversedByteOrder;
    uint32_t padUnknown;
    uint32_t pixelSize;
    uint32_t rowAlignSize;
    uint32_t tileAlignSize;
    uint32_t numTileColsMinus1;
    uint32_t numTileRowsMinus1;

    // ADD THESE:
    uint8_t offsetFieldBytes;   // 4, 5, 6, or 8
    uint8_t sizeFieldBytes;     // 0, 3, 4, or 8
    bool sequentialHint;        // tiles_are_in_mdat_order flag

    
    void print() const {
        std::cout << "Profile: " << profile;
        
        std::cout << ", Components: [";
        for (size_t i = 0; i < componentFormat.size(); ++i) {
            if (i > 0) std::cout << ", ";
            std::cout << (int)componentFormat[i];
        }
        std::cout << "]";
        
        std::cout << ", Sampling: ";
        switch (samplingType) {
            case 0: std::cout << "4:4:4"; break;
            case 1: std::cout << "4:2:2"; break;
            case 2: std::cout << "4:2:0"; break;
            case 3: std::cout << "4:1:1"; break;
            case 4: std::cout << "4:0:0"; break;
            default: std::cout << "Unknown(" << (int)samplingType << ")"; break;
        }
        
        std::cout << ", Interleave: ";
        switch (interlaceMode) {
            case 0: std::cout << "Component"; break;
            case 1: std::cout << "Pixel"; break;
            case 2: std::cout << "Mixed"; break;
            case 3: std::cout << "Row"; break;
            case 4: std::cout << "Tile-component"; break;
            case 5: std::cout << "Multi-Y"; break;
            default: std::cout << "Unknown(" << (int)interlaceMode << ")"; break;
        }
        
        if (blockSize > 0) {
            std::cout << ", Block size: " << (int)blockSize;
        }
        
        if (componentsPadded) {
            std::cout << ", Padded";
        }
        
        if (pixelSize > 0) {
            std::cout << ", Pixel size: " << pixelSize;
        }
        
        if (rowAlignSize > 0) {
            std::cout << ", Row align: " << rowAlignSize;
        }
        
        if (tileAlignSize > 0) {
            std::cout << ", Tile align: " << tileAlignSize;
        }
        
        // ★★★ THE KEY ADDITION - SHOW TILE GRID ★★★
        if (numTileColsMinus1 > 0 || numTileRowsMinus1 > 0) {
            uint32_t cols = numTileColsMinus1 + 1;
            uint32_t rows = numTileRowsMinus1 + 1;
            std::cout << ", ★ TILE GRID: " << cols << "×" << rows << " (" << (cols*rows) << " tiles)";
        }
    }
};


// Parse icef (Image Compressed Extent Format) box for tile offsets

struct IcefUnit {
    uint64_t unit_offset;
    uint64_t unit_size;
};

struct IcefBox {
    std::vector<IcefUnit> units;
    
    void print() const {
        std::cout << "icef box: " << units.size() << " units\n";
        for (size_t i = 0; i < std::min(size_t(5), units.size()); ++i) {
            std::cout << "  Unit " << i << ": offset=" << units[i].unit_offset 
                      << ", size=" << units[i].unit_size << "\n";
        }
        if (units.size() > 5) {
            std::cout << "  ... (" << (units.size() - 5) << " more units)\n";
        }
    }
};

// Item location entry
struct ItemLocationEntry {
    uint32_t itemID;
    uint16_t constructionMethod;  // 0 = file offset, 1 = idat offset
    uint16_t dataReferenceIndex;
    uint64_t baseOffset;
    std::vector<std::pair<uint64_t, uint64_t>> extents; // (offset, length) pairs
    
    // Calculate absolute file offset and total size
    uint64_t getAbsoluteOffset() const {
        if (extents.empty()) return baseOffset;
        return baseOffset + extents[0].first;
    }
    
    uint64_t getTotalSize() const {
        uint64_t total = 0;
        for (const auto& [offset, length] : extents) {
            total += length;
        }
        return total;
    }
    
    void print() const {
        std::cout << "    Item ID " << itemID << ":\n";
        std::cout << "      Construction Method: " << constructionMethod 
                  << (constructionMethod == 0 ? " (file offset)" : " (idat offset)") << "\n";
        std::cout << "      Data Reference Index: " << dataReferenceIndex << "\n";
        std::cout << "      Base Offset: " << baseOffset << "\n";
        std::cout << "      Extents: " << extents.size() << "\n";
        for (size_t i = 0; i < extents.size(); ++i) {
            std::cout << "        [" << i << "] Offset: " << extents[i].first 
                      << ", Length: " << extents[i].second << "\n";
        }
        std::cout << "      Total Size: " << getTotalSize() << " bytes\n";
        std::cout << "      Absolute Offset: " << getAbsoluteOffset() << " bytes\n";
    }
};

// Item information entry
struct ItemInfoEntry {
    uint32_t itemID;
    uint16_t itemProtectionIndex;
    std::string itemType;  // 4-character code
    std::string itemName;
    std::string contentType;
    std::string contentEncoding;
    std::string itemUriType;
    
    void print() const {
        std::cout << "    Item ID " << itemID << ":\n";
        std::cout << "      Type: " << itemType << "\n";
        if (!itemName.empty()) 
            std::cout << "      Name: " << itemName << "\n";
        if (!contentType.empty()) 
            std::cout << "      Content Type: " << contentType << "\n";
        if (!contentEncoding.empty()) 
            std::cout << "      Content Encoding: " << contentEncoding << "\n";
        if (itemProtectionIndex > 0) 
            std::cout << "      Protection Index: " << itemProtectionIndex << "\n";
    }
};

// Item reference
struct ItemReference {
    std::string referenceType;  // 4-character code (e.g., 'dimg', 'thmb', 'auxl')
    uint32_t fromItemID;
    std::vector<uint32_t> toItemIDs;
    
    void print() const {
        std::cout << "    " << referenceType << ": Item " << fromItemID 
                  << " -> [";
        for (size_t i = 0; i < toItemIDs.size(); ++i) {
            if (i > 0) std::cout << ", ";
            std::cout << toItemIDs[i];
        }
        std::cout << "]\n";
    }
};

// Image spatial extents (from ispe property)
struct ImageSpatialExtents {
    uint32_t width;
    uint32_t height;
    
    void print() const {
        std::cout << width << "x" << height;
    }
};

// Pixel information (from pixi property)
struct PixelInformation {
    std::vector<uint8_t> bitsPerChannel;
    
    void print() const {
        std::cout << "Channels: " << bitsPerChannel.size() << " [";
        for (size_t i = 0; i < bitsPerChannel.size(); ++i) {
            if (i > 0) std::cout << ", ";
            std::cout << (int)bitsPerChannel[i];
        }
        std::cout << " bits]";
    }
};

// Colour information (from colr property)
struct ColourInformation {
    std::string colourType;
    uint16_t colourPrimaries;
    uint16_t transferCharacteristics;
    uint16_t matrixCoefficients;
    bool fullRangeFlag;
    std::vector<uint8_t> iccProfile;
    
    void print() const {
        std::cout << "Type: " << colourType;
        if (colourType == "nclx") {
            std::cout << ", Primaries: " << colourPrimaries
                      << ", Transfer: " << transferCharacteristics
                      << ", Matrix: " << matrixCoefficients
                      << ", Full Range: " << (fullRangeFlag ? "yes" : "no");
        } else if (colourType == "rICC" || colourType == "prof") {
            std::cout << ", ICC Profile: " << iccProfile.size() << " bytes";
        }
    }
};

// Auxiliary type property
struct AuxiliaryTypeProperty {
    std::string auxType;
    std::vector<uint8_t> auxSubtypes;
    
    void print() const {
        std::cout << "Type: " << auxType;
        if (!auxSubtypes.empty()) {
            std::cout << ", Subtypes: " << auxSubtypes.size() << " bytes";
        }
    }
};

// Item property
struct ItemProperty {
    uint32_t propertyIndex;
    std::string propertyType;
    bool essential;
    
    // Property data
    std::optional<ImageSpatialExtents> ispe;
    std::optional<PixelInformation> pixi;
    std::optional<ColourInformation> colr;
    std::optional<AuxiliaryTypeProperty> auxC;
    std::optional<UncompressedConfig> uncC;
    std::optional<IcefBox> icef;
    std::optional<TiliConfiguration> tili;

    void print() const {
        std::cout << "      [" << propertyIndex << "] " << propertyType;
        if (essential) std::cout << " (essential)";
        std::cout << ": ";
        
        if (ispe) ispe->print();
        else if (pixi) pixi->print();
        else if (colr) colr->print();
        else if (auxC) auxC->print();
        else if (uncC) uncC->print();
        else if (icef) icef->print();
        else if (tili) tili->print();
        
        std::cout << "\n";
    }
};

// Item property association
struct ItemPropertyAssociation {
    uint32_t itemID;
    std::vector<ItemProperty> properties;
    
    void print() const {
        std::cout << "    Item ID " << itemID << ": " << properties.size() << " properties\n";
        for (const auto& prop : properties) {
            prop.print();
        }
    }
};

// Data information
struct DataInformation {
    std::string url;
    std::string urn;
    uint16_t dataReferenceIndex;
    
    void print() const {
        std::cout << "    Data Reference [" << dataReferenceIndex << "]: ";
        if (!url.empty()) std::cout << "URL: " << url;
        if (!urn.empty()) std::cout << "URN: " << urn;
        std::cout << "\n";
    }
};

// Handler information
struct HandlerInfo {
    std::string handlerType;
    std::string name;
    
    void print() const {
        std::cout << "  Handler: " << handlerType;
        if (!name.empty()) std::cout << " (" << name << ")";
        std::cout << "\n";
    }
};

// Item data (idat) information
struct ItemDataInfo {
    uint64_t offset;  // Offset in file where idat box is located
    uint64_t size;    // Size of idat data
    bool present;
    
    ItemDataInfo() : offset(0), size(0), present(false) {}
    
    void print() const {
        if (present) {
            std::cout << "  Item Data Box (idat): Present\n";
            std::cout << "    Offset: " << offset << " bytes\n";
            std::cout << "    Size: " << size << " bytes\n";
        } else {
            std::cout << "  Item Data Box (idat): Not present\n";
        }
    }
};

// Tile/Item data request
struct ItemDataRequest {
    uint32_t itemID;
    std::string itemType;
    std::string name;
    uint64_t offset;
    uint64_t byteCount;
    uint32_t width;
    uint32_t height;
    bool isThumbnail;
    bool isAuxiliary;
    std::string auxType;
    int pyramidLevel;  // 0 = full resolution, 1+ = overview levels
    bool usesIdat;     // true if data is in idat box
    std::vector<uint8_t> bitsPerChannel;
    std::optional<ColourInformation> colourInfo;
    std::optional<UncompressedConfig> uncompressedConfig;
    
    void print() const {
        std::cout << "Item ID: " << itemID;
        if (!name.empty()) std::cout << " (\"" << name << "\")";
        std::cout << "\n";
        std::cout << "  Type: " << itemType;
        if (isThumbnail) std::cout << " [THUMBNAIL]";
        if (isAuxiliary) std::cout << " [AUXILIARY: " << auxType << "]";
        std::cout << "\n";
        std::cout << "  Dimensions: " << width << " x " << height << "\n";
        if (!bitsPerChannel.empty()) {
            std::cout << "  Bits per Channel: [";
            for (size_t i = 0; i < bitsPerChannel.size(); ++i) {
                if (i > 0) std::cout << ", ";
                std::cout << (int)bitsPerChannel[i];
            }
            std::cout << "]\n";
        }
        if (colourInfo) {
            std::cout << "  Colour Info: ";
            colourInfo->print();
            std::cout << "\n";
        }
        if (uncompressedConfig) {
            std::cout << "  Uncompressed Config: ";
            uncompressedConfig->print();
            std::cout << "\n";
        }
        std::cout << "  Pyramid Level: " << pyramidLevel << "\n";
        std::cout << "  Data Location: " << (usesIdat ? "idat box" : "file offset") << "\n";
        std::cout << "  Offset: " << offset << " bytes\n";
        std::cout << "  Size: " << byteCount << " bytes\n";
    }
};

// Add this parser for grpl box

struct EntityGroup {
    uint32_t groupID;
    std::string groupType; // 4-char code
    std::vector<uint32_t> entityIDs;
    
    void print() const {
        std::cout << "  Group ID " << groupID << " ('" << groupType << "'): ";
        std::cout << entityIDs.size() << " entities [";
        for (size_t i = 0; i < entityIDs.size(); ++i) {
            if (i > 0) std::cout << ", ";
            std::cout << entityIDs[i];
        }
        std::cout << "]\n";
    }
};

// Add this after the ItemDataRequest structure in your meta box parser section

struct TileInfo {
    uint32_t itemID;
    uint32_t tileIndex;           // Index within the item (for multi-extent)
    uint32_t row;                 // Tile row in grid (if applicable)
    uint32_t col;                 // Tile column in grid (if applicable)
    int pyramidLevel;             // 0 = full res, 1+ = overview
    uint32_t width;               // Tile width in pixels
    uint32_t height;              // Tile height in pixels
    uint64_t byteOffset;          // Absolute file offset
    uint64_t byteCount;           // Size in bytes
    std::string compression;      // "none", "deflate", "zlib", etc.
    std::string tilingScheme;     // "grid", "unci", "tili"
    
    // Spatial extent
    uint32_t xOffset;             // X offset in full image
    uint32_t yOffset;             // Y offset in full image
    
    // Additional metadata
    std::vector<uint8_t> bitsPerChannel;
    std::optional<ColourInformation> colourInfo;
    std::optional<UncompressedConfig> uncompressedConfig;
     std::optional<TiliConfiguration> tili; 
    
    void print() const {
        std::cout << "Tile [Level=" << pyramidLevel 
                  << ", Row=" << row << ", Col=" << col 
                  << ", Index=" << tileIndex << "]:\n";
        std::cout << "  Item ID: " << itemID << "\n";
        std::cout << "  Scheme: " << tilingScheme << "\n";
        std::cout << "  Dimensions: " << width << "x" << height << "\n";
        std::cout << "  Position: (" << xOffset << ", " << yOffset << ")\n";
        std::cout << "  Offset: " << byteOffset << " bytes\n";
        std::cout << "  Size: " << byteCount << " bytes\n";
        std::cout << "  Compression: " << compression << "\n";
    }
};

// Structure to hold pyramid level information for UNCI
struct UnciPyramidLevel {
    int level;                    // 0 = full resolution
    uint32_t imageWidth;
    uint32_t imageHeight;
    uint32_t numCols;
    uint32_t numRows;
    uint32_t tileWidth;
    uint32_t tileHeight;
    std::vector<TileInfo> tiles;
    uint32_t itemID;              // Item ID for this level
    std::optional<UncompressedConfig> uncC;
    std::optional<IcefBox> icef;
};

// Parsed meta box result
struct MetaBoxInfo {
    HandlerInfo handler;
    std::vector<ItemInfoEntry> items;
    std::vector<ItemLocationEntry> itemLocations;
    std::vector<ItemReference> itemReferences;
    std::vector<ItemPropertyAssociation> itemPropertyAssociations;
    std::vector<ItemProperty> propertyContainer;  // All properties from ipco
    std::vector<DataInformation> dataReferences;
    ItemDataInfo itemData;
    uint32_t primaryItemID;
    bool hasPrimaryItem;
    std::vector<EntityGroup> entityGroups;

    size_t idatOffset = 0;  // Offset to start of idat box data
    size_t mdatOffset = 0;  // Offset to start of mdat box data

    MetaBoxInfo() : primaryItemID(0), hasPrimaryItem(false) {}
    
    void printSummary() const {
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Meta Box Detailed Summary                         ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        handler.print();
        
        std::cout << "\nPrimary Item ID: ";
        if (hasPrimaryItem) {
            std::cout << primaryItemID << "\n";
        } else {
            std::cout << "Not specified\n";
        }
        
        std::cout << "\n--- Item Information (iinf) ---\n";
        std::cout << "Total Items: " << items.size() << "\n";
        for (const auto& item : items) {
            item.print();
        }
        
        std::cout << "\n--- Item Locations (iloc) ---\n";
        std::cout << "Total Locations: " << itemLocations.size() << "\n";
        for (const auto& loc : itemLocations) {
            loc.print();
        }
        
        std::cout << "\n--- Item References (iref) ---\n";
        std::cout << "Total References: " << itemReferences.size() << "\n";
        for (const auto& ref : itemReferences) {
            ref.print();
        }
        
        std::cout << "\n--- Item Properties (iprp) ---\n";
        std::cout << "Property Container: " << propertyContainer.size() << " properties\n";
        std::cout << "Property Associations: " << itemPropertyAssociations.size() << " items\n";
        for (const auto& assoc : itemPropertyAssociations) {
            assoc.print();
        }
        
        if (!dataReferences.empty()) {
            std::cout << "\n--- Data Information (dinf) ---\n";
            for (const auto& dref : dataReferences) {
                dref.print();
            }
        }
        
        std::cout << "\n--- Item Data (idat) ---\n";
        itemData.print();
    }
};

// Helper to convert 4 bytes to string
inline std::string fourCCToString(uint32_t fourcc) {
    char str[5] = {0};
    str[0] = (fourcc >> 24) & 0xFF;
    str[1] = (fourcc >> 16) & 0xFF;
    str[2] = (fourcc >> 8) & 0xFF;
    str[3] = fourcc & 0xFF;
    return std::string(str);
}

// Parse grpl (entity group list) box
void parseGrplBox(const uint8_t* data, size_t dataSize, size_t offset, 
                  size_t size, MetaBoxInfo& info, std::vector<EntityGroup>& groups) {
    std::cout << "  Entity Group List Box\n";
    
    size_t pos = offset;
    size_t endPos = offset + size;
    
    while (pos + 8 <= endPos && pos < dataSize) {
        BoxHeader groupHeader = parseBoxHeader(data, dataSize, pos);
        
        std::cout << "    Found group box: '" << groupHeader.typeString() 
                  << "' at position " << pos << ", size: " << groupHeader.size << "\n";
        
        EntityGroup group;
        group.groupType = groupHeader.typeString();
        
        size_t groupPos = pos + groupHeader.headerSize;
        
        // Read version and flags
        uint8_t version = data[groupPos];
        uint32_t flags = readBE32(data + groupPos) & 0x00FFFFFF;
        groupPos += 4;
        
        std::cout << "      Version: " << (int)version << ", Flags: " << flags << "\n";
        
        // Read group_id
        group.groupID = readBE32(data + groupPos);
        groupPos += 4;
        
        std::cout << "      Group ID: " << group.groupID << "\n";
        
        // Read num_entities_in_group
        uint32_t numEntities = readBE32(data + groupPos);
        groupPos += 4;
        
        std::cout << "      Num entities: " << numEntities << "\n";
        
        // Read entity IDs
        for (uint32_t i = 0; i < numEntities && groupPos + 4 <= pos + groupHeader.size; ++i) {
            uint32_t entityID = readBE32(data + groupPos);
            groupPos += 4;
            group.entityIDs.push_back(entityID);
            std::cout << "        Entity " << i << ": " << entityID << "\n";
        }
        
        groups.push_back(group);
        pos += groupHeader.size;
    }
}

// Forward declaration
void parseMetaBoxRecursive(const uint8_t* data, size_t dataSize, size_t offset, 
                           size_t endOffset, size_t fileOffset, MetaBoxInfo& info, int depth);

// Parse hdlr (handler) box
void parseHdlrBox(const uint8_t* data, size_t offset, size_t size, MetaBoxInfo& info, int depth = 0) {
    if (size < 24) return;
    
    // Skip version and flags (4 bytes) and pre_defined (4 bytes)
    uint32_t handlerType = readBE32(data + offset + 8);
    info.handler.handlerType = fourCCToString(handlerType);
    
    // Skip reserved bytes (12 bytes)
    size_t nameOffset = offset + 24;
    if (nameOffset < offset + size) {
        std::string name;
        while (nameOffset < offset + size && data[nameOffset] != 0) {
            name += static_cast<char>(data[nameOffset++]);
        }
        info.handler.name = name;
    }
    
    std::cout << std::string(depth * 2, ' ') << "  Handler Type: " 
              << info.handler.handlerType;
    if (!info.handler.name.empty()) {
        std::cout << " (" << info.handler.name << ")";
    }
    std::cout << "\n";
}

// Parse pitm (primary item) box
void parsePitmBox(const uint8_t* data, size_t offset, size_t size, MetaBoxInfo& info) {
    if (size < 6) return;
    
    uint8_t version = data[offset];
    if (version == 0) {
        info.primaryItemID = readBE32(data + offset + 4) & 0xFFFF;
    } else {
        info.primaryItemID = readBE32(data + offset + 4);
    }
    info.hasPrimaryItem = true;
    std::cout << "  Primary Item ID: " << info.primaryItemID << "\n";
}

// Parse iloc (item location) box - ENHANCED WITH DEBUGGING
// REPLACE the parseIlocBox function with this debug version:
// Special iloc parser for grid tiling
// Correct GRID iloc parser - handles compact tile storage format
void parseIlocBox(const uint8_t* data, size_t offset, size_t size, MetaBoxInfo& info) {
    if (size < 8) return;
    
    uint8_t version = data[offset];
    size_t pos = offset + 4;  // Skip version + flags
    
    // Read size fields (4 bits each)
    uint8_t sizes = data[pos++];
    uint8_t offsetSize = (sizes >> 4) & 0x0F;
    uint8_t lengthSize = sizes & 0x0F;
    
    uint8_t baseOffsetSize = (data[pos] >> 4) & 0x0F;
    uint8_t indexSize = 0;
    if (version == 1 || version == 2) {
        indexSize = data[pos] & 0x0F;
    }
    pos++;
    
    std::cout << "  Item Location Box: version=" << (int)version << "\n";
    std::cout << "    offsetSize: " << (int)offsetSize << " bytes\n";
    std::cout << "    lengthSize: " << (int)lengthSize << " bytes\n";
    std::cout << "    baseOffsetSize: " << (int)baseOffsetSize << " bytes\n";
    std::cout << "    indexSize: " << (int)indexSize << " bytes\n";
    
    // Read item count according to spec
    uint32_t itemCount;
    if (version < 2) {
        // 16-bit item count (2 bytes)
        itemCount = (data[pos] << 8) | data[pos + 1];
        pos += 2;
    } else {
        // 32-bit item count (4 bytes)
        itemCount = readBE32(data + pos);
        pos += 4;
    }
    
    std::cout << "  Item count: " << itemCount << " items\n";
    
    // Parse each item
    for (uint32_t i = 0; i < itemCount && pos < offset + size; ++i) {
        ItemLocationEntry entry;
        
        // Read item ID according to spec
        if (version < 2) {
            // 16-bit item ID (2 bytes)
            if (pos + 2 > offset + size) break;
            entry.itemID = (data[pos] << 8) | data[pos + 1];
            pos += 2;
        } else {
            // 32-bit item ID (4 bytes)
            if (pos + 4 > offset + size) break;
            entry.itemID = readBE32(data + pos);
            pos += 4;
        }
        
        // Read construction method (version 1 or 2 only)
        if (version == 1 || version == 2) {
            if (pos + 2 > offset + size) break;
            uint16_t temp = (data[pos] << 8) | data[pos + 1];
            entry.constructionMethod = temp & 0x0F;
            pos += 2;
        } else {
            entry.constructionMethod = 0;
        }
        
        // Read data reference index (16-bit)
        if (pos + 2 > offset + size) break;
        entry.dataReferenceIndex = (data[pos] << 8) | data[pos + 1];
        pos += 2;
        
        // Read base offset (variable size)
        entry.baseOffset = 0;
        if (baseOffsetSize == 4) {
            if (pos + 4 > offset + size) break;
            entry.baseOffset = readBE32(data + pos);
            pos += 4;
        } else if (baseOffsetSize == 8) {
            if (pos + 8 > offset + size) break;
            entry.baseOffset = readBE64(data + pos);
            pos += 8;
        }
        
        // Read extent count (16-bit)
        if (pos + 2 > offset + size) break;
        uint16_t extentCount = (data[pos] << 8) | data[pos + 1];
        pos += 2;
        
        // Debug output for first few items
        if (i < 5) {
            std::cout << "  Item " << entry.itemID 
                      << ": baseOffset=" << entry.baseOffset 
                      << ", constructionMethod=" << entry.constructionMethod 
                      << ", extentCount=" << extentCount << "\n";
        }
        
        // Parse extents
        for (uint16_t j = 0; j < extentCount && pos < offset + size; ++j) {
            uint64_t extentIndex = 0;
            uint64_t extentOffset = 0;
            uint64_t extentLength = 0;
            
            // Read extent index if present (variable size)
            if ((version == 1 || version == 2) && indexSize > 0) {
                if (indexSize == 4) {
                    if (pos + 4 > offset + size) break;
                    extentIndex = readBE32(data + pos);
                    pos += 4;
                } else if (indexSize == 8) {
                    if (pos + 8 > offset + size) break;
                    extentIndex = readBE64(data + pos);
                    pos += 8;
                }
            }
            
            // Read extent offset (variable size)
            if (offsetSize == 4) {
                if (pos + 4 > offset + size) break;
                extentOffset = readBE32(data + pos);
                pos += 4;
            } else if (offsetSize == 8) {
                if (pos + 8 > offset + size) break;
                extentOffset = readBE64(data + pos);
                pos += 8;
            }
            
            // Read extent length (variable size)
            if (lengthSize == 4) {
                if (pos + 4 > offset + size) break;
                extentLength = readBE32(data + pos);
                pos += 4;
            } else if (lengthSize == 8) {
                if (pos + 8 > offset + size) break;
                extentLength = readBE64(data + pos);
                pos += 8;
            }
            
            // Debug output for first few items
            if (i < 5 && j == 0) {
                std::cout << "    Extent: offset=" << extentOffset 
                          << ", length=" << extentLength << "\n";
            }
            
            entry.extents.push_back({extentOffset, extentLength});
        }
        
        info.itemLocations.push_back(entry);
    }
    
    std::cout << "  Successfully parsed " << info.itemLocations.size() << " item locations\n";
}

// FIXED iinf parser - the bug was reading entry count
void parseIinfBox(const uint8_t* data, size_t dataSize, size_t offset, 
                  size_t size, MetaBoxInfo& info) {
    if (size < 6) return;
    
    uint8_t version = data[offset];
    size_t pos = offset + 4; // Skip version and flags
    
    // BUG FIX: entry count reading
    uint32_t entryCount;
    if (version == 0) {
        entryCount = readBE32(data + pos) & 0xFFFF; // 16-bit for version 0
        pos += 2;
    } else {
        entryCount = readBE32(data + pos); // 32-bit for version 1+
        pos += 4;
    }
    
    std::cout << "  Item Information Box: entryCount=" << entryCount 
              << " (version " << (int)version << ")\n";
    
    // Debug: Show what we're about to parse
    std::cout << "    Parse range: " << pos << " to " << (offset + size) << "\n";
    std::cout << "    Available bytes: " << ((offset + size) - pos) << "\n";
    
    // Check if entryCount makes sense
    if (entryCount == 0 || entryCount > 10000) {
        std::cout << "    WARNING: Suspicious entry count, trying to parse anyway...\n";
    }
    
    size_t endPos = offset + size;
    uint32_t itemsFound = 0;
    
    // Parse 'infe' boxes
    while (pos + 8 <= endPos && pos < dataSize) {
        BoxHeader infeHeader = parseBoxHeader(data, dataSize, pos);
        
        if (infeHeader.type != 0x696E6665) { // 'infe'
            std::cout << "    Unexpected box type: '" << infeHeader.typeString() << "'\n";
            break;
        }
        
        ItemInfoEntry entry;
        size_t infePos = pos + infeHeader.headerSize;
        
        uint8_t infeVersion = data[infePos];
        infePos += 4; // Skip version and flags
        
        if (infeVersion >= 2) {
            // Read item_ID
            if (infeVersion == 2) {
                entry.itemID = readBE32(data + infePos) & 0xFFFF;
                infePos += 2;
            } else {
                entry.itemID = readBE32(data + infePos);
                infePos += 4;
            }
            
            // Read item_protection_index
            entry.itemProtectionIndex = readBE32(data + infePos) & 0xFFFF;
            infePos += 2;
            
            // Read item_type
            uint32_t itemType = readBE32(data + infePos);
            entry.itemType = fourCCToString(itemType);
            infePos += 4;
            
            // Read item_name (null-terminated)
            while (infePos < pos + infeHeader.size && data[infePos] != 0) {
                entry.itemName += static_cast<char>(data[infePos++]);
            }
            if (infePos < dataSize && data[infePos] == 0) infePos++;
            
            if (itemsFound < 5 || itemsFound % 100 == 0) {
                std::cout << "    Item " << itemsFound << ": ID=" << entry.itemID 
                          << ", type='" << entry.itemType << "'\n";
            }
            
            info.items.push_back(entry);
            itemsFound++;
        }
        
        pos += infeHeader.size;
    }
    
    std::cout << "    Parsed " << itemsFound << " items\n";
}

// Parse iref (item reference) box
void parseIrefBox(const uint8_t* data, size_t dataSize, size_t offset, 
                  size_t size, MetaBoxInfo& info) {
    if (size < 4) return;
    
    uint8_t version = data[offset];
    size_t pos = offset + 4;
    size_t endPos = offset + size;
    
    std::cout << "  Item Reference Box (version " << (int)version << ")\n";
    
    while (pos + 8 <= endPos && pos < dataSize) {
        BoxHeader refHeader = parseBoxHeader(data, dataSize, pos);
        
        ItemReference ref;
        ref.referenceType = fourCCToString(refHeader.type);
        
        size_t refPos = pos + refHeader.headerSize;
        size_t refEndPos = pos + refHeader.size;
        
        // Read from_item_ID
        if (version == 0) {
            ref.fromItemID = readBE32(data + refPos) & 0xFFFF;
            refPos += 2;
        } else {
            ref.fromItemID = readBE32(data + refPos);
            refPos += 4;
        }
        
        // Skip the "count" field (it's wrong in this file format)
        uint16_t ignoredField = readBE32(data + refPos) & 0xFFFF;
        refPos += 2;
        
        // Calculate ACTUAL count from remaining bytes
        size_t remainingBytes = refEndPos - refPos;
        size_t itemIDSize = (version == 0) ? 2 : 4;
        size_t actualCount = remainingBytes / itemIDSize;
        
        std::cout << "    " << ref.referenceType << ": Item " << ref.fromItemID 
                  << " -> " << actualCount << " tiles\n";
        
        // Read all item IDs
        for (size_t i = 0; i < actualCount && refPos + itemIDSize <= refEndPos; ++i) {
            uint32_t toItemID;
            if (version == 0) {
                toItemID = readBE32(data + refPos) & 0xFFFF;
                refPos += 2;
            } else {
                toItemID = readBE32(data + refPos);
                refPos += 4;
            }
            ref.toItemIDs.push_back(toItemID);
        }
        
        info.itemReferences.push_back(ref);
        pos += refHeader.size;
    }
    
    std::cout << "    Total references parsed: " << info.itemReferences.size() << "\n";
}

// Parse ispe (image spatial extents) property
ImageSpatialExtents parseIspeProperty(const uint8_t* data, size_t offset, size_t size) {
    ImageSpatialExtents ispe;
    size_t pos = offset + 4; // Skip version and flags
    
    ispe.width = readBE32(data + pos);
    ispe.height = readBE32(data + pos + 4);
    
    return ispe;
}

// Parse pixi (pixel information) property
PixelInformation parsePixiProperty(const uint8_t* data, size_t offset, size_t size) {
    PixelInformation pixi;
    size_t pos = offset + 4; // Skip version and flags
    
    uint8_t numChannels = data[pos++];
    for (uint8_t i = 0; i < numChannels && pos < offset + size; ++i) {
        pixi.bitsPerChannel.push_back(data[pos++]);
    }
    
    return pixi;
}

// Parse colr (colour information) property
ColourInformation parseColrProperty(const uint8_t* data, size_t offset, size_t size) {
    ColourInformation colr;
    size_t pos = offset;
    
    // Read colour type (4 bytes)
    uint32_t colourType = readBE32(data + pos);
    colr.colourType = fourCCToString(colourType);
    pos += 4;
    
    if (colr.colourType == "nclx") {
        colr.colourPrimaries = readBE32(data + pos) & 0xFFFF;
        pos += 2;
        colr.transferCharacteristics = readBE32(data + pos) & 0xFFFF;
        pos += 2;
        colr.matrixCoefficients = readBE32(data + pos) & 0xFFFF;
        pos += 2;
        colr.fullRangeFlag = (data[pos] & 0x80) != 0;
    } else if (colr.colourType == "rICC" || colr.colourType == "prof") {
        // ICC profile data
        while (pos < offset + size) {
            colr.iccProfile.push_back(data[pos++]);
        }
    }
    
    return colr;
}

// Parse auxC (auxiliary type) property
AuxiliaryTypeProperty parseAuxCProperty(const uint8_t* data, size_t offset, size_t size) {
    AuxiliaryTypeProperty auxC;
    size_t pos = offset + 4; // Skip version and flags
    
    // Read null-terminated aux type string
    while (pos < offset + size && data[pos] != 0) {
        auxC.auxType += static_cast<char>(data[pos++]);
    }
    pos++; // Skip null terminator
    
    // Read aux subtypes if present
    while (pos < offset + size) {
        auxC.auxSubtypes.push_back(data[pos++]);
    }
    
    return auxC;
}

// Parse uncC (Uncompressed Codec Configuration) property
UncompressedConfig parseUncCProperty(const uint8_t* data, size_t offset, size_t size) {
    UncompressedConfig uncC = {};
    
    if (size < 12) return uncC;
    
    size_t pos = offset;
    
    // Read version and flags (FullBox header)
    uint8_t version = data[pos++];
    uint32_t flags = ((uint32_t)data[pos] << 16) | ((uint32_t)data[pos+1] << 8) | data[pos+2];
    pos += 3;
    
    std::cout << "      Version: " << (int)version << ", Flags: 0x" << std::hex << flags << std::dec << "\n";
    
    //-------
    std::cout << "      uncC Version: " << (int)version << ", Flags: 0x" << std::hex << flags << std::dec << "\n";
    
    // Decode offset and size field lengths from flags
    // Bits 0-1: offset field length
    uint8_t offsetLengthCode = flags & 0x03;
    // Bits 2-3: size field length  
    uint8_t sizeLengthCode = (flags >> 2) & 0x03;
    
    // Map codes to byte lengths
    const uint8_t offsetBytesMap[] = {0, 3, 4, 8};  // 0=none, 1=24bit, 2=32bit, 3=64bit
    const uint8_t sizeBytesMap[] = {0, 3, 4, 8};    // 0=inferred, 1=24bit, 2=32bit, 3=64bit
    
    uncC.offsetFieldBytes = offsetBytesMap[offsetLengthCode];
    uncC.sizeFieldBytes = sizeBytesMap[sizeLengthCode];

    // Extract sequential hint (bit 4)
    uncC.sequentialHint = (flags & 0x10) != 0;    
    
    std::cout << "      Decoded from uncC flags:\n";
    std::cout << "        Offset field: " << (int)uncC.offsetFieldBytes << " bytes";
    if (uncC.offsetFieldBytes == 0) std::cout << " (no offset field)";
    std::cout << "\n";
    std::cout << "        Size field: " << (int)uncC.sizeFieldBytes << " bytes";
    if (uncC.sizeFieldBytes == 0) std::cout << " (inferred from offsets)";
    std::cout << "\n";
    std::cout << "        Sequential hint: " << (uncC.sequentialHint ? "yes" : "no") << "\n";
    //--------


    // Profile (4 bytes)
    uncC.profile = readBE32(data + pos);
    pos += 4;
    
    // Component count (4 bytes)
    uint32_t componentCount = readBE32(data + pos);
    pos += 4;
    
    std::cout << "      Components: " << componentCount << ", pos after count: " << (pos-offset) << "\n";
    
    // CRITICAL: Try 5 bytes per component (spec-compliant)
    // Format: 2-byte index + 1-byte bit_depth + 1-byte format + 1-byte align
    for (uint32_t i = 0; i < componentCount && pos + 5 <= offset + size; ++i) {
        uint16_t index = (data[pos] << 8) | data[pos + 1];
        pos += 2;
        uint8_t bit_depth = data[pos++];
        uint8_t format = data[pos++];
        uint8_t align_size = data[pos++];
        
        uncC.componentFormat.push_back(bit_depth);
        
        if (i == 0) {
            std::cout << "      Component " << i << ": bytes=";
            printf("%02x %02x %02x %02x %02x", 
                   data[pos-5], data[pos-4], data[pos-3], data[pos-2], data[pos-1]);
            std::cout << " → idx=" << index << ", depth=" << (int)bit_depth << "\n";
        }
    }
    
    std::cout << "      After components: pos=" << (pos-offset) << "\n";
    
    // Single-byte fields
    if (pos < offset + size) uncC.samplingType = data[pos++];
    if (pos < offset + size) uncC.interlaceMode = data[pos++];
    if (pos < offset + size) uncC.blockSize = data[pos++];
    if (pos < offset + size) {
        uint8_t flagsByte = data[pos++];
        uncC.componentsPadded = (flagsByte & 0x80) != 0;
    }
    
    std::cout << "      After single-byte fields: pos=" << (pos-offset) << "\n";
    
    // Read optional 32-bit fields if flags indicate OR if data is present
    // (libheif seems to always include them)
    if ((flags & 0x01) || pos + 4 <= offset + size) {
        uncC.pixelSize = readBE32(data + pos);
        pos += 4;
    }
    
    if ((flags & 0x02) || pos + 4 <= offset + size) {
        uncC.rowAlignSize = readBE32(data + pos);
        pos += 4;
    }
    
    if ((flags & 0x04) || pos + 4 <= offset + size) {
        uncC.tileAlignSize = readBE32(data + pos);
        pos += 4;
    }
    
    std::cout << "      After optional fields: pos=" << (pos-offset) << "\n";
    std::cout << "      Remaining bytes: " << (offset + size - pos) << "\n";
    
    // ALWAYS try to read tile dimensions (libheif extension?)
    if (pos + 4 <= offset + size) {
        uncC.numTileColsMinus1 = readBE32(data + pos);
        pos += 4;
        std::cout << "      ★★★ TILE COLUMNS: " << (uncC.numTileColsMinus1 + 1) 
                  << " (at pos " << (pos-offset-4) << ") ★★★\n";
    }
    
    if (pos + 4 <= offset + size) {
        uncC.numTileRowsMinus1 = readBE32(data + pos);
        pos += 4;
        std::cout << "      ★★★ TILE ROWS: " << (uncC.numTileRowsMinus1 + 1) 
                  << " (at pos " << (pos-offset-4) << ") ★★★\n";
    }
    
    std::cout << "      Total consumed: " << (pos-offset) << " of " << size << " bytes\n";
    
    return uncC;
}

// IMPROVED icef parser - correctly handles tile offset/size pairs
// CORRECTED icef parser
// CORRECTED icef parser - properly handles cumulative offsets
// CORRECTED: icef parser - stores raw values for later mapping
// CORRECTED: icef parser - SORT values before computing sizes
// CORRECTED: icef parser - stores tile extents in file order
// FINAL CORRECT VERSION: icef stores relative END offsets
// FINAL CORRECTED VERSION

IcefBox parseIcefProperty(const uint8_t* data, size_t offset, size_t size, 
                          const UncompressedConfig* uncC) {
    IcefBox icef;
    
    if (size < 5) {
        std::cout << "      ⚠ icef box too small: " << size << " bytes\n";
        return icef;
    }
    
    size_t pos = offset;
    
    // Read FullBox header (version + flags)
    uint8_t version = data[pos];
    uint32_t flags = readBE32(data + pos) & 0x00FFFFFF;
    pos += 4;
    
    std::cout << "      icef box:\n";
    std::cout << "        Version: " << (int)version << ", Flags: 0x" << std::hex << flags << std::dec << "\n";
    
    // Get field lengths from uncC configuration
    uint8_t offset_field_bytes = 0;
    uint8_t size_field_bytes = 0;
    bool has_offset_field = false;
    bool has_size_field = false;
    bool sequential_hint = false;

    if (uncC != nullptr) {
        offset_field_bytes = uncC->offsetFieldBytes;
        size_field_bytes = uncC->sizeFieldBytes;
        sequential_hint = uncC->sequentialHint; 
        has_offset_field = (offset_field_bytes > 0);
        has_size_field = (size_field_bytes > 0);
        
        std::cout << "        Using field lengths from uncC:\n";
        std::cout << "          offset_field_bytes: " << (int)offset_field_bytes << "\n";
        std::cout << "          size_field_bytes: " << (int)size_field_bytes << "\n";
        std::cout << "          sequential_hint: " << (sequential_hint ? "yes" : "no") << "\n";
        std::cout << "          has_offset_field: " << (has_offset_field ? "yes" : "no") << "\n";
        std::cout << "          has_size_field: " << (has_size_field ? "yes" : "no") << "\n";

    } else {
        std::cout << "        ⚠ No uncC config provided, will infer format\n";
    }
    
    // Read the format/header byte (if present)
    uint8_t header_byte = data[pos++];
    std::cout << "        Header byte: 0x" << std::hex << (int)header_byte << std::dec << "\n";
    
    // Read 4-byte tile count
    uint32_t tile_count = readBE32(data + pos);
    pos += 4;
    
    std::cout << "        Tile count: " << tile_count << "\n";
    
    // If uncC didn't provide field lengths, infer from remaining bytes
    if (!has_offset_field && !has_size_field) {
        size_t remaining = (offset + size) - pos;
        
        std::cout << "        Inferring format from remaining bytes...\n";
        std::cout << "        Remaining bytes: " << remaining << "\n";
        
        if (tile_count > 0 && remaining % tile_count == 0) {
            size_t bytes_per_tile = remaining / tile_count;
            std::cout << "        Bytes per tile: " << bytes_per_tile << "\n";
            
            // Assume size-only format (most common for sequential storage)
            if (bytes_per_tile == 3 || bytes_per_tile == 4 || bytes_per_tile == 8) {
                has_size_field = true;
                size_field_bytes = bytes_per_tile;
                std::cout << "        Inferred: size-only format, " << (int)size_field_bytes << " bytes per size\n";
            } else if (bytes_per_tile == 6) {
                // Could be 3+3
                has_offset_field = true;
                has_size_field = true;
                offset_field_bytes = 3;
                size_field_bytes = 3;
                std::cout << "        Inferred: offset+size format, 3+3 bytes\n";
            } else if (bytes_per_tile == 8) {
                // Could be 4+4 or 8-byte single field
                has_offset_field = true;
                has_size_field = true;
                offset_field_bytes = 4;
                size_field_bytes = 4;
                std::cout << "        Inferred: offset+size format, 4+4 bytes\n";
            }
        } else {
            std::cout << "        ✗ Cannot infer format - data size doesn't divide evenly\n";
            return icef;
        }
    }
    
    // Calculate expected bytes
    size_t entry_size = offset_field_bytes + size_field_bytes;
    size_t remaining = (offset + size) - pos;
    size_t expected_bytes = tile_count * entry_size;
    
    std::cout << "        Entry size: " << entry_size << " bytes\n";
    std::cout << "        Remaining bytes: " << remaining << "\n";
    std::cout << "        Expected bytes: " << expected_bytes << "\n";
    
    if (remaining != expected_bytes) {
        std::cout << "        ⚠ Size mismatch!\n";
    } else {
        std::cout << "        ✓ Size matches\n";
    }
    
    // Read tile data
    std::vector<uint64_t> offsets;
    std::vector<uint64_t> sizes;
    
    for (size_t i = 0; i < tile_count; ++i) {
        if (pos + entry_size > offset + size) {
            std::cout << "        ⚠ Ran out of data at entry " << i << "\n";
            break;
        }
        
        // Read offset field if present
        uint64_t offset_value = 0;
        if (has_offset_field) {
            for (size_t b = 0; b < offset_field_bytes; ++b) {
                offset_value = (offset_value << 8) | data[pos++];
            }
            offsets.push_back(offset_value);
        }
        
        // Read size field if present
        uint64_t size_value = 0;
        if (has_size_field) {
            for (size_t b = 0; b < size_field_bytes; ++b) {
                size_value = (size_value << 8) | data[pos++];
            }
            sizes.push_back(size_value);
        }
        
        if (i < 10) {
            std::cout << "        Entry " << i << ":";
            if (has_offset_field) std::cout << " offset=" << offset_value;
            if (has_size_field) std::cout << " size=" << size_value;
            std::cout << "\n";
        }
    }
    
    std::cout << "        Read " << tile_count << " entries\n";
    
    // Build IcefUnits based on what data we have
    
    if (has_offset_field && has_size_field) {
        // ============================================================
        // CASE 1: Both offset and size fields present
        // ============================================================
        std::cout << "        Building units: BOTH offset and size present\n";
        
        for (size_t i = 0; i < tile_count; ++i) {
            IcefUnit unit;
            unit.unit_offset = offsets[i];
            unit.unit_size = sizes[i];
            icef.units.push_back(unit);
            
            if (i < 5 || i == tile_count - 1) {
                std::cout << "        Tile " << i << ": offset=" << unit.unit_offset 
                          << ", size=" << unit.unit_size << "\n";
            }
        }
        
    } else if (has_size_field && !has_offset_field) {
        // ============================================================
        // CASE 2: Size-only format - calculate offsets
        // ============================================================
        std::cout << "        Building units: SIZE-ONLY (offsets calculated)\n";
        if (!sequential_hint) {
            std::cout << "        WARNING: No sequential hint, sizes may not be in file order\n";
        }
        
        uint64_t cumulative_offset = 0;
        
        for (size_t i = 0; i < sizes.size(); ++i) {
            IcefUnit unit;
            unit.unit_offset = cumulative_offset;
            unit.unit_size = sizes[i];
            icef.units.push_back(unit);
            
            cumulative_offset += sizes[i];
            
            if (i < 5 || i == sizes.size() - 1) {
                std::cout << "        Tile " << i << ": offset=" << unit.unit_offset 
                          << " (calculated), size=" << unit.unit_size << "\n";
            }
        }
        
        std::cout << "        Total accumulated size: " << cumulative_offset << " bytes\n";
        
    } else if (has_offset_field && !has_size_field) {
        // ============================================================
        // CASE 3: Offset-only format - infer sizes
        // ============================================================
        std::cout << "        Building units: OFFSET-ONLY (sizes inferred)\n";
        if (!sequential_hint) {
            std::cout << "        WARNING: No sequential hint, offsets may not be in file order\n";
        }

        // Check if offsets are absolute or deltas
        bool offsets_increasing = true;
        for (size_t i = 1; i < offsets.size(); ++i) {
            if (offsets[i] < offsets[i-1]) {
                offsets_increasing = false;
                break;
            }
        }
        
        if (!offsets_increasing) {
            // Offsets appear to be deltas - convert to cumulative
            std::cout << "        Offsets appear to be deltas, converting to cumulative...\n";
            uint64_t cumulative = 0;
            for (size_t i = 0; i < offsets.size(); ++i) {
                if (i == 0) {
                    cumulative = offsets[i];
                } else {
                    cumulative += offsets[i];
                }
                offsets[i] = cumulative;
                
                if (i < 5) {
                    std::cout << "        Tile " << i << ": cumulative_offset=" << cumulative << "\n";
                }
            }
        }
        
        // Infer sizes from consecutive offsets
        for (size_t i = 0; i < offsets.size(); ++i) {
            IcefUnit unit;
            unit.unit_offset = offsets[i];
            
            if (i + 1 < offsets.size()) {
                // Size = next offset - current offset
                unit.unit_size = offsets[i + 1] - offsets[i];
            } else {
                // Last tile: size is unknown here, will be calculated later
                // Mark as 0 so it can be determined from iloc extent
                unit.unit_size = 0;
                std::cout << "        Tile " << i << ": offset=" << unit.unit_offset 
                          << ", size=UNKNOWN (will be calculated from iloc extent)\n";
            }
            
            icef.units.push_back(unit);
            
            if (i < offsets.size() - 1 && i < 5) {
                std::cout << "        Tile " << i << ": offset=" << unit.unit_offset 
                          << ", size=" << unit.unit_size << " (inferred)\n";
            }
        }
    } else {
        std::cout << "        ✗ ERROR: No fields present!\n";
        return icef;
    }
    
    std::cout << "      ✓ Parsed " << icef.units.size() << " tile extents\n";
    std::cout << "      NOTE: Offsets are RELATIVE to item's baseOffset from iloc!\n";
    
    return icef;
}


// Enhanced ipma parser with debugging - ADD THIS BEFORE parseIprpBox

void parseIpmaBox(const uint8_t* data, size_t dataSize, size_t offset, 
                  size_t size, MetaBoxInfo& info,
                  const std::vector<ItemProperty>& propertyContainer) {
    
    std::cout << "    Item Property Association Box:\n";
    
    if (size < 8) {
        std::cout << "      Too small\n";
        return;
    }
    
    size_t pos = offset;
    uint8_t version = data[pos];
    uint32_t flags = readBE32(data + pos) & 0x00FFFFFF;
    pos += 4;
    
    std::cout << "      Version: " << (int)version << ", Flags: 0x" << std::hex << flags << std::dec << "\n";
    
    if (pos + 4 > offset + size) return;
    
    uint32_t entryCount = readBE32(data + pos);
    pos += 4;
    
    std::cout << "      Entry count: " << entryCount << "\n";
    
    for (uint32_t i = 0; i < entryCount && pos < offset + size; ++i) {
        ItemPropertyAssociation assoc;
        
        // Read item_ID
        if (version < 1) {
            if (pos + 2 > offset + size) break;
            assoc.itemID = readBE32(data + pos) & 0xFFFF;
            pos += 2;
        } else {
            if (pos + 4 > offset + size) break;
            assoc.itemID = readBE32(data + pos);
            pos += 4;
        }
        
        std::cout << "      Entry " << i << ": Item ID " << assoc.itemID << "\n";
        
        // Read association count
        if (pos + 1 > offset + size) break;
        uint8_t associationCount = data[pos++];
        
        std::cout << "        Associations: " << (int)associationCount << "\n";
        
        // Read associations
        for (uint8_t j = 0; j < associationCount && pos < offset + size; ++j) {
            bool essential;
            uint32_t propertyIndex;
            
            if (flags & 1) { // 15-bit property index
                if (pos + 2 > offset + size) break;
                uint16_t temp = readBE32(data + pos) & 0xFFFF;
                essential = (temp & 0x8000) != 0;
                propertyIndex = temp & 0x7FFF;
                pos += 2;
            } else { // 7-bit property index
                if (pos + 1 > offset + size) break;
                uint8_t temp = data[pos++];
                essential = (temp & 0x80) != 0;
                propertyIndex = temp & 0x7F;
            }
            
            std::cout << "          [" << j << "] Property " << propertyIndex;
            if (essential) std::cout << " (essential)";
            
            // Get property from container
            if (propertyIndex > 0 && propertyIndex <= propertyContainer.size()) {
                ItemProperty prop = propertyContainer[propertyIndex - 1];
                prop.essential = essential;
                assoc.properties.push_back(prop);
                
                std::cout << " - " << prop.propertyType;
                if (prop.ispe) {
                    std::cout << " [" << prop.ispe->width << "×" << prop.ispe->height << "]";
                }
            } else {
                std::cout << " - INVALID INDEX";
            }
            std::cout << "\n";
        }
        
        info.itemPropertyAssociations.push_back(assoc);
    }
    
    std::cout << "      Total associations created: " << info.itemPropertyAssociations.size() << "\n";
}


// Parse tilC property box
// Parse tilC property box (note: it's tilC, not tili!)

// First, let's verify the tilC parsing is reading the right data
TiliConfiguration parseTilCProperty(const uint8_t* data, size_t offset, size_t size) {
    TiliConfiguration tili = {};
    tili.hasExplicitOffsets = false;
    
    if (size < 12) return tili;
    
    size_t pos = offset;
    
    // Read version and flags
    uint8_t version = data[pos];
    tili.flags = readBE32(data + pos) & 0x00FFFFFF;  // Extract flags (24 bits)
    pos += 4;
    

    // Decode flags according to tili spec
    uint8_t offsetFieldLength = 32;  // Default
    uint8_t sizeFieldLength = 0;     // Default: inferred

    // Flags bit 0-1: offset_field_length
    uint8_t offsetLengthCode = tili.flags & 0x03;
    switch (offsetLengthCode) {
        case 0: offsetFieldLength = 32; break;
        case 1: offsetFieldLength = 40; break;
        case 2: offsetFieldLength = 48; break;
        case 3: offsetFieldLength = 64; break;
    }

    // Flags bit 2-3: size_field_length
    uint8_t sizeLengthCode = (tili.flags >> 2) & 0x03;
    switch (sizeLengthCode) {
        case 0: sizeFieldLength = 0; break;   // Inferred
        case 1: sizeFieldLength = 24; break;
        case 2: sizeFieldLength = 32; break;
        case 3: sizeFieldLength = 64; break;
    }

    // Flags bit 4: tiles_are_in_mdat_order
    bool sequentialHint = (tili.flags & 0x10) != 0;

    std::cout << "  tilC flags decoded:\n";
    std::cout << "    Offset field: " << (int)offsetFieldLength << " bits\n";
    std::cout << "    Size field: " << (int)sizeFieldLength << " bits\n";
    std::cout << "    Sequential: " << (sequentialHint ? "yes" : "no") << "\n";

    // Store these in your TiliConfiguration
    tili.offsetFieldBytes = offsetFieldLength / 8;
    tili.sizeFieldBytes = sizeFieldLength / 8;
    tili.sequentialHint = sequentialHint;
    
    // Read tile dimensions
    tili.tileWidth = readBE32(data + pos);
    pos += 4;
    tili.tileHeight = readBE32(data + pos);
    pos += 4;
    
    std::cout << "  Tile dimensions: " << tili.tileWidth << "×" << tili.tileHeight << "\n";
    std::cout << "  Flags: 0x" << std::hex << tili.flags << std::dec << "\n";
        
    // Calculate expected grid (will be validated against actual data)
    // This is a placeholder - actual grid comes from image dimensions
    
    // Check for optional tile offset table
    size_t remainingBytes = (offset + size) - pos;
    
    if (remainingBytes >= 4) {
        // Might have tile offset table
        // Read first value to check if it looks like an offset
        uint32_t possibleOffset = readBE32(data + pos);
        
        // Heuristic: tile offsets should be reasonable (< 1GB)
        if (possibleOffset > 0 && possibleOffset < 1000000000) {
            size_t numOffsets = remainingBytes / 4;
            std::cout << "  Found potential offset table: " << numOffsets << " entries\n";
            
            tili.hasExplicitOffsets = true;
            for (size_t i = 0; i < numOffsets && pos + 4 <= offset + size; ++i) {
                uint64_t offset_val = readBE32(data + pos);
                tili.tileOffsets.push_back(offset_val);
                pos += 4;
            }
            
            std::cout << "  ✓ Loaded " << tili.tileOffsets.size() << " tile offsets\n";
        }
    }
    
    return tili;
}

// Update the parseIprpBox function to properly call ipma parser
void parseIprpBox(const uint8_t* data, size_t dataSize, size_t offset, 
                  size_t size, MetaBoxInfo& info) {
    std::cout << "  Item Properties Box\n";
    // In parseIprpBox, when parsing properties:
    std::optional<UncompressedConfig> currentUncC;
    
    size_t pos = offset;
    size_t endPos = offset + size;
    
    // First pass: parse ipco (property container)
    while (pos + 8 <= endPos && pos < dataSize) {
        BoxHeader propHeader = parseBoxHeader(data, dataSize, pos);
        
        if (propHeader.type == 0x6970636F) { // 'ipco'
            std::cout << "    Item Property Container: parsing properties...\n";
            
            size_t ipcoPos = pos + propHeader.headerSize;
            size_t ipcoEnd = pos + propHeader.size;
            uint32_t propertyIndex = 1;
            
            while (ipcoPos + 8 <= ipcoEnd && ipcoPos < dataSize) {
                BoxHeader itemPropHeader = parseBoxHeader(data, dataSize, ipcoPos);
                
                ItemProperty prop;
                prop.propertyIndex = propertyIndex++;
                prop.propertyType = fourCCToString(itemPropHeader.type);
                prop.essential = false;
                
                size_t propContentOffset = ipcoPos + itemPropHeader.headerSize;
                size_t propContentSize = itemPropHeader.size - itemPropHeader.headerSize;
                
                if (itemPropHeader.type == 0x69737065) { // 'ispe'
                    prop.ispe = parseIspeProperty(data, propContentOffset, propContentSize);
                    std::cout << "      [" << prop.propertyIndex << "] ispe: " 
                              << prop.ispe->width << "×" << prop.ispe->height << "\n";
                              
                } else if (itemPropHeader.type == 0x70697869) { // 'pixi'
                    prop.pixi = parsePixiProperty(data, propContentOffset, propContentSize);
                    std::cout << "      [" << prop.propertyIndex << "] pixi: ";
                    prop.pixi->print();
                    std::cout << "\n";
                    
                } else if (itemPropHeader.type == 0x636F6C72) { // 'colr'
                    prop.colr = parseColrProperty(data, propContentOffset, propContentSize);
                    std::cout << "      [" << prop.propertyIndex << "] colr: ";
                    prop.colr->print();
                    std::cout << "\n";
                    
                } else if (itemPropHeader.type == 0x61757843) { // 'auxC'
                    prop.auxC = parseAuxCProperty(data, propContentOffset, propContentSize);
                    std::cout << "      [" << prop.propertyIndex << "] auxC: ";
                    prop.auxC->print();
                    std::cout << "\n";
                    
                } else if (itemPropHeader.type == 0x756E6343) { // 'uncC'
                    prop.uncC = parseUncCProperty(data, propContentOffset, propContentSize);
                    std::cout << "      [" << prop.propertyIndex << "] ★★★ uncC: ";
                    prop.uncC->print();
                    std::cout << " ★★★\n";
                    currentUncC = prop.uncC; // Store for use in parsing other properties
                    
                } else if (itemPropHeader.type == 0x69636566) { // 'icef'
                    prop.icef = parseIcefProperty(data, propContentOffset, propContentSize,
                                currentUncC.has_value() ? &currentUncC.value() : nullptr);
                    std::cout << "      [" << prop.propertyIndex << "] icef: " 
                              << prop.icef->units.size() << " tile offsets\n";
                } else if (itemPropHeader.type == 0x74696C43) { // 'tilC'
                    std::cout << "      Found tilC box:\n";
                    std::cout << "        Box size from header: " << itemPropHeader.size << "\n";
                    std::cout << "        Header size: " << itemPropHeader.headerSize << "\n";
                    std::cout << "        Content size: " << (itemPropHeader.size - itemPropHeader.headerSize) << "\n";
                    
                    // The content size should be passed to the parser, not the full box size
                    size_t actualContentSize = itemPropHeader.size - itemPropHeader.headerSize;
                    
                    prop.tili = parseTilCProperty(data, propContentOffset, actualContentSize);
                    prop.propertyType = "tilC";
                    std::cout << "      [" << prop.propertyIndex << "] ★★★ tilC: ";
                    if (prop.tili) prop.tili->print();
                    std::cout << " ★★★\n";
                } else {
                    std::cout << "      [" << prop.propertyIndex << "] " 
                              << prop.propertyType << " (not parsed)\n";
                }
                
                info.propertyContainer.push_back(prop);
                ipcoPos += itemPropHeader.size;
            }
            
        } else if (propHeader.type == 0x69706D61) { // 'ipma'
            parseIpmaBox(data, dataSize, pos + propHeader.headerSize, 
                        propHeader.size - propHeader.headerSize, 
                        info, info.propertyContainer);
        } else {
            std::cout << "    Skipping box '" << fourCCToString(propHeader.type) 
                      << "' of size " << propHeader.size << "\n";
        }
        
        pos += propHeader.size;
    }
}

// Parse dinf (data information) box
void parseDinfBox(const uint8_t* data, size_t dataSize, size_t offset, 
                  size_t size, MetaBoxInfo& info) {
    std::cout << "  Data Information Box\n";
    
    size_t pos = offset;
    size_t endPos = offset + size;
    
    // Look for dref (data reference) box
    while (pos + 8 <= endPos && pos < dataSize) {
        BoxHeader drefHeader = parseBoxHeader(data, dataSize, pos);
        
        if (drefHeader.type == 0x64726566) { // 'dref'
            size_t drefPos = pos + drefHeader.headerSize;
            drefPos += 4; // Skip version and flags
            
            uint32_t entryCount = readBE32(data + drefPos);
            drefPos += 4;
            
            std::cout << "    Data Reference Box: " << entryCount << " entries\n";
            
            uint16_t refIndex = 1;
            while (drefPos + 8 <= pos + drefHeader.size && drefPos < dataSize) {
                BoxHeader entryHeader = parseBoxHeader(data, dataSize, drefPos);
                
                DataInformation dinfo;
                dinfo.dataReferenceIndex = refIndex++;
                
                size_t entryPos = drefPos + entryHeader.headerSize;
                uint32_t flags = readBE32(data + entryPos - 4) & 0x00FFFFFF;
                
                if (entryHeader.type == 0x75726C20) { // 'url '
                    // Skip version and flags (already read)
                    if (!(flags & 1) && entryPos < drefPos + entryHeader.size) {
                        // Location string present
                        while (entryPos < drefPos + entryHeader.size && data[entryPos] != 0) {
                            dinfo.url += static_cast<char>(data[entryPos++]);
                        }
                    }
                } else if (entryHeader.type == 0x75726E20) { // 'urn '
                    // Skip version and flags
                    if (!(flags & 1) && entryPos < drefPos + entryHeader.size) {
                        // Name string
                        std::string name;
                        while (entryPos < drefPos + entryHeader.size && data[entryPos] != 0) {
                            name += static_cast<char>(data[entryPos++]);
                        }
                        entryPos++; // Skip null
                        
                        // Location string
                        while (entryPos < drefPos + entryHeader.size && data[entryPos] != 0) {
                            dinfo.urn += static_cast<char>(data[entryPos++]);
                        }
                    }
                }
                
                info.dataReferences.push_back(dinfo);
                drefPos += entryHeader.size;
            }
        }
        
        pos += drefHeader.size;
    }
}

// Parse idat (item data) box
void parseIdatBox(size_t fileOffset, size_t boxOffset, size_t size, MetaBoxInfo& info) {
    std::cout << "  Item Data Box (idat)\n";
    std::cout << "    File Offset: " << (fileOffset + boxOffset) << "\n";
    std::cout << "    Data Size: " << size << " bytes\n";
    
    info.itemData.present = true;
    info.itemData.offset = fileOffset + boxOffset;
    info.itemData.size = size;
}

// Parse meta box recursively
void parseMetaBoxRecursive(const uint8_t* data, size_t dataSize, size_t offset, 
                           size_t endOffset, size_t fileOffset, MetaBoxInfo& info, int depth) {
    size_t pos = offset;
    
    // Skip version and flags if present (meta is a full box)
    if (pos + 4 <= dataSize) {
        pos += 4;
    }
    
    while (pos + 8 <= endOffset && pos < dataSize) {
        BoxHeader header;
        try {
            header = parseBoxHeader(data, dataSize, pos);
        } catch (...) {
            break;
        }
        
        if (pos + header.size > endOffset) break;
        
        std::string indent(depth * 2, ' ');
        std::cout << indent << "Box: '" << header.typeString() 
                  << "' at offset " << pos << ", size: " << header.size << "\n";
        
        size_t contentOffset = pos + header.headerSize;
        size_t contentSize = header.size - header.headerSize;
        
        // Parse specific boxes
        if (header.type == 0x68646C72) { // 'hdlr'
            parseHdlrBox(data, contentOffset, contentSize, info, depth);
        } else if (header.type == 0x7069746D) { // 'pitm'
            parsePitmBox(data, contentOffset, contentSize, info);
        } else if (header.type == 0x696C6F63) { // 'iloc'
            parseIlocBox(data, contentOffset, contentSize, info);
        } else if (header.type == 0x69696E66) { // 'iinf'
            parseIinfBox(data, dataSize, contentOffset, contentSize, info);
        } else if (header.type == 0x69726566) { // 'iref'
            parseIrefBox(data, dataSize, contentOffset, contentSize, info);
        } else if (header.type == 0x69707270) { // 'iprp'
            parseIprpBox(data, dataSize, contentOffset, contentSize, info);
        } else if (header.type == 0x6772706C) { // 'grpl'
            parseGrplBox(data, dataSize, contentOffset, contentSize, info, info.entityGroups);
        } else if (header.type == 0x64696E66) { // 'dinf'
            parseDinfBox(data, dataSize, contentOffset, contentSize, info);
        } else if (header.type ==  0x6D646174 ) { // 'mdat'
            std::cout << "Box: 'mdat' at offset " << pos 
                    << ", header size: " << header.headerSize 
                    << ", data starts at: " << (pos + header.headerSize) << "\n";
            info.mdatOffset = pos + header.headerSize;  // Add this field to MetaBoxInfo
        } else if (header.type == 0x69646174) { // 'idat'
                std::cout << "Box: 'idat' at offset " << pos 
              << ", header size: " << header.headerSize 
              << ", data starts at: " << (pos + header.headerSize) 
              << ", size: " << contentSize << "\n";
            info.idatOffset = pos + header.headerSize;  // Add this to MetaBoxInfo
            parseIdatBox(fileOffset, pos, contentSize, info);
        }
        
        pos += header.size;
    }
}

// Main parsing function
MetaBoxInfo parseMetaBox(const std::vector<uint8_t>& data, size_t fileOffset = 0) {
    MetaBoxInfo info;
    
    std::cout << "=== Parsing Meta Box ===\n";
    std::cout << "Data size: " << data.size() << " bytes, File offset: " << fileOffset << "\n\n";
    
    if (data.size() < 8) {
        throw std::runtime_error("Data too small for meta box");
    }
    
    // Verify this is a meta box
    BoxHeader metaHeader = parseBoxHeader(data.data(), data.size(), 0);
    if (metaHeader.type != 0x6D657461) {
        throw std::runtime_error("Not a meta box");
    }
    
    // Parse meta box contents
    parseMetaBoxRecursive(data.data(), data.size(), metaHeader.headerSize, 
                         safe_min(metaHeader.size, data.size()), fileOffset, info, 0);
    
    return info;
}

// Build item data requests with pyramid/tile information
std::vector<ItemDataRequest> buildItemDataRequests(const MetaBoxInfo& info) {
    std::vector<ItemDataRequest> requests;
    
    // Build lookup maps
    std::map<uint32_t, ItemInfoEntry> itemInfoMap;
    for (const auto& item : info.items) {
        itemInfoMap[item.itemID] = item;
    }
    
    std::map<uint32_t, ItemLocationEntry> itemLocMap;
    for (const auto& loc : info.itemLocations) {
        itemLocMap[loc.itemID] = loc;
    }
    
    std::map<uint32_t, ItemPropertyAssociation> itemPropMap;
    for (const auto& assoc : info.itemPropertyAssociations) {
        itemPropMap[assoc.itemID] = assoc;
    }
    
    // Identify thumbnails and auxiliary images
    std::set<uint32_t> thumbnailIDs;
    std::set<uint32_t> auxiliaryIDs;
    std::map<uint32_t, std::string> auxiliaryTypeMap;
    std::map<uint32_t, uint32_t> derivedFromMap; // derived -> base mapping
    
    for (const auto& ref : info.itemReferences) {
        if (ref.referenceType == "thmb") {
            for (uint32_t id : ref.toItemIDs) {
                thumbnailIDs.insert(id);
            }
        } else if (ref.referenceType == "auxl") {
            for (uint32_t id : ref.toItemIDs) {
                auxiliaryIDs.insert(id);
            }
        } else if (ref.referenceType == "dimg") {
            // Derived image - could be pyramid level
            for (uint32_t id : ref.toItemIDs) {
                derivedFromMap[id] = ref.fromItemID;
            }
        }
    }
    
    // Build requests for all items with locations
    for (const auto& [itemID, loc] : itemLocMap) {
        if (itemInfoMap.find(itemID) == itemInfoMap.end()) continue;
        
        ItemDataRequest req;
        req.itemID = itemID;
        req.itemType = itemInfoMap[itemID].itemType;
        req.name = itemInfoMap[itemID].itemName;
        req.offset = loc.getAbsoluteOffset();
        req.byteCount = loc.getTotalSize();
        req.isThumbnail = thumbnailIDs.count(itemID) > 0;
        req.isAuxiliary = auxiliaryIDs.count(itemID) > 0;
        req.auxType = "";
        req.pyramidLevel = 0;
        req.usesIdat = (loc.constructionMethod == 1);
        req.width = 0;
        req.height = 0;
        
        // Get properties if available
        if (itemPropMap.find(itemID) != itemPropMap.end()) {
            for (const auto& prop : itemPropMap[itemID].properties) {
                if (prop.ispe) {
                    req.width = prop.ispe->width;
                    req.height = prop.ispe->height;
                }
                if (prop.pixi) {
                    req.bitsPerChannel = prop.pixi->bitsPerChannel;
                }
                if (prop.colr) {
                    req.colourInfo = prop.colr;
                }
                if (prop.auxC) {
                    req.auxType = prop.auxC->auxType;
                }
                if (prop.uncC) {
                    req.uncompressedConfig = prop.uncC;
                }
            }
        }
        
        // Determine pyramid level (based on derivation chain)
        uint32_t currentID = itemID;
        while (derivedFromMap.find(currentID) != derivedFromMap.end()) {
            req.pyramidLevel++;
            currentID = derivedFromMap[currentID];
            if (req.pyramidLevel > 10) break; // Safety limit
        }
        
        // Adjust offset if using idat
        if (req.usesIdat && info.itemData.present) {
            req.offset = info.itemData.offset + req.offset;
        }
        
        requests.push_back(req);
    }
    
    // Sort by pyramid level (full resolution first) then by item ID
    std::sort(requests.begin(), requests.end(), 
              [](const ItemDataRequest& a, const ItemDataRequest& b) {
                  if (a.pyramidLevel != b.pyramidLevel) 
                      return a.pyramidLevel < b.pyramidLevel;
                  return a.itemID < b.itemID;
              });
    
    return requests;
}

// Add this to your meta box parser section

// Calculate tile grid information from uncC
struct TileGrid {
    uint32_t numCols;
    uint32_t numRows;
    uint32_t tileWidth;
    uint32_t tileHeight;
    uint32_t imageWidth;
    uint32_t imageHeight;
    
    // Calculate tile position
    void getTilePosition(uint32_t tileIndex, uint32_t& row, uint32_t& col) const {
        row = tileIndex / numCols;
        col = tileIndex % numCols;
    }
    
    // Calculate tile spatial extent
    void getTileExtent(uint32_t row, uint32_t col, 
                       uint32_t& x, uint32_t& y, 
                       uint32_t& w, uint32_t& h) const {
        x = col * tileWidth;
        y = row * tileHeight;
        w = std::min(tileWidth, imageWidth - x);
        h = std::min(tileHeight, imageHeight - y);
    }
};

TileGrid calculateTileGrid(uint32_t imageWidth, uint32_t imageHeight,
                           const UncompressedConfig& uncC) {
    TileGrid grid;
    grid.imageWidth = imageWidth;
    grid.imageHeight = imageHeight;
    grid.numCols = uncC.numTileColsMinus1 + 1;
    grid.numRows = uncC.numTileRowsMinus1 + 1;
    
    // Calculate tile dimensions
    grid.tileWidth = (imageWidth + grid.numCols - 1) / grid.numCols;
    grid.tileHeight = (imageHeight + grid.numRows - 1) / grid.numRows;
    
    return grid;
}

// Add to meta box parser section

std::string detectCompression(const std::string& itemType, 
                              const ItemPropertyAssociation* props) {
    // Check item type
    if (itemType == "unci") return "none";
    if (itemType == "jpeg") return "jpeg";
    if (itemType == "hvc1" || itemType == "hev1") return "hevc";
    if (itemType == "av01") return "av1";
    
    // Check for compression properties in iprp
    // You may need to add parsing for compression-specific properties
    
    // Check for deflate/zlib in content encoding
    // This would come from infe box content_encoding field
    
    return "unknown";
}

// Add this BEFORE the extractAllTiles function
// Helper to debug the item structure

void debugItemStructure(const MetaBoxInfo& info) {
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Debug: Item Structure Analysis                    ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    std::cout << "Total Items: " << info.items.size() << "\n";
    std::cout << "Items with Location: " << info.itemLocations.size() << "\n";
    std::cout << "References: " << info.itemReferences.size() << "\n";
    std::cout << "Primary Item ID: " << (info.hasPrimaryItem ? std::to_string(info.primaryItemID) : "none") << "\n\n";
    
    // Show all item locations first
    std::cout << "Item Locations (iloc):\n";
    std::cout << "─────────────────────────────────────────────────────────────\n";
    for (const auto& loc : info.itemLocations) {
        std::cout << "Item ID " << loc.itemID << ":\n";
        std::cout << "  Construction Method: " << loc.constructionMethod << "\n";
        std::cout << "  Base Offset: " << loc.baseOffset << "\n";
        std::cout << "  Extents: " << loc.extents.size() << "\n";
        for (size_t i = 0; i < loc.extents.size(); ++i) {
            std::cout << "    [" << i << "] offset=" << loc.extents[i].first 
                      << ", length=" << loc.extents[i].second << "\n";
        }
        std::cout << "  Total Size: " << loc.getTotalSize() << " bytes\n";
        std::cout << "  Absolute Offset: " << loc.getAbsoluteOffset() << " bytes\n\n";
    }
    
    // Show all references
    std::cout << "Item References (iref):\n";
    std::cout << "─────────────────────────────────────────────────────────────\n";
    for (const auto& ref : info.itemReferences) {
        std::cout << "Reference Type: '" << ref.referenceType << "'\n";
        std::cout << "  From Item: " << ref.fromItemID << "\n";
        std::cout << "  To Items: [";
        for (size_t i = 0; i < ref.toItemIDs.size(); ++i) {
            if (i > 0) std::cout << ", ";
            std::cout << ref.toItemIDs[i];
        }
        std::cout << "]\n\n";
    }
    
    // Build reference maps
    std::map<uint32_t, std::vector<uint32_t>> referencesFrom; // from -> [to list]
    std::map<uint32_t, std::vector<std::pair<std::string, uint32_t>>> referencesTo; // to -> [(type, from) list]
    
    for (const auto& ref : info.itemReferences) {
        for (uint32_t toID : ref.toItemIDs) {
            referencesFrom[ref.fromItemID].push_back(toID);
            referencesTo[toID].push_back({ref.referenceType, ref.fromItemID});
        }
    }
    
    // Show all items with their relationships
    std::cout << "Item Information (iinf):\n";
    std::cout << "─────────────────────────────────────────────────────────────\n";
    
    std::set<uint32_t> seenIDs;
    
    for (const auto& item : info.items) {
        // Check for duplicate IDs
        if (seenIDs.count(item.itemID)) {
            std::cout << "WARNING: Duplicate Item ID " << item.itemID << " detected!\n";
        }
        seenIDs.insert(item.itemID);
        
        std::cout << "Item ID " << item.itemID << ": '" << item.itemType << "'";
        if (!item.itemName.empty()) {
            std::cout << " [\"" << item.itemName << "\"]";
        }
        std::cout << "\n";
        
        // Check if it has location
        bool hasLocation = false;
        for (const auto& loc : info.itemLocations) {
            if (loc.itemID == item.itemID) {
                hasLocation = true;
                std::cout << "  Has location: " << loc.getTotalSize() << " bytes";
                if (loc.extents.size() > 1) {
                    std::cout << " (" << loc.extents.size() << " extents)";
                }
                std::cout << "\n";
                break;
            }
        }
        if (!hasLocation) {
            std::cout << "  No location data\n";
        }
        
        // Check if it has properties
        for (const auto& assoc : info.itemPropertyAssociations) {
            if (assoc.itemID == item.itemID) {
                std::cout << "  Properties: " << assoc.properties.size() << "\n";
                
                // Show key properties
                for (const auto& prop : assoc.properties) {
                    if (prop.ispe) {
                        std::cout << "    - ispe: " << prop.ispe->width << "×" << prop.ispe->height << "\n";
                    }
                    if (prop.uncC) {
                        uint32_t cols = prop.uncC->numTileColsMinus1 + 1;
                        uint32_t rows = prop.uncC->numTileRowsMinus1 + 1;
                        std::cout << "    - uncC: " << cols << "×" << rows << " tile grid\n";
                    }
                    if (prop.pixi) {
                        std::cout << "    - pixi: ";
                        for (size_t i = 0; i < prop.pixi->bitsPerChannel.size(); ++i) {
                            if (i > 0) std::cout << ", ";
                            std::cout << (int)prop.pixi->bitsPerChannel[i];
                        }
                        std::cout << " bits\n";
                    }
                }
                break;
            }
        }
        
        // Show references FROM this item
        if (referencesFrom.find(item.itemID) != referencesFrom.end()) {
            std::cout << "  References TO: [";
            for (size_t i = 0; i < referencesFrom[item.itemID].size(); ++i) {
                if (i > 0) std::cout << ", ";
                std::cout << referencesFrom[item.itemID][i];
            }
            std::cout << "]\n";
        }
        
        // Show references TO this item
        if (referencesTo.find(item.itemID) != referencesTo.end()) {
            std::cout << "  Referenced BY: ";
            for (size_t i = 0; i < referencesTo[item.itemID].size(); ++i) {
                if (i > 0) std::cout << ", ";
                std::cout << "Item " << referencesTo[item.itemID][i].second 
                          << " ('" << referencesTo[item.itemID][i].first << "')";
            }
            std::cout << "\n";
        }
        
        std::cout << "\n";
    }
}


// COMPLETE REWRITE of extractAllTiles function


std::vector<TileInfo> extractAllTiles(const MetaBoxInfo& info) {
    std::vector<TileInfo> tiles;
    
    // Build lookup maps
    std::map<uint32_t, ItemInfoEntry> itemInfoMap;
    for (const auto& item : info.items) {
        itemInfoMap[item.itemID] = item;
    }
    
    std::map<uint32_t, ItemLocationEntry> itemLocMap;
    for (const auto& loc : info.itemLocations) {
        itemLocMap[loc.itemID] = loc;
    }
    
    std::map<uint32_t, ItemPropertyAssociation> itemPropMap;
    for (const auto& assoc : info.itemPropertyAssociations) {
        itemPropMap[assoc.itemID] = assoc;
    }
    
    // Identify pyramid structure from "base" references
    std::map<uint32_t, std::vector<uint32_t>> pyramidLevels; // parent -> [level items]
    std::map<uint32_t, uint32_t> itemToParent; // item -> parent
    
    for (const auto& ref : info.itemReferences) {
        if (ref.referenceType == "base") {
            // fromItemID references toItemIDs as base levels
            for (uint32_t baseID : ref.toItemIDs) {
                pyramidLevels[baseID].push_back(ref.fromItemID);
                itemToParent[ref.fromItemID] = baseID;
            }
        }
    }
    
    std::cout << "\nPyramid Structure Analysis:\n";
    std::cout << "Found " << pyramidLevels.size() << " pyramid parent(s)\n";
    
    for (const auto& [parentID, levels] : pyramidLevels) {
        std::cout << "  Parent " << parentID << " has " << levels.size() << " level(s)\n";
    }
    
    // Check if items have multiple extents (tiles within each level)
    std::cout << "\nChecking for tiled levels:\n";
    
    for (const auto& [itemID, loc] : itemLocMap) {
        if (itemInfoMap.find(itemID) == itemInfoMap.end()) continue;
        
        const auto& itemInfo = itemInfoMap[itemID];
        
        std::cout << "Item " << itemID << " ('" << itemInfo.itemType << "'): ";
        std::cout << loc.extents.size() << " extent(s), ";
        std::cout << "total size: " << loc.getTotalSize() << " bytes\n";
        
        if (loc.extents.size() > 1) {
            std::cout << "  -> This appears to be a tiled item!\n";
        }
        
        // Get properties
        uint32_t width = 0, height = 0;
        std::optional<UncompressedConfig> uncC;
        std::vector<uint8_t> bitsPerChannel;
        std::optional<ColourInformation> colourInfo;
        
        if (itemPropMap.find(itemID) != itemPropMap.end()) {
            for (const auto& prop : itemPropMap[itemID].properties) {
                if (prop.ispe) {
                    width = prop.ispe->width;
                    height = prop.ispe->height;
                }
                if (prop.uncC) {
                    uncC = prop.uncC;
                }
                if (prop.pixi) {
                    bitsPerChannel = prop.pixi->bitsPerChannel;
                }
                if (prop.colr) {
                    colourInfo = prop.colr;
                }
            }
        }
        
        // Determine pyramid level
        int pyramidLevel = 0;
        
        // If this item is in pyramidLevels as a key, it's the base (level 0)
        if (pyramidLevels.find(itemID) != pyramidLevels.end()) {
            pyramidLevel = 0;
            std::cout << "  -> This is a pyramid parent (base level)\n";
        }
        // If this item is in itemToParent, find its depth
        else if (itemToParent.find(itemID) != itemToParent.end()) {
            uint32_t current = itemID;
            while (itemToParent.find(current) != itemToParent.end()) {
                pyramidLevel++;
                current = itemToParent[current];
                if (pyramidLevel > 10) break; // Safety
            }
            std::cout << "  -> This is pyramid level " << pyramidLevel << "\n";
        }
        
        // Calculate tile grid
        TileGrid grid;
        grid.imageWidth = width;
        grid.imageHeight = height;
        
        if (uncC.has_value() && (uncC->numTileColsMinus1 > 0 || uncC->numTileRowsMinus1 > 0)) {
            grid.numCols = uncC->numTileColsMinus1 + 1;
            grid.numRows = uncC->numTileRowsMinus1 + 1;
            grid.tileWidth = width / grid.numCols;
            grid.tileHeight = height / grid.numRows;
            
            std::cout << "  -> Grid from uncC: " << grid.numCols << "×" << grid.numRows 
                      << ", tile size: " << grid.tileWidth << "×" << grid.tileHeight << "\n";
        } else if (loc.extents.size() > 1) {
            // Estimate grid from extent count
            uint32_t numTiles = loc.extents.size();
            grid.numCols = static_cast<uint32_t>(std::sqrt(numTiles));
            grid.numRows = (numTiles + grid.numCols - 1) / grid.numCols;
            grid.tileWidth = width / grid.numCols;
            grid.tileHeight = height / grid.numRows;
            
            std::cout << "  -> Grid estimated: " << grid.numCols << "×" << grid.numRows 
                      << ", tile size: " << grid.tileWidth << "×" << grid.tileHeight << "\n";
        } else {
            // Single extent
            grid.numCols = 1;
            grid.numRows = 1;
            grid.tileWidth = width;
            grid.tileHeight = height;
        }
        
        // Process each extent as a tile
        for (size_t extentIdx = 0; extentIdx < loc.extents.size(); ++extentIdx) {
            TileInfo tile;
            tile.itemID = itemID;
            tile.tileIndex = extentIdx;
            tile.pyramidLevel = pyramidLevel;
            tile.tilingScheme = (itemInfo.itemType == "unci") ? "unci" : "unknown";
            tile.bitsPerChannel = bitsPerChannel;
            tile.colourInfo = colourInfo;
            tile.uncompressedConfig = uncC;
            
            // Calculate position
            if (loc.extents.size() > 1) {
                grid.getTilePosition(extentIdx, tile.row, tile.col);
                uint32_t x, y, w, h;
                grid.getTileExtent(tile.row, tile.col, x, y, w, h);
                tile.xOffset = x;
                tile.yOffset = y;
                tile.width = w;
                tile.height = h;
            } else {
                tile.row = 0;
                tile.col = 0;
                tile.xOffset = 0;
                tile.yOffset = 0;
                tile.width = width;
                tile.height = height;
            }
            
            // Detect compression
            tile.compression = detectCompression(itemInfo.itemType, 
                                                 itemPropMap.find(itemID) != itemPropMap.end() 
                                                 ? &itemPropMap[itemID] : nullptr);
            
            // Check for deflate in item name
            if (!itemInfo.itemName.empty()) {
                std::string nameLower = itemInfo.itemName;
                std::transform(nameLower.begin(), nameLower.end(), nameLower.begin(), ::tolower);
                if (nameLower.find("deflate") != std::string::npos || 
                    nameLower.find("defl") != std::string::npos) {
                    tile.compression = "deflate";
                }
            }
            
            if (itemInfo.itemType == "unci") {
                // unci items might be compressed or uncompressed
                // Check if there's a compression indicator
                if (tile.compression == "unknown") {
                    // Assume uncompressed unless indicated otherwise
                    tile.compression = "none";
                }
            }
            
            // Calculate byte offset
            uint64_t absOffset = loc.baseOffset + loc.extents[extentIdx].first;
            if (loc.constructionMethod == 1 && info.itemData.present) {
                absOffset = info.itemData.offset + absOffset;
            }
            tile.byteOffset = absOffset;
            tile.byteCount = loc.extents[extentIdx].second;
            
            tiles.push_back(tile);
        }
    }
    
    std::cout << "\nTotal tiles extracted: " << tiles.size() << "\n";
    
    // Sort by pyramid level, then row/col
    std::sort(tiles.begin(), tiles.end(), 
              [](const TileInfo& a, const TileInfo& b) {
                  if (a.pyramidLevel != b.pyramidLevel) 
                      return a.pyramidLevel < b.pyramidLevel;
                  if (a.row != b.row) 
                      return a.row < b.row;
                  return a.col < b.col;
              });
    
    return tiles;
}

// Special handler for unci multi-level structure with single iloc entry

std::vector<TileInfo> extractTilesFromUnciMultiLevel(const MetaBoxInfo& info) {
    std::vector<TileInfo> tiles;
    
    std::cout << "\n=== Special Handler: unci Multi-Level Structure ===\n";
    
    // We have 5 ispe properties representing pyramid levels
    // Properties come in groups of 3-4 for each level
    
    struct LevelInfo {
        int level;
        uint32_t width;
        uint32_t height;
        std::optional<UncompressedConfig> uncC;
    };
    
    std::vector<LevelInfo> levels;
    
    // Parse properties to identify levels
    // Pattern: uncC, cmpd, ispe, cmpC, icef (repeated)
    for (size_t i = 0; i < info.propertyContainer.size(); ++i) {
        const auto& prop = info.propertyContainer[i];
        
        if (prop.ispe) {
            LevelInfo level;
            level.width = prop.ispe->width;
            level.height = prop.ispe->height;
            level.level = levels.size();
            
            // Look for corresponding uncC (should be a few properties earlier)
            for (int j = std::max(0, (int)i - 5); j < (int)i; ++j) {
                if (info.propertyContainer[j].uncC) {
                    level.uncC = info.propertyContainer[j].uncC;
                    break;
                }
            }
            
            levels.push_back(level);
            
            std::cout << "Level " << level.level << ": " << level.width << "×" << level.height;
            if (level.uncC) {
                uint32_t cols = level.uncC->numTileColsMinus1 + 1;
                uint32_t rows = level.uncC->numTileRowsMinus1 + 1;
                std::cout << " [" << cols << "×" << rows << " tiles]";
            }
            std::cout << "\n";
        }
    }
    
    // Now we need to figure out how tiles are stored in mdat
    // With 0 extents in iloc, the tiles must be stored sequentially in mdat
    
    if (info.itemLocations.empty()) {
        std::cout << "No item locations found\n";
        return tiles;
    }
    
    const auto& mainLoc = info.itemLocations[0];
    uint64_t mdatStart = mainLoc.baseOffset;
    
    std::cout << "\nmdat data starts at offset: " << mdatStart << "\n";
    std::cout << "Total file size should be around: " << (mdatStart + 313920009) << " bytes\n";
    
    // Calculate tile sizes for each level
    for (const auto& level : levels) {
        uint32_t numCols = 1, numRows = 1;
        
        if (level.uncC) {
            numCols = level.uncC->numTileColsMinus1 + 1;
            numRows = level.uncC->numTileRowsMinus1 + 1;
        }
        
        uint32_t numTiles = numCols * numRows;
        uint32_t tileWidth = level.width / numCols;
        uint32_t tileHeight = level.height / numRows;
        
        // Estimate tile size (this is a guess - actual size depends on compression)
        // For uncompressed RGB: width * height * 3 bytes
        // For deflate: could be much smaller
        
        std::cout << "\nLevel " << level.level << " (" << level.width << "×" << level.height << "):\n";
        std::cout << "  Grid: " << numCols << "×" << numRows << " = " << numTiles << " tiles\n";
        std::cout << "  Tile dimensions: " << tileWidth << "×" << tileHeight << "\n";
        
        // We can't determine exact tile byte offsets without reading the file
        // or having more metadata, but we can create placeholder entries
        
        std::cout << "  WARNING: Cannot determine exact tile byte offsets without reading file structure\n";
        std::cout << "  Tiles are likely stored sequentially in mdat box\n";
    }
    
    std::cout << "\n=== SOLUTION REQUIRED ===\n";
    std::cout << "Your HEIF file uses a structure where:\n";
    std::cout << "1. All pyramid levels share Item ID 0\n";
    std::cout << "2. Tile data is stored in mdat (313 MB)\n";
    std::cout << "3. Individual tile offsets are NOT in iloc\n";
    std::cout << "\nTo extract tiles, you need to:\n";
    std::cout << "A. Parse the grpl (entity group) box to find tile organization\n";
    std::cout << "B. Or read the mdat box structure directly\n";
    std::cout << "C. Or use the HEIF writing tool's metadata to understand tile layout\n";
    
    return tiles;
}



// COMPLETE SOLUTION: Parse libheif UNCI tiled pyramid files

// Add after the LibheifTileInfo structure

struct LibheifTileInfo {
    uint32_t itemID;           // Parent item ID (1-5)
    int pyramidLevel;          // 0 = highest res
    uint32_t imageWidth;
    uint32_t imageHeight;
    uint32_t numCols;
    uint32_t numRows;
    uint32_t tileWidth;
    uint32_t tileHeight;
    std::vector<TileInfo> tiles;  // Individual tile metadata
};

std::vector<LibheifTileInfo> extractLibheifUnciTiles(const MetaBoxInfo& info, 
                                                      const std::string& filepath,
                                                      bool isFile) {
    std::vector<LibheifTileInfo> result;
    
    std::cout << "\n=== Extracting libheif UNCI Tiled Pyramids ===\n";
    
    // Get pyramid entity group
    std::vector<uint32_t> pyramidItemIDs;
    for (const auto& group : info.entityGroups) {
        if (group.groupType == "pymd") {
            pyramidItemIDs = group.entityIDs;
            std::cout << "Found pyramid group with " << pyramidItemIDs.size() << " levels: [";
            for (size_t i = 0; i < pyramidItemIDs.size(); ++i) {
                if (i > 0) std::cout << ", ";
                std::cout << pyramidItemIDs[i];
            }
            std::cout << "]\n";
            break;
        }
    }
    
    // IGNORE pyramid group IDs - use actual item IDs from property associations!
    std::cout << "\nUsing item IDs from property associations instead...\n";
    
    // Build map of item ID to dimensions from property associations
    struct ItemDimensions {
        uint32_t itemID;
        uint32_t width;
        uint32_t height;
        std::optional<UncompressedConfig> uncC;
    };
    
    std::vector<ItemDimensions> itemDims;
    
    for (const auto& assoc : info.itemPropertyAssociations) {
        ItemDimensions dims;
        dims.itemID = assoc.itemID;
        dims.width = 0;
        dims.height = 0;
        
        for (const auto& prop : assoc.properties) {
            if (prop.ispe) {
                dims.width = prop.ispe->width;
                dims.height = prop.ispe->height;
            }
            if (prop.uncC) {
                dims.uncC = prop.uncC;
            }
        }
        
        if (dims.width > 0 && dims.height > 0) {
            itemDims.push_back(dims);
            std::cout << "  Item " << dims.itemID << ": " << dims.width << "×" << dims.height;
            if (dims.uncC && (dims.uncC->numTileColsMinus1 > 0 || dims.uncC->numTileRowsMinus1 > 0)) {
                uint32_t cols = dims.uncC->numTileColsMinus1 + 1;
                uint32_t rows = dims.uncC->numTileRowsMinus1 + 1;
                std::cout << " [" << cols << "×" << rows << " tiles]";
            }
            std::cout << "\n";
        }
    }
    
    // Sort by size (largest first = level 0)
    std::sort(itemDims.begin(), itemDims.end(), 
              [](const ItemDimensions& a, const ItemDimensions& b) {
                  uint64_t areaA = static_cast<uint64_t>(a.width) * a.height;
                  uint64_t areaB = static_cast<uint64_t>(b.width) * b.height;
                  return areaA > areaB;
              });
    
    std::cout << "\nProcessing " << itemDims.size() << " pyramid levels (sorted by size)...\n";
    
    // Process each level
    for (size_t levelIdx = 0; levelIdx < itemDims.size(); ++levelIdx) {
        const auto& dims = itemDims[levelIdx];
        
        LibheifTileInfo levelInfo;
        levelInfo.itemID = dims.itemID;
        levelInfo.pyramidLevel = levelIdx;
        levelInfo.imageWidth = dims.width;
        levelInfo.imageHeight = dims.height;
        levelInfo.tileWidth = 256;  // Known from heif-info
        levelInfo.tileHeight = 256;
        
        std::cout << "\n--- Level " << levelIdx << " (Item ID " << dims.itemID << ") ---\n";
        std::cout << "Dimensions: " << levelInfo.imageWidth << "×" << levelInfo.imageHeight << "\n";
        
        // Get tile grid from uncC or calculate
        if (dims.uncC && (dims.uncC->numTileColsMinus1 > 0 || dims.uncC->numTileRowsMinus1 > 0)) {
            levelInfo.numCols = dims.uncC->numTileColsMinus1 + 1;
            levelInfo.numRows = dims.uncC->numTileRowsMinus1 + 1;
            std::cout << "Tile grid from uncC: " << levelInfo.numCols << "×" << levelInfo.numRows << "\n";
        } else {
            levelInfo.numCols = (levelInfo.imageWidth + levelInfo.tileWidth - 1) / levelInfo.tileWidth;
            levelInfo.numRows = (levelInfo.imageHeight + levelInfo.tileHeight - 1) / levelInfo.tileHeight;
            std::cout << "Calculated tile grid: " << levelInfo.numCols << "×" << levelInfo.numRows << "\n";
        }
        
        uint32_t numTiles = levelInfo.numCols * levelInfo.numRows;
        std::cout << "Total tiles: " << numTiles << "\n";
        
        // Check if this item has location data
        bool foundLocation = false;
        for (const auto& loc : info.itemLocations) {
            if (loc.itemID == dims.itemID) {
                std::cout << "Found iloc entry for item " << loc.itemID << " with " << loc.extents.size() << " extents\n";
                
                for (size_t i = 0; i < loc.extents.size() && i < numTiles; ++i) {
                    TileInfo tile;
                    tile.itemID = dims.itemID;
                    tile.tileIndex = i;
                    tile.pyramidLevel = levelIdx;
                    tile.tilingScheme = "unci";
                    tile.compression = "deflate";
                    
                    tile.row = i / levelInfo.numCols;
                    tile.col = i % levelInfo.numCols;
                    
                    tile.xOffset = tile.col * levelInfo.tileWidth;
                    tile.yOffset = tile.row * levelInfo.tileHeight;
                    tile.width = std::min(levelInfo.tileWidth, levelInfo.imageWidth - tile.xOffset);
                    tile.height = std::min(levelInfo.tileHeight, levelInfo.imageHeight - tile.yOffset);
                    
                    uint64_t absOffset = loc.baseOffset + loc.extents[i].first;
                    if (loc.constructionMethod == 1 && info.itemData.present) {
                        absOffset = info.itemData.offset + absOffset;
                    }
                    tile.byteOffset = absOffset;
                    tile.byteCount = loc.extents[i].second;
                    
                    levelInfo.tiles.push_back(tile);
                }
                
                foundLocation = true;
                std::cout << "Extracted " << levelInfo.tiles.size() << " tile locations\n";
                break;
            }
        }
        
        // If no location found, create placeholder tiles
        if (!foundLocation) {
            std::cout << "No iloc entry found for this item - creating placeholders\n";
            
            for (uint32_t i = 0; i < numTiles; ++i) {
                TileInfo tile;
                tile.itemID = dims.itemID;
                tile.tileIndex = i;
                tile.pyramidLevel = levelIdx;
                tile.tilingScheme = "unci";
                tile.compression = "deflate";
                
                tile.row = i / levelInfo.numCols;
                tile.col = i % levelInfo.numCols;
                tile.xOffset = tile.col * levelInfo.tileWidth;
                tile.yOffset = tile.row * levelInfo.tileHeight;
                tile.width = std::min(levelInfo.tileWidth, levelInfo.imageWidth - tile.xOffset);
                tile.height = std::min(levelInfo.tileHeight, levelInfo.imageHeight - tile.yOffset);
                
                tile.byteOffset = 0;
                tile.byteCount = 0;
                
                levelInfo.tiles.push_back(tile);
            }
        }
        
        result.push_back(levelInfo);
    }
    
    return result;
}

// Updated: Extract tiles using icef offsets

std::vector<LibheifTileInfo> extractLibheifUnciTilesWithOffsets(
    const MetaBoxInfo& info, 
    const std::string& filepath,
    bool isFile) {
    
    std::vector<LibheifTileInfo> result;
    
    std::cout << "\n=== Extracting UNCI Tiles with icef Offsets ===\n";
    
    // Build item dimensions map
    struct ItemDimensions {
        uint32_t itemID;
        uint32_t width;
        uint32_t height;
        std::optional<UncompressedConfig> uncC;
        std::optional<IcefBox> icef;
    };
    
    std::vector<ItemDimensions> itemDims;
    
    for (const auto& assoc : info.itemPropertyAssociations) {
        ItemDimensions dims;
        dims.itemID = assoc.itemID;
        dims.width = 0;
        dims.height = 0;
        
        for (const auto& prop : assoc.properties) {
            if (prop.ispe) {
                dims.width = prop.ispe->width;
                dims.height = prop.ispe->height;
            }
            if (prop.uncC) {
                dims.uncC = prop.uncC;
            }
            if (prop.icef) {
                dims.icef = prop.icef;
            }
        }
        
        if (dims.width > 0) {
            itemDims.push_back(dims);
            std::cout << "  Item " << dims.itemID << ": " << dims.width << "×" << dims.height;
            if (dims.icef) {
                std::cout << " [" << dims.icef->units.size() << " icef units]";
            }
            std::cout << "\n";
        }
    }
    
    // Sort by size
    std::sort(itemDims.begin(), itemDims.end(), 
              [](const ItemDimensions& a, const ItemDimensions& b) {
                  return (uint64_t)a.width * a.height > (uint64_t)b.width * b.height;
              });
    
    // Get mdat offset
    uint64_t mdatDataStart = 6555; // From iloc base_offset
    
    // Process each level
    for (size_t levelIdx = 0; levelIdx < itemDims.size(); ++levelIdx) {
        const auto& dims = itemDims[levelIdx];
        
        LibheifTileInfo levelInfo;
        levelInfo.itemID = dims.itemID;
        levelInfo.pyramidLevel = levelIdx;
        levelInfo.imageWidth = dims.width;
        levelInfo.imageHeight = dims.height;
        levelInfo.tileWidth = 256;
        levelInfo.tileHeight = 256;
        
        levelInfo.numCols = (dims.width + 255) / 256;
        levelInfo.numRows = (dims.height + 255) / 256;
        
        std::cout << "\nLevel " << levelIdx << " (Item " << dims.itemID << "):\n";
        std::cout << "  Grid: " << levelInfo.numCols << "×" << levelInfo.numRows << "\n";
        
        uint32_t numTiles = levelInfo.numCols * levelInfo.numRows;
        
        // Use icef offsets if available
        if (dims.icef && dims.icef->units.size() == numTiles) {
            std::cout << "  Using icef offsets for " << numTiles << " tiles\n";
            
            for (uint32_t i = 0; i < numTiles; ++i) {
                TileInfo tile;
                tile.itemID = dims.itemID;
                tile.tileIndex = i;
                tile.pyramidLevel = levelIdx;
                tile.tilingScheme = "unci";
                tile.compression = "deflate";
                
                tile.row = i / levelInfo.numCols;
                tile.col = i % levelInfo.numCols;
                tile.xOffset = tile.col * 256;
                tile.yOffset = tile.row * 256;
                tile.width = std::min(256u, dims.width - tile.xOffset);
                tile.height = std::min(256u, dims.height - tile.yOffset);
                
                // Get offset from icef
                tile.byteOffset = mdatDataStart + dims.icef->units[i].unit_offset;
                tile.byteCount = dims.icef->units[i].unit_size;
                
                levelInfo.tiles.push_back(tile);
            }
            
            std::cout << "  ✓ Extracted " << levelInfo.tiles.size() << " tiles with offsets\n";
        } else {
            std::cout << "  ✗ No icef data or size mismatch\n";
        }
        
        result.push_back(levelInfo);
    }
    
    return result;
}


// Complete tile extraction with absolute byte offsets for grid tiling
std::vector<LibheifTileInfo> extractGridTilesWithOffsets(
    const MetaBoxInfo& info,
    uint64_t mdatBaseOffset = 0) {
    
    std::vector<LibheifTileInfo> result;
    
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Extracting Grid Tiles with Byte Offsets          ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    // Build item dimensions map with properties
    struct ItemDimensions {
        uint32_t itemID;
        uint32_t width;
        uint32_t height;
        std::optional<UncompressedConfig> uncC;
        std::optional<IcefBox> icef;
        const ItemLocationEntry* location;
    };
    
    std::vector<ItemDimensions> itemDims;
    
    // Build lookup map for locations
    std::map<uint32_t, const ItemLocationEntry*> locMap;
    for (const auto& loc : info.itemLocations) {
        locMap[loc.itemID] = &loc;
    }
    
    // Gather all items with properties
    for (const auto& assoc : info.itemPropertyAssociations) {
        ItemDimensions dims;
        dims.itemID = assoc.itemID;
        dims.width = 0;
        dims.height = 0;
        dims.location = nullptr;
        
        // Get location
        if (locMap.find(assoc.itemID) != locMap.end()) {
            dims.location = locMap[assoc.itemID];
        }
        
        // Get properties
        for (const auto& prop : assoc.properties) {
            if (prop.ispe) {
                dims.width = prop.ispe->width;
                dims.height = prop.ispe->height;
            }
            if (prop.uncC) {
                dims.uncC = prop.uncC;
            }
            if (prop.icef) {
                dims.icef = prop.icef;
            }
        }
        
        if (dims.width > 0 && dims.height > 0) {
            itemDims.push_back(dims);
        }
    }
    
    // Sort by size (largest first)
    std::sort(itemDims.begin(), itemDims.end(),
              [](const ItemDimensions& a, const ItemDimensions& b) {
                  return (uint64_t)a.width * a.height > (uint64_t)b.width * b.height;
              });
    
    std::cout << "Found " << itemDims.size() << " items with dimensions\n\n";
    
    // Determine mdat data start offset
    uint64_t mdatDataStart = mdatBaseOffset;
    if (mdatDataStart == 0 && !info.itemLocations.empty()) {
        mdatDataStart = info.itemLocations[0].baseOffset;
    }
    
    std::cout << "Using mdat data start offset: " << mdatDataStart << "\n\n";
    
    // Process each item/level
    for (size_t levelIdx = 0; levelIdx < itemDims.size(); ++levelIdx) {
        const auto& dims = itemDims[levelIdx];
        
        LibheifTileInfo levelInfo;
        levelInfo.itemID = dims.itemID;
        levelInfo.pyramidLevel = levelIdx;
        levelInfo.imageWidth = dims.width;
        levelInfo.imageHeight = dims.height;
        levelInfo.tileWidth = 256;  // Standard tile size
        levelInfo.tileHeight = 256;
        
        // Calculate grid from uncC or estimate
        if (dims.uncC && (dims.uncC->numTileColsMinus1 > 0 || dims.uncC->numTileRowsMinus1 > 0)) {
            levelInfo.numCols = dims.uncC->numTileColsMinus1 + 1;
            levelInfo.numRows = dims.uncC->numTileRowsMinus1 + 1;
        } else {
            levelInfo.numCols = (dims.width + 255) / 256;
            levelInfo.numRows = (dims.height + 255) / 256;
        }
        
        uint32_t numTiles = levelInfo.numCols * levelInfo.numRows;
        
        std::cout << "─────────────────────────────────────────────────────────────\n";
        std::cout << "Level " << levelIdx << " (Item " << dims.itemID << "):\n";
        std::cout << "  Dimensions: " << levelInfo.imageWidth << "×" << levelInfo.imageHeight << "\n";
        std::cout << "  Grid: " << levelInfo.numCols << "×" << levelInfo.numRows << "\n";
        std::cout << "  Total tiles: " << numTiles << "\n";
        
        // METHOD 1: Use icef if available
        if (dims.icef && dims.icef->units.size() == numTiles) {
            std::cout << "  ✓ Using icef offsets\n";
            
            for (uint32_t i = 0; i < numTiles; ++i) {
                TileInfo tile;
                tile.itemID = dims.itemID;
                tile.tileIndex = i;
                tile.pyramidLevel = levelIdx;
                tile.tilingScheme = "grid";
                tile.compression = "deflate";  // Adjust based on actual compression
                
                tile.row = i / levelInfo.numCols;
                tile.col = i % levelInfo.numCols;
                tile.xOffset = tile.col * 256;
                tile.yOffset = tile.row * 256;
                tile.width = std::min(256u, dims.width - tile.xOffset);
                tile.height = std::min(256u, dims.height - tile.yOffset);
                
                // Get absolute offset from icef
                tile.byteOffset = mdatDataStart + dims.icef->units[i].unit_offset;
                tile.byteCount = dims.icef->units[i].unit_size;
                
                levelInfo.tiles.push_back(tile);
            }
            
            std::cout << "  ✓ Extracted " << levelInfo.tiles.size() << " tiles with offsets\n";
            
        // METHOD 2: Use iloc extents
        } else if (dims.location && dims.location->extents.size() == numTiles) {
            std::cout << "  ✓ Using iloc extents\n";
            
            for (uint32_t i = 0; i < numTiles; ++i) {
                TileInfo tile;
                tile.itemID = dims.itemID;
                tile.tileIndex = i;
                tile.pyramidLevel = levelIdx;
                tile.tilingScheme = "grid";
                tile.compression = "deflate";
                
                tile.row = i / levelInfo.numCols;
                tile.col = i % levelInfo.numCols;
                tile.xOffset = tile.col * 256;
                tile.yOffset = tile.row * 256;
                tile.width = std::min(256u, dims.width - tile.xOffset);
                tile.height = std::min(256u, dims.height - tile.yOffset);
                
                // Get offset from iloc
                const auto& extent = dims.location->extents[i];
                tile.byteOffset = dims.location->baseOffset + extent.first;
                tile.byteCount = extent.second;
                
                // Adjust for construction method
                if (dims.location->constructionMethod == 1 && info.itemData.present) {
                    tile.byteOffset = info.itemData.offset + extent.first;
                }
                
                levelInfo.tiles.push_back(tile);
            }
            
            std::cout << "  ✓ Extracted " << levelInfo.tiles.size() << " tiles with offsets\n";
            
        } else {
            std::cout << "  ✗ No offset information available\n";
            std::cout << "    icef units: " << (dims.icef ? dims.icef->units.size() : 0) << "\n";
            std::cout << "    iloc extents: " << (dims.location ? dims.location->extents.size() : 0) << "\n";
        }
        
        result.push_back(levelInfo);
        std::cout << "\n";
    }
    
    return result;
}

// Complete grid tile extraction with byte offsets
std::vector<TileInfo> extractGridTilesComplete(const MetaBoxInfo& info, size_t metaBoxOffset, size_t metaBoxSize) {
    std::vector<TileInfo> tiles;
    
    size_t mdatBaseOffset = metaBoxOffset + metaBoxSize; // Assuming mdat starts immediately after meta box
    
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Complete Grid Tile Extraction                     ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    // Build lookup maps
    std::map<uint32_t, ItemLocationEntry> itemLocMap;
    for (const auto& loc : info.itemLocations) {
        itemLocMap[loc.itemID] = loc;
    }
    
    std::map<uint32_t, ItemPropertyAssociation> itemPropMap;
    for (const auto& assoc : info.itemPropertyAssociations) {
        itemPropMap[assoc.itemID] = assoc;
    }
    
    // Get full image dimensions from ipma entry 0 (grid container)
    uint32_t fullImageWidth = 0;
    uint32_t fullImageHeight = 0;
    uint32_t tileWidth = 256; // Default, will be updated from actual tile properties
    uint32_t tileHeight = 256;
    
    if (!info.itemPropertyAssociations.empty()) {
        const auto& gridProps = info.itemPropertyAssociations[0].properties;
        for (const auto& prop : gridProps) {
            if (prop.ispe) {
                fullImageWidth = prop.ispe->width;
                fullImageHeight = prop.ispe->height;
                std::cout << "Full image dimensions from grid container: " 
                          << fullImageWidth << "×" << fullImageHeight << "\n";
                break;
            }
        }
    }
    
    if (fullImageWidth == 0 || fullImageHeight == 0) {
        std::cerr << "ERROR: Could not determine full image dimensions!\n";
        return tiles;
    }
    
    // Get actual tile size from first tile's ispe property
    if (info.itemPropertyAssociations.size() > 1) {
        const auto& firstTileProps = info.itemPropertyAssociations[1].properties;
        for (const auto& prop : firstTileProps) {
            if (prop.ispe) {
                tileWidth = prop.ispe->width; 
                tileHeight = prop.ispe->height;
                std::cout << "Tile size from first tile: " << tileWidth << "×" << tileHeight << "\n";
                break;
            }
        }
    }
    
    std::cout << "\n";

    std::cout << "Full image: " << fullImageWidth << "×" << fullImageHeight << "\n";
    std::cout << "Tile size: " << tileWidth << "×" << tileHeight << "\n";
    std::cout << "Total tiles in iloc: " << itemLocMap.size() << "\n\n";
    
    // Calculate grid from full image and tile size
    uint32_t numCols = (fullImageWidth + tileWidth - 1) / tileWidth;
    uint32_t numRows = (fullImageHeight + tileHeight - 1) / tileHeight;
    
    std::cout << "Calculated grid: " << numCols << "×" << numRows 
              << " = " << (numCols * numRows) << " tiles\n";
    std::cout << "Actual tiles: " << itemLocMap.size() << "\n\n";
    
    if (numCols * numRows != itemLocMap.size()) {
        std::cout << "⚠ Warning: Expected " << (numCols * numRows) 
                  << " tiles but have " << itemLocMap.size() << "\n\n";
    }
    
    // Create tile entries
    for (const auto& [itemID, loc] : itemLocMap) {
        if (loc.extents.empty()) continue;
        
        TileInfo tile;
        tile.itemID = itemID;
        tile.tileIndex = itemID - 2; // Tiles start at ID 2
        tile.pyramidLevel = 0;
        tile.tilingScheme = "grid";
        tile.compression = "deflate";
        
        // Calculate grid position (row-major order)
        tile.row = tile.tileIndex / numCols;
        tile.col = tile.tileIndex % numCols;
        tile.xOffset = tile.col * tileWidth;
        tile.yOffset = tile.row * tileHeight;
        tile.width = tileWidth;
        tile.height = tileHeight;
        
        // Edge tiles might be smaller
        if (tile.xOffset + tileWidth > fullImageWidth) {
            tile.width = fullImageWidth - tile.xOffset;
        }
        if (tile.yOffset + tileHeight > fullImageHeight) {
            tile.height = fullImageHeight - tile.yOffset;
        }
        
        // Get byte offset and count
        tile.byteOffset = mdatBaseOffset + loc.extents[0].first;
        tile.byteCount = loc.extents[0].second;
        
        // Get tile-specific properties if available
        if (itemPropMap.find(itemID) != itemPropMap.end()) {
            for (const auto& prop : itemPropMap[itemID].properties) {
                if (prop.pixi) {
                    tile.bitsPerChannel = prop.pixi->bitsPerChannel;
                }
                if (prop.colr) {
                    tile.colourInfo = prop.colr;
                }
                if (prop.uncC) {
                    tile.uncompressedConfig = prop.uncC;
                }
            }
        }
        
        tiles.push_back(tile);
    }
    
    // Sort by tile index
    std::sort(tiles.begin(), tiles.end(), 
              [](const TileInfo& a, const TileInfo& b) {
                  return a.tileIndex < b.tileIndex;
              });
    
    std::cout << "✓ Extracted " << tiles.size() << " tiles with complete metadata\n\n";
    
    // Show sample tiles including edge tiles
    std::cout << "Sample tiles:\n";
    std::cout << "First row:\n";
    for (size_t i = 0; i < std::min(size_t(3), tiles.size()); ++i) {
        const auto& tile = tiles[i];
        std::cout << "  Tile " << i << " [" << tile.row << "," << tile.col << "]: "
                  << tile.width << "×" << tile.height << "px at (" 
                  << tile.xOffset << "," << tile.yOffset << "), "
                  << tile.byteCount << " bytes @ offset " << tile.byteOffset << "\n";
    }
    
    // Show last tile (bottom-right corner, likely edge tile)
    if (tiles.size() > 0) {
        const auto& lastTile = tiles.back();
        std::cout << "\nLast tile (bottom-right):\n";
        std::cout << "  Tile " << (tiles.size()-1) << " [" << lastTile.row << "," << lastTile.col << "]: "
                  << lastTile.width << "×" << lastTile.height << "px at (" 
                  << lastTile.xOffset << "," << lastTile.yOffset << "), "
                  << lastTile.byteCount << " bytes\n";
    }
    
    return tiles;
}

// Add after the existing tile extraction functions

// Extract tiles specifically for GRID tiling scheme
// In grid tiling, each tile is a separate item
std::vector<TileInfo> extractGridTiles(const MetaBoxInfo& info) {
    std::vector<TileInfo> tiles;
    
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Complete Grid Tile Extraction                     ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    // Get full image dimensions from first ipma entry (grid container)
    uint32_t fullImageWidth = 0;
    uint32_t fullImageHeight = 0;
    uint32_t tileWidth = 0;
    uint32_t tileHeight = 0;
    
    if (!info.itemPropertyAssociations.empty()) {
        for (const auto& prop : info.itemPropertyAssociations[0].properties) {
            if (prop.ispe) {
                fullImageWidth = prop.ispe->width;
                fullImageHeight = prop.ispe->height;
                break;
            }
        }
    }
    
    // Get tile dimensions from second ipma entry (first tile)
    if (info.itemPropertyAssociations.size() > 1) {
        for (const auto& prop : info.itemPropertyAssociations[1].properties) {
            if (prop.ispe) {
                tileWidth = prop.ispe->width;
                tileHeight = prop.ispe->height;
                break;
            }
        }
    }
    
    std::cout << "Full image dimensions from grid container: " << fullImageWidth << "×" << fullImageHeight << "\n";
    std::cout << "Tile size from first tile: " << tileWidth << "×" << tileHeight << "\n\n";
    
    // Calculate grid
    uint32_t numCols = (fullImageWidth + tileWidth - 1) / tileWidth;
    uint32_t numRows = (fullImageHeight + tileHeight - 1) / tileHeight;
    uint32_t expectedTiles = numCols * numRows;
    
    std::cout << "Full image: " << fullImageWidth << "×" << fullImageHeight << "\n";
    std::cout << "Tile size: " << tileWidth << "×" << tileHeight << "\n";
    std::cout << "Total items in iloc: " << info.itemLocations.size() << "\n";
    std::cout << "\nCalculated grid: " << numCols << "×" << numRows << " = " << expectedTiles << " tiles\n";
    
    // Count actual tile items (skip item 1 which is the grid container)
    size_t actualTileCount = 0;
    for (const auto& loc : info.itemLocations) {
        if (loc.itemID > 1) actualTileCount++;  // Item 1 is grid container, items 2+ are tiles
    }
    
    std::cout << "Actual tiles: " << actualTileCount << "\n";
    
    if (actualTileCount != expectedTiles) {
        std::cout << "\n⚠ Warning: Expected " << expectedTiles << " tiles but have " << actualTileCount << "\n";
    }
    
    std::cout << "\n✓ Extracted " << actualTileCount << " tiles with complete metadata\n";
    
    // Extract tiles (skip item 1)
    size_t tileIndex = 0;
    for (const auto& loc : info.itemLocations) {
        // Skip item 1 (grid container)
        if (loc.itemID == 1) continue;
        
        TileInfo tile;
        tile.itemID = loc.itemID;
        tile.tileIndex = tileIndex;
        tile.tilingScheme = "grid";
        
        // Calculate grid position
        tile.row = tileIndex / numCols;
        tile.col = tileIndex % numCols;
        tile.xOffset = tile.col * tileWidth;
        tile.yOffset = tile.row * tileHeight;
        tile.width = tileWidth;
        tile.height = tileHeight;
        
        // Handle edge tiles
        if (tile.xOffset + tileWidth > fullImageWidth) {
            tile.width = fullImageWidth - tile.xOffset;
        }
        if (tile.yOffset + tileHeight > fullImageHeight) {
            tile.height = fullImageHeight - tile.yOffset;
        }
        
        // Get byte offset and count from iloc
        if (!loc.extents.empty()) {
            tile.byteOffset = loc.baseOffset + loc.extents[0].first;
            tile.byteCount = loc.extents[0].second;
        }
        
        // Get properties from ipma (index = itemID - 1, but skip item 1)
        size_t ipmaIndex = loc.itemID - 1;
        if (ipmaIndex < info.itemPropertyAssociations.size()) {
            for (const auto& prop : info.itemPropertyAssociations[ipmaIndex].properties) {
                if (prop.ispe) {
                    // Use actual tile dimensions from ispe
                    tile.width = prop.ispe->width;
                    tile.height = prop.ispe->height;
                }
                if (prop.pixi) {
                    tile.bitsPerChannel = prop.pixi->bitsPerChannel;
                }
                if (prop.colr) {
                    tile.colourInfo = prop.colr;
                }
                if (prop.uncC) {
                    tile.uncompressedConfig = prop.uncC;
                    tile.compression = "deflate";
                }
            }
        }
        
        tiles.push_back(tile);
        tileIndex++;
    }
    
    // Show sample tiles
    std::cout << "\nSample tiles:\n";
    std::cout << "First row:\n";
    for (size_t i = 0; i < std::min(size_t(5), tiles.size()); ++i) {
        const auto& t = tiles[i];
        std::cout << "  Tile " << i << " [" << t.row << "," << t.col << "]: "
                  << t.width << "×" << t.height << "px at (" << t.xOffset << "," << t.yOffset << "), "
                  << t.byteCount << " bytes @ offset " << t.byteOffset << "\n";
    }
    
    if (!tiles.empty()) {
        const auto& last = tiles.back();
        std::cout << "\nLast tile (bottom-right):\n";
        std::cout << "  Tile " << (tiles.size()-1) << " [" << last.row << "," << last.col << "]: "
                  << last.width << "×" << last.height << "px at (" << last.xOffset << "," << last.yOffset << "), "
                  << last.byteCount << " bytes @ offset " << last.byteOffset << "\n";
    }
    
    return tiles;
}

// Corrected extraction for GRID tiling with pyramid levels
// CORRECTED: Understanding that item IDs = tile counts
std::vector<LibheifTileInfo> extractGridTilesWithPyramid(const MetaBoxInfo& info) {
    std::vector<LibheifTileInfo> result;
    
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Grid Tiles with Pyramid Levels                    ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    // Get full image dimensions from ipma entry 0 (grid container)
    uint32_t fullImageWidth = 0;
    uint32_t fullImageHeight = 0;
    uint32_t tileSize = 256;  // Default
    
    if (!info.itemPropertyAssociations.empty()) {
        for (const auto& prop : info.itemPropertyAssociations[0].properties) {
            if (prop.ispe) {
                fullImageWidth = prop.ispe->width;
                fullImageHeight = prop.ispe->height;
                std::cout << "Full image dimensions: " << fullImageWidth << "×" << fullImageHeight << "\n";
                break;
            }
        }
    }
    
    // Get tile size from first tile's ispe
    if (info.itemPropertyAssociations.size() > 1) {
        for (const auto& prop : info.itemPropertyAssociations[1].properties) {
            if (prop.ispe) {
                tileSize = prop.ispe->width;  // Assume square tiles
                std::cout << "Tile size: " << tileSize << "×" << tileSize << "\n";
                break;
            }
        }
    }
    
    // Find dimg references to identify pyramid levels
    std::vector<std::pair<uint32_t, uint32_t>> levels;  // (levelItemID, tileCount)
    
    for (const auto& ref : info.itemReferences) {
        if (ref.referenceType == "dimg") {
            levels.push_back({ref.fromItemID, ref.toItemIDs.size()});
        }
    }
    
    // Sort by tile count descending (highest resolution first)
    std::sort(levels.begin(), levels.end(), 
              [](const auto& a, const auto& b) { return a.second > b.second; });
    
    std::cout << "\nFound " << levels.size() << " pyramid levels:\n";
    for (size_t i = 0; i < levels.size(); ++i) {
        std::cout << "  Level " << i << ": " << levels[i].second << " tiles\n";
    }
    std::cout << "\n";
    
    // Build iloc lookup by itemID
    std::map<uint32_t, ItemLocationEntry> ilocByItemID;
    for (const auto& loc : info.itemLocations) {
        ilocByItemID[loc.itemID] = loc;
    }
    
    // Build ipma lookup by entry index
    std::map<size_t, std::vector<ItemProperty>> ipmaByIndex;
    for (size_t i = 0; i < info.itemPropertyAssociations.size(); ++i) {
        ipmaByIndex[i] = info.itemPropertyAssociations[i].properties;
    }
    
    // Process each pyramid level
    size_t ilocItemID = 2;  // Start from item 2 (first tile, item 1 is grid container)
    size_t ipmaIndex = 1;   // ipma index 0 is grid container, start at 1
    
    for (size_t levelIdx = 0; levelIdx < levels.size(); ++levelIdx) {
        const auto& [levelItemID, tileCount] = levels[levelIdx];
        
        LibheifTileInfo levelInfo;
        levelInfo.itemID = levelItemID;
        levelInfo.pyramidLevel = levelIdx;
        levelInfo.tileWidth = tileSize;
        levelInfo.tileHeight = tileSize;
        
        // Calculate level dimensions
        uint32_t levelWidth = fullImageWidth >> levelIdx;
        uint32_t levelHeight = fullImageHeight >> levelIdx;
        
        levelInfo.imageWidth = levelWidth;
        levelInfo.imageHeight = levelHeight;
        levelInfo.numCols = (levelWidth + tileSize - 1) / tileSize;
        levelInfo.numRows = (levelHeight + tileSize - 1) / tileSize;
        
        std::cout << "─────────────────────────────────────────────────────────────\n";
        std::cout << "Level " << levelIdx << " (Item " << levelItemID << "):\n";
        std::cout << "  Image: " << levelWidth << "×" << levelHeight << "\n";
        std::cout << "  Grid: " << levelInfo.numCols << "×" << levelInfo.numRows << "\n";
        std::cout << "  Expected tiles: " << (levelInfo.numCols * levelInfo.numRows) << "\n";
        std::cout << "  Actual tiles: " << tileCount << "\n";
        
        std::cout << "  Starting at iloc item " << ilocItemID << ", ipma index " << ipmaIndex << "\n";

        // Process tiles for this level
        for (size_t i = 0; i < tileCount; ++i) {
            TileInfo tile;
            
            // Get location from iloc by sequential item ID
            if (ilocByItemID.find(ilocItemID) != ilocByItemID.end()) {
                const auto& loc = ilocByItemID[ilocItemID];
                tile.itemID = loc.itemID;
                
                if (!loc.extents.empty()) {
                    tile.byteOffset = loc.baseOffset + loc.extents[0].first;
                    tile.byteCount = loc.extents[0].second;
                }
            }
            
            tile.tileIndex = i;
            tile.pyramidLevel = levelIdx;
            tile.tilingScheme = "grid";
            tile.compression = "deflate";
            
            // Calculate grid position
            tile.row = i / levelInfo.numCols;
            tile.col = i % levelInfo.numCols;
            tile.xOffset = tile.col * tileSize;
            tile.yOffset = tile.row * tileSize;
            tile.width = tileSize;
            tile.height = tileSize;
            
            // Handle edge tiles
            if (tile.xOffset + tileSize > levelWidth) {
                tile.width = levelWidth - tile.xOffset;
            }
            if (tile.yOffset + tileSize > levelHeight) {
                tile.height = levelHeight - tile.yOffset;
            }
            
            // Get properties from ipma
            if (ipmaByIndex.find(ipmaIndex) != ipmaByIndex.end()) {
                for (const auto& prop : ipmaByIndex[ipmaIndex]) {
                    if (prop.ispe) {
                        tile.width = prop.ispe->width;
                        tile.height = prop.ispe->height;
                    }
                    if (prop.pixi) {
                        tile.bitsPerChannel = prop.pixi->bitsPerChannel;
                    }
                    if (prop.colr) {
                        tile.colourInfo = prop.colr;
                    }
                    if (prop.uncC) {
                        tile.uncompressedConfig = prop.uncC;
                    }
                }
            }
            
            levelInfo.tiles.push_back(tile);
            ilocItemID++;   // Move to next iloc item
            ipmaIndex++;    // Move to next ipma entry
        }
        

    // IMPORTANT: Skip the next level's grid container in both iloc AND ipma!
    if (levelIdx < levels.size() - 1) {  // Not the last level
        ilocItemID++;  // Skip next level's container in iloc
        ipmaIndex++;   // Skip next level's container in ipma
        std::cout << "  Skipping level container, next iloc item: " << ilocItemID << ", ipma: " << ipmaIndex << "\n";
    }
    
    std::cout << "  ✓ Extracted " << levelInfo.tiles.size() << " tiles\n\n";
        
        result.push_back(levelInfo);
    }
    
    // Summary
    std::cout << "╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Pyramid Summary                                   ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    int totalTiles = 0;
    for (const auto& level : result) {
        totalTiles += level.tiles.size();
        std::cout << "Level " << level.pyramidLevel << ": " 
                  << level.imageWidth << "×" << level.imageHeight << ", "
                  << level.tiles.size() << " tiles\n";
    }
    std::cout << "\nTotal: " << totalTiles << " tiles across " << result.size() << " levels\n";
    
    // Show sample tiles from each level
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Sample Tiles from Each Level                      ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    for (const auto& level : result) {
        std::cout << "Level " << level.pyramidLevel << " samples:\n";
        for (size_t i = 0; i < std::min(size_t(3), level.tiles.size()); ++i) {
            const auto& t = level.tiles[i];
            std::cout << "  Tile [" << t.row << "," << t.col << "]: "
                      << t.width << "×" << t.height << "px, "
                      << t.byteCount << " bytes @ offset " << t.byteOffset << "\n";
        }
        std::cout << "\n";
    }
    
    return result;
}

#endif // HEIF_META_BOX_PARSER_DEFINED

#### Extract tileinfo unci

In [ ]:
#ifndef HEIF_META_BOX_PARSER_UNCI_DEFINED
#define HEIF_META_BOX_PARSER_UNCI_DEFINED

std::vector<TileInfo> extractUnciTiles(const MetaBoxInfo& info) {
    std::vector<TileInfo> tiles;
    
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Unci Tile Extraction                              ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    // Safety checks
    if (info.itemPropertyAssociations.empty()) {
        std::cerr << "ERROR: No property associations\n";
        return tiles;
    }
    
    // Get properties
    uint32_t fullImageWidth = 0;
    uint32_t fullImageHeight = 0;
    std::optional<UncompressedConfig> uncC;
    std::optional<IcefBox> icef;
    
    const auto& props = info.itemPropertyAssociations[0].properties;
    
    for (const auto& prop : props) {
        if (prop.ispe && prop.ispe.has_value()) {
            fullImageWidth = prop.ispe->width;
            fullImageHeight = prop.ispe->height;
        }
        if (prop.uncC && prop.uncC.has_value()) {
            uncC = prop.uncC;
        }
        if (prop.icef && prop.icef.has_value()) {
            icef = prop.icef;
        }
    }
    
    if (!uncC.has_value()) {
        std::cerr << "ERROR: Missing uncC\n";
        return tiles;
    }
    
    // Get grid from uncC
    uint32_t numCols = uncC->numTileColsMinus1 + 1;
    uint32_t numRows = uncC->numTileRowsMinus1 + 1;
    uint32_t tileWidth = fullImageWidth / numCols;
    uint32_t tileHeight = fullImageHeight / numRows;
    uint32_t expectedTiles = numCols * numRows;
    
    std::cout << "Full image: " << fullImageWidth << "×" << fullImageHeight << "\n";
    std::cout << "Tile grid: " << numCols << "×" << numRows << " = " << expectedTiles << " tiles\n";
    std::cout << "Tile size: " << tileWidth << "×" << tileHeight << "\n";
    
    if (!icef.has_value() || icef->units.empty()) {
        std::cerr << "ERROR: No icef data\n";
        return tiles;
    }
    
    std::cout << "icef entries: " << icef->units.size() << "\n";
    
    // Get base offset
    if (info.itemLocations.empty()) {
        std::cerr << "ERROR: No item locations\n";
        return tiles;
    }
    
    uint64_t baseOffset = info.itemLocations[0].baseOffset;
    uint64_t totalItemSize = info.itemLocations[0].getTotalSize();
    
    std::cout << "Base offset from iloc: " << baseOffset << "\n";
    std::cout << "Total item size: " << totalItemSize << " bytes\n\n";
    
    // Extract tiles
    size_t numTilesToExtract = std::min(expectedTiles, (uint32_t)icef->units.size());
    
    for (size_t i = 0; i < numTilesToExtract; ++i) {
        TileInfo tile;
        
        tile.itemID = 1;
        tile.tileIndex = i;
        tile.tilingScheme = "unci";
        tile.compression = "deflate";
        tile.pyramidLevel = 0;
        
        // Calculate grid position
        tile.row = i / numCols;
        tile.col = i % numCols;
        tile.xOffset = tile.col * tileWidth;
        tile.yOffset = tile.row * tileHeight;
        tile.width = std::min(tileWidth, fullImageWidth - tile.xOffset);
        tile.height = std::min(tileHeight, fullImageHeight - tile.yOffset);
        
        // Get byte info from icef
        tile.byteOffset = baseOffset + icef->units[i].unit_offset;
        tile.byteCount = icef->units[i].unit_size;
        
        // Handle last tile if size is unknown (offset-only case)
        if (tile.byteCount == 0 && i == numTilesToExtract - 1) {
            // Last tile extends from its offset to the end of item data
            uint64_t lastTileRelativeOffset = icef->units[i].unit_offset;
            
            if (totalItemSize > lastTileRelativeOffset) {
                tile.byteCount = totalItemSize - lastTileRelativeOffset;
                std::cout << "  Last tile size calculated from iloc extent: " 
                          << tile.byteCount << " bytes\n";
                std::cout << "    (total_size=" << totalItemSize 
                          << " - last_offset=" << lastTileRelativeOffset << ")\n";
            } else {
                std::cerr << "  ⚠ ERROR: Last tile offset (" << lastTileRelativeOffset 
                          << ") exceeds total item size (" << totalItemSize << ")\n";
            }
        }
        
        tile.uncompressedConfig = uncC;
        
        tiles.push_back(tile);
    }
    
    std::cout << "\n✓ Extracted " << tiles.size() << " unci tiles\n\n";
    
    // Show samples
    for (size_t i = 0; i < std::min(size_t(5), tiles.size()); ++i) {
        const auto& t = tiles[i];
        std::cout << "  Tile " << i << " [" << t.row << "," << t.col << "]: "
                  << t.width << "×" << t.height << "px at (" << t.xOffset << "," << t.yOffset << "), "
                  << t.byteCount << " bytes @ " << t.byteOffset << "\n";
    }
    
    return tiles;
}


// Extract UNCI pyramid tiles from meta box
std::vector<UnciPyramidLevel> extractUnciPyramidTiles(const MetaBoxInfo& info) {
    std::vector<UnciPyramidLevel> pyramid;
    
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          UNCI Pyramid Tile Extraction                      ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    std::cout << "Found " << info.itemPropertyAssociations.size() << " property associations\n\n";
    
    // Get base offset from first iloc entry
    uint64_t baseOffsetStart = info.itemLocations.empty() ? 0 : info.itemLocations[0].baseOffset;
    
    // Process each property association as a pyramid level
    for (size_t levelIdx = 0; levelIdx < info.itemPropertyAssociations.size(); ++levelIdx) {
        const auto& assoc = info.itemPropertyAssociations[levelIdx];
        
        UnciPyramidLevel level;
        level.level = levelIdx;
        level.itemID = assoc.itemID;
        level.imageWidth = 0;
        level.imageHeight = 0;
        level.numCols = 0;
        level.numRows = 0;
        level.tileWidth = 0;
        level.tileHeight = 0;
        
        std::cout << "─────────────────────────────────────────────────────────────\n";
        std::cout << "Level " << levelIdx << " (Item ID " << assoc.itemID << "):\n";
        
        // Extract properties
        for (const auto& prop : assoc.properties) {
            if (prop.ispe && prop.ispe.has_value()) {
                level.imageWidth = prop.ispe->width;
                level.imageHeight = prop.ispe->height;
                std::cout << "  Image size: " << level.imageWidth << "×" << level.imageHeight << "\n";
            }
            if (prop.uncC && prop.uncC.has_value()) {
                level.uncC = prop.uncC;
                level.numCols = prop.uncC->numTileColsMinus1 + 1;
                level.numRows = prop.uncC->numTileRowsMinus1 + 1;
                std::cout << "  Tile grid from uncC: " << level.numCols << "×" << level.numRows 
                          << " = " << (level.numCols * level.numRows) << " tiles\n";
            }
            if (prop.icef && prop.icef.has_value()) {
                level.icef = prop.icef;
                std::cout << "  icef entries: " << prop.icef->units.size() << "\n";
            }
        }
        
        // Validate
        if (level.imageWidth == 0 || level.imageHeight == 0) {
            std::cout << "  ✗ Invalid image dimensions\n";
            continue;
        }
        
        if (!level.icef.has_value() || level.icef->units.empty()) {
            std::cout << "  ✗ No icef tile information\n";
            continue;
        }
        
        // Calculate tile size from grid and image size
        if (level.numCols > 0 && level.numRows > 0) {
            level.tileWidth = (level.imageWidth + level.numCols - 1) / level.numCols;
            level.tileHeight = (level.imageHeight + level.numRows - 1) / level.numRows;
            std::cout << "  Calculated tile size: " << level.tileWidth << "×" << level.tileHeight << "\n";
        } else {
            std::cout << "  ✗ No tile grid information\n";
            continue;
        }
        
        // Get base offset for this level from iloc
        uint64_t levelBaseOffset = baseOffsetStart;
        if (levelIdx < info.itemLocations.size()) {
            levelBaseOffset = info.itemLocations[levelIdx].baseOffset;
            std::cout << "  Base offset: " << levelBaseOffset << "\n";
        }
        
        // Calculate expected tiles
        size_t expectedTiles = level.numCols * level.numRows;
        size_t actualIcefEntries = level.icef->units.size();
        
        std::cout << "  Expected tiles: " << expectedTiles << "\n";
        std::cout << "  icef entries: " << actualIcefEntries << "\n";
        
        // Determine if we need to skip first entry (padding)
        size_t startIdx = 0;
        uint64_t accumulatedOffset = 0;
        
        if (actualIcefEntries > expectedTiles) {
            // Extra entry - likely padding
            if (level.icef->units[0].unit_size <= 5) {
                std::cout << "  Skipping first icef entry (padding, size=" 
                          << level.icef->units[0].unit_size << ")\n";
                startIdx = 1;
                accumulatedOffset = level.icef->units[0].unit_size; // Start after padding
            }
        } else if (actualIcefEntries < expectedTiles) {
            std::cout << "  ⚠ Warning: icef has fewer entries than expected tiles\n";
        }
        
        // Extract tiles using accumulated offsets
        for (size_t i = startIdx; i < actualIcefEntries && (i - startIdx) < expectedTiles; ++i) {
            TileInfo tile;
            
            tile.itemID = assoc.itemID;
            tile.tileIndex = i - startIdx;
            tile.tilingScheme = "unci";
            tile.compression = "deflate";
            tile.pyramidLevel = levelIdx;
            
            // Calculate grid position
            tile.row = tile.tileIndex / level.numCols;
            tile.col = tile.tileIndex % level.numCols;
            tile.xOffset = tile.col * level.tileWidth;
            tile.yOffset = tile.row * level.tileHeight;
            tile.width = std::min(level.tileWidth, level.imageWidth - tile.xOffset);
            tile.height = std::min(level.tileHeight, level.imageHeight - tile.yOffset);
            
            // Use accumulated offset (the icef offset is relative to start of this level's data)
            // The icef->unit_offset field contains the accumulated offset within this level
            tile.byteOffset = levelBaseOffset + level.icef->units[i].unit_offset;
            tile.byteCount = level.icef->units[i].unit_size;
            
            // Add metadata
            tile.uncompressedConfig = level.uncC;
            
            level.tiles.push_back(tile);
        }
        
        std::cout << "  ✓ Extracted " << level.tiles.size() << " tiles\n";
        
        pyramid.push_back(level);
    }
    
    // Summary
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Pyramid Summary                                   ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    int totalTiles = 0;
    for (const auto& level : pyramid) {
        totalTiles += level.tiles.size();
        std::cout << "Level " << level.level << ": " 
                  << level.imageWidth << "×" << level.imageHeight << ", "
                  << level.tiles.size() << " tiles ("
                  << level.numCols << "×" << level.numRows << " grid)\n";
    }
    
    std::cout << "\nTotal: " << totalTiles << " tiles across " 
              << pyramid.size() << " level(s)\n";
    
    return pyramid;
}


#endif // HEIF_META_BOX_PARSER_UNCI_DEFINED

#### Extract tileinfo tili

##### Non-pyramid tiled HEIF

In [ ]:
#ifndef HEIF_META_BOX_PARSER_TILI_TILES_DEFINED
#define HEIF_META_BOX_PARSER_TILI_TILES_DEFINED

// Structure to hold tili metadata extraction results
struct TiliMetadataResult {
    std::vector<TileInfo> tiles;  // Tiles with placeholders for offset/count
    size_t offsetTablePosition;    // Absolute file position of offset table
    size_t offsetTableSize;        // Size of offset table in bytes
    uint8_t offsetFieldBytes;      // Bytes per offset field (4, 5, 6, or 8)
    uint8_t sizeFieldBytes;        // Bytes per size field (0, 3, 4, or 8)
    bool sequentialHint;           // Whether tiles stored sequentially
    bool success;                  // Whether metadata extraction succeeded
};

// Extract tili metadata from meta box (without reading tile data)
TiliMetadataResult extractTiliMetadataFromMeta(const MetaBoxInfo& info, size_t mdatDataStart) {
    TiliMetadataResult result;
    result.success = false;
    result.offsetTablePosition = 0;
    result.offsetTableSize = 0;
    result.offsetFieldBytes = 4;  // Default: 32-bit offsets
    result.sizeFieldBytes = 0;    // Default: inferred sizes
    result.sequentialHint = true; // Default: sequential
    
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Extracting Tili Metadata from Meta Box            ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    // Step 1: Find the main tili item and its properties
    uint32_t mainItemID = 0;
    std::optional<TiliConfiguration> tiliConfig;
    std::optional<ImageSpatialExtents> imageSize;
    std::vector<uint8_t> bitsPerChannel;
    std::optional<ColourInformation> colourInfo;
    uint32_t numCols = 0;
    uint32_t numRows = 0;
    uint32_t tileWidth = 0;
    uint32_t tileHeight = 0;
    
    std::cout << "Step 1: Analyzing items...\n";
    for (const auto& item : info.items) {
        std::cout << "  Item " << item.itemID << ": type='" << item.itemType << "'";
        if (!item.itemName.empty()) {
            std::cout << ", name='" << item.itemName << "'";
        }
        std::cout << "\n";
        
        if (item.itemType == "tili") {
            mainItemID = item.itemID;
            std::cout << "    → This is the main tili item\n";
        }
    }
    
    if (mainItemID == 0) {
        // Fallback: use first item with tili properties
        for (const auto& assoc : info.itemPropertyAssociations) {
            for (const auto& prop : assoc.properties) {
                if (prop.propertyType == "tilC" || prop.tili) {
                    mainItemID = assoc.itemID;
                    std::cout << "  Using item " << mainItemID << " (has tilC property)\n";
                    break;
                }
            }
            if (mainItemID > 0) break;
        }
    }
    
    if (mainItemID == 0) {
        std::cerr << "✗ No tili item found\n";
        return result;
    }
    
    // Step 2: Extract properties for the main item
    std::cout << "\nStep 2: Extracting properties for item " << mainItemID << "...\n";
    
    for (const auto& assoc : info.itemPropertyAssociations) {
        if (assoc.itemID == mainItemID) {
            for (const auto& prop : assoc.properties) {
                if (prop.ispe) {
                    imageSize = prop.ispe;
                    std::cout << "  ✓ Image size: " << imageSize->width << "×" << imageSize->height << "\n";
                }
                
                if (prop.tili) {
                    tiliConfig = prop.tili;
                    tileWidth = tiliConfig->tileWidth;
                    tileHeight = tiliConfig->tileHeight;
                    std::cout << "  ✓ Tile size: " << tileWidth << "×" << tileHeight << "\n";
                    
                    if (tiliConfig->hasExplicitOffsets) {
                        std::cout << "  ⚠ tilC has explicit offsets (unusual for tili format)\n";
                    }
                }
                
                if (prop.pixi) {
                    bitsPerChannel = prop.pixi->bitsPerChannel;
                    std::cout << "  ✓ Channels: " << bitsPerChannel.size() << " [";
                    for (size_t i = 0; i < bitsPerChannel.size(); ++i) {
                        if (i > 0) std::cout << ", ";
                        std::cout << (int)bitsPerChannel[i];
                    }
                    std::cout << " bits]\n";
                }
                
                if (prop.colr) {
                    colourInfo = prop.colr;
                    std::cout << "  ✓ Color info: " << colourInfo->colourType << "\n";
                }
            }
            break;
        }
    }
    
    if (!imageSize || tileWidth == 0 || tileHeight == 0) {
        std::cerr << "✗ Missing required properties (ispe or tilC)\n";
        return result;
    }
    
    // Step 3: Calculate grid dimensions
    numCols = (imageSize->width + tileWidth - 1) / tileWidth;
    numRows = (imageSize->height + tileHeight - 1) / tileHeight;
    uint32_t numTiles = numCols * numRows;
    
    std::cout << "\nStep 3: Calculated grid:\n";
    std::cout << "  Columns: " << numCols << "\n";
    std::cout << "  Rows: " << numRows << "\n";
    std::cout << "  Total tiles: " << numTiles << "\n";
    
    // Step 4: Parse tilC flags to determine offset table format
    // TODO: Extract actual flags from tilC box
    // For now, use defaults (32-bit offsets, no size field, sequential)
    result.offsetFieldBytes = 4;  // 32-bit offsets
    result.sizeFieldBytes = 0;    // Sizes inferred from offsets
    result.sequentialHint = true; // Sequential storage
    
    std::cout << "\nStep 4: Offset table format:\n";
    std::cout << "  Offset field: " << (result.offsetFieldBytes * 8) << " bits\n";
    std::cout << "  Size field: " << (result.sizeFieldBytes * 8) << " bits";
    if (result.sizeFieldBytes == 0) {
        std::cout << " (inferred)";
    }
    std::cout << "\n";
    std::cout << "  Sequential hint: " << (result.sequentialHint ? "yes" : "no") << "\n";
    
    // Step 5: Calculate offset table location and size
    size_t entrySize = result.offsetFieldBytes + result.sizeFieldBytes;
    result.offsetTableSize = numTiles * entrySize;
    result.offsetTablePosition = mdatDataStart;
    
    std::cout << "\nStep 5: Offset table location:\n";
    std::cout << "  Entry size: " << entrySize << " bytes\n";
    std::cout << "  Table size: " << result.offsetTableSize << " bytes\n";
    std::cout << "  File position: " << result.offsetTablePosition << "\n";
    
    // Step 6: Create tile metadata with placeholder offsets
    std::cout << "\nStep 6: Creating tile metadata...\n";
    
    for (uint32_t i = 0; i < numTiles; ++i) {
        TileInfo tile;
        tile.itemID = mainItemID;
        tile.tileIndex = i;
        tile.pyramidLevel = 0;
        tile.tilingScheme = "tili";
        tile.compression = "deflate";  // Could be extracted from tilC
        
        tile.row = i / numCols;
        tile.col = i % numCols;
        tile.xOffset = tile.col * tileWidth;
        tile.yOffset = tile.row * tileHeight;
        tile.width = std::min(tileWidth, imageSize->width - tile.xOffset);
        tile.height = std::min(tileHeight, imageSize->height - tile.yOffset);
        
        // ⭐ Placeholder values - to be populated later
        tile.byteOffset = 0;
        tile.byteCount = 0;
        
        tile.bitsPerChannel = bitsPerChannel;
        tile.colourInfo = colourInfo;
        tile.tili = tiliConfig;
        
        result.tiles.push_back(tile);
    }
    
    std::cout << "  ✓ Created " << result.tiles.size() << " tile metadata entries\n";
    
    result.success = true;
    return result;
}

// Parse offset table data and populate tile offsets/sizes
// Add this improved parser right after the TiliMetadataResult structure
bool populateTileOffsetsFromTable(
    std::vector<TileInfo>& tiles,
    const std::vector<uint8_t>& tableData,
    size_t offsetTableFilePosition,
    uint8_t offsetFieldBytes,
    uint8_t sizeFieldBytes,
    bool sequentialHint) {
    
    // Add validation for reasonable field sizes
    if (offsetFieldBytes == 0 || offsetFieldBytes > 8) {
        std::cerr << "Invalid offset field size: " << (int)offsetFieldBytes << "\n";
        return false;
    }
    
    if (sizeFieldBytes > 8) {
        std::cerr << "Invalid size field size: " << (int)sizeFieldBytes << "\n";
        return false;
    }

    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Parsing Tili Offset Table                         ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    std::cout << "Table data size: " << tableData.size() << " bytes\n";
    std::cout << "Offset field: " << (offsetFieldBytes * 8) << " bits\n";
    std::cout << "Size field: " << (sizeFieldBytes * 8) << " bits\n";
    std::cout << "Sequential hint: " << (sequentialHint ? "yes" : "no") << "\n";
    std::cout << "Number of tiles: " << tiles.size() << "\n\n";
    
    size_t entrySize = offsetFieldBytes + sizeFieldBytes;
    size_t expectedTableSize = tiles.size() * entrySize;
    
    if (tableData.size() < expectedTableSize) {
        std::cerr << "✗ Table data too small\n";
        return false;
    }
    
    // DEBUG: Show raw table bytes
    std::cout << "Raw offset table (first 64 bytes):\n  ";
    for (size_t i = 0; i < std::min(size_t(64), tableData.size()); ++i) {
        printf("%02X ", tableData[i]);
        if ((i + 1) % 16 == 0 && i + 1 < std::min(size_t(64), tableData.size())) {
            std::cout << "\n  ";
        }
    }
    std::cout << "\n\n";

    

    std::cout << "Step 1: Parsing " << tiles.size() << " table entries...\n";
    
    std::vector<uint64_t> tileOffsets;
    std::vector<uint64_t> tileSizes;
    
    size_t tablePos = 0;
    
    // Read offsets (BIG-ENDIAN per ISOBMFF spec)
    for (size_t i = 0; i < tiles.size(); ++i) {
        uint64_t offset = 0;
        
        // Read big-endian offset
        for (int j = 0; j < offsetFieldBytes; ++j) {
            offset = (offset << 8) | tableData[tablePos + j];
        }
        tablePos += offsetFieldBytes;
        tileOffsets.push_back(offset);
        
        // Read size if present (also big-endian)
        if (sizeFieldBytes > 0) {
            uint64_t size = 0;
            for (int j = 0; j < sizeFieldBytes; ++j) {
                size = (size << 8) | tableData[tablePos + j];
            }
            tablePos += sizeFieldBytes;
            tileSizes.push_back(size);
        }
        
        if (i < 10 || (i + 1) % 100 == 0 || i == tiles.size() - 1) {
            std::cout << "  Tile " << i << ": relative offset=" << offset;
            if (sizeFieldBytes > 0) {
                std::cout << ", size=" << tileSizes.back();
            }
            std::cout << " → absolute offset=" << (offsetTableFilePosition + offset) << "\n";
        }
    }
    
    std::cout << "\n  ✓ Parsed " << tileOffsets.size() << " offsets\n";
    

    // Step 2: Compute sizes if not present
    if (tileSizes.empty()) {
        std::cout << "\nStep 2: Computing tile sizes from offsets...\n";
        
        // Separate tiles into categories
        std::map<uint64_t, std::vector<size_t>> offsetToTiles;  // offset -> [tile indices]
        std::vector<size_t> referenceTiles;  // Tiles with offset=1 (reference to lower level)
        
        for (size_t i = 0; i < tileOffsets.size(); ++i) {
            if (tileOffsets[i] == 0) {
                // Empty tile - skip
                continue;
            } else if (tileOffsets[i] == 1) {
                // Reference to lower resolution level
                referenceTiles.push_back(i);
            } else {
                // Normal tile - group by offset
                offsetToTiles[tileOffsets[i]].push_back(i);
            }
        }
        
        std::cout << "  Unique tile offsets: " << offsetToTiles.size() << "\n";
        std::cout << "  Reference tiles: " << referenceTiles.size() << "\n";
        
        // Sort unique offsets
        std::vector<std::pair<uint64_t, std::vector<size_t>>> sortedOffsets(
            offsetToTiles.begin(), offsetToTiles.end());
        std::sort(sortedOffsets.begin(), sortedOffsets.end());
        
        // Initialize all sizes to 0
        tileSizes.resize(tileOffsets.size(), 0);
        
        // Compute sizes for unique tiles
        for (size_t i = 0; i < sortedOffsets.size(); ++i) {
            uint64_t currentOffset = sortedOffsets[i].first;
            const auto& tileIndices = sortedOffsets[i].second;
            
            uint64_t size;
            if (i + 1 < sortedOffsets.size()) {
                // Size is difference to next unique offset
                uint64_t nextOffset = sortedOffsets[i + 1].first;
                size = nextOffset - currentOffset;
            } else {
                // Last tile - size is unknown (will need file size or mdat end)
                size = 0;  // Mark as TBD
            }
            
            // Assign this size to all tiles sharing this offset
            for (size_t tileIdx : tileIndices) {
                tileSizes[tileIdx] = size;
            }
            
            if (i < 5) {
                std::cout << "    Tiles at offset " << currentOffset << ": "
                        << tileIndices.size() << " tiles, size=" << size << "\n";
            }
        }
        
        // Mark reference tiles (size stays 0, but we know why)
        for (size_t tileIdx : referenceTiles) {
            tileSizes[tileIdx] = 0;  // Explicitly 0 for reference
        }
        
        std::cout << "  ✓ Computed sizes for " << offsetToTiles.size() << " unique tiles\n";
    }

    // Step 3: Populate tile metadata with reference info
    std::cout << "\nStep 3: Populating tile metadata...\n";

    size_t validTiles = 0;
    size_t emptyTiles = 0;
    size_t referenceTiles = 0;

    for (size_t i = 0; i < tiles.size(); ++i) {
        if (tileOffsets[i] == 0) {
            tiles[i].byteOffset = 0;
            tiles[i].byteCount = 0;
            emptyTiles++;
        } else if (tileOffsets[i] == 1) {
            tiles[i].byteOffset = 1;  // Special marker
            tiles[i].byteCount = 0;
            referenceTiles++;
        } else {
            tiles[i].byteOffset = offsetTableFilePosition + tileOffsets[i];
            tiles[i].byteCount = tileSizes[i];
            
            if (tileSizes[i] > 0) {
                validTiles++;
            } else {
                // This tile shares an offset with another tile (duplicate data)
                referenceTiles++;
            }
        }
    }

    std::cout << "  ✓ Populated " << tiles.size() << " tiles:\n";
    std::cout << "    Unique tiles: " << validTiles << "\n";
    std::cout << "    Reference tiles: " << referenceTiles << "\n";
    std::cout << "    Empty tiles: " << emptyTiles << "\n";

    // Step 4: Show sample tiles with reference info
    std::cout << "\nStep 4: Sample tiles:\n";
    for (size_t i = 0; i < std::min(size_t(10), tiles.size()); ++i) {
        const auto& t = tiles[i];
        std::cout << "  Tile " << i << " [" << t.row << "," << t.col << "]: ";
        
        if (t.byteOffset == 0) {
            std::cout << "EMPTY\n";
        } else if (t.byteOffset == 1) {
            std::cout << "REFERENCE (to lower resolution)\n";
        } else if (t.byteCount == 0) {
            std::cout << "DUPLICATE (shares data with another tile at offset " 
                    << t.byteOffset << ")\n";
        } else {
            std::cout << "offset=" << t.byteOffset << ", size=" << t.byteCount << "\n";
        }
    }
    
    return true;
}

// Main function: Extract tili tiles with complete offset/size information
std::vector<TileInfo> extractTiliTilesSingleLevel(
    const MetaBoxInfo& info,
    const std::string& filepath,
    size_t mdatDataStart) {
    
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Complete Tili Tile Extraction                     ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    // Step 1: Extract metadata
    auto metadata = extractTiliMetadataFromMeta(info, mdatDataStart);
    
    if (!metadata.success) {
        std::cerr << "\n✗ Failed to extract tili metadata\n";
        return {};
    }
    
    std::cout << "✓ Metadata extraction complete\n";
    std::cout << "  Tiles: " << metadata.tiles.size() << "\n\n";
    
    // Step 2: Check if offset table exists or tiles are directly stored
    std::cout << "Checking for offset table...\n";
    
    // Try to decompress from mdatDataStart
    bool hasOffsetTable = true;
    try {
        auto testData = readFileRange(filepath, mdatDataStart, 4096);
        std::vector<uint8_t> decomp(786432);
        z_stream stream = {};
        stream.avail_in = testData.size();
        stream.next_in = testData.data();
        stream.avail_out = decomp.size();
        stream.next_out = decomp.data();
        
        int ret = inflateInit2(&stream, -MAX_WBITS);
        if (ret == Z_OK) {
            ret = inflate(&stream, Z_NO_FLUSH);
            inflateEnd(&stream);
            
            if (ret == Z_OK || ret == Z_STREAM_END) {
                hasOffsetTable = false;
                std::cout << "  → No offset table found (non-standard)\n";
                std::cout << "  → Tiles stored directly starting at offset " << mdatDataStart << "\n\n";
            }
        }
    } catch (...) {}
    
    if (hasOffsetTable) {
        std::cout << "  → Offset table present (standard format)\n\n";
        
        // Read offset table
        auto tableData = readFileRange(filepath, 
                                       metadata.offsetTablePosition, 
                                       metadata.offsetTableSize);
        
        // Parse and populate
        populateTileOffsetsFromTable(
            metadata.tiles,
            tableData,
            metadata.offsetTablePosition,
            metadata.offsetFieldBytes,
            metadata.sizeFieldBytes,
            metadata.sequentialHint
        );
        
        return metadata.tiles;
    }
    
    // No offset table - tiles stored sequentially
    std::cout << "╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Sequential Tile Scanning (No Offset Table)        ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    size_t currentOffset = mdatDataStart;
    size_t expectedDecompressedSize = metadata.tiles[0].width * 
                                      metadata.tiles[0].height * 3;
    std::vector<uint8_t> decompressedBuffer(expectedDecompressedSize);
    
    std::cout << "Scanning " << metadata.tiles.size() << " tiles sequentially...\n\n";
    
    for (size_t i = 0; i < metadata.tiles.size(); ++i) {
        auto& tile = metadata.tiles[i];
        
        try {
            auto compressedData = readFileRange(filepath, currentOffset, 2097152);
            
            z_stream stream = {};
            stream.avail_in = compressedData.size();
            stream.next_in = compressedData.data();
            stream.avail_out = decompressedBuffer.size();
            stream.next_out = decompressedBuffer.data();
            
            int ret = inflateInit2(&stream, -MAX_WBITS);
            if (ret != Z_OK) {
                std::cerr << "  ✗ Tile " << i << ": inflateInit failed\n";
                break;
            }
            
            ret = inflate(&stream, Z_FINISH);
            inflateEnd(&stream);
            
            if (ret == Z_STREAM_END) {
                tile.byteOffset = currentOffset;
                tile.byteCount = stream.total_in;
                currentOffset += stream.total_in;
                
                if (i < 13 || (i + 1) % 50 == 0 || i == metadata.tiles.size() - 1) {
                    std::cout << "Tile " << std::setw(4) << i << ": "
                              << "offset " << std::setw(10) << tile.byteOffset << ", "
                              << "compressed " << std::setw(7) << tile.byteCount << " bytes, "
                              << "decompressed " << std::setw(7) << stream.total_out << " bytes\n";
                }
            } else {
                std::cerr << "  ✗ Tile " << i << ": decompression failed (code " << ret << ")\n";
                break;
            }
            
        } catch (const std::exception& e) {
            std::cerr << "  ✗ Tile " << i << ": " << e.what() << "\n";
            break;
        }
    }
    
    std::cout << "\n✓ Sequential scan complete\n";
    std::cout << "  Total tiles: " << metadata.tiles.size() << "\n";
    std::cout << "  Final offset: " << currentOffset << "\n";
    std::cout << "  Total data: " << (currentOffset - mdatDataStart) << " bytes\n";
    
    return metadata.tiles;
}

#endif // HEIF_META_BOX_PARSER_TILI_TILES_DEFINED


##### Helper Functions for Mapping ids

In [ ]:
#ifndef HELPER_FUNCTIONS_FOR_ID_MAPPING_DEFINED
#define HELPER_FUNCTIONS_FOR_ID_MAPPING_DEFINED

// Add this helper function to detect the best matching strategy
enum class IlocMatchStrategy {
    BY_ITEM_ID,      // Direct item ID match (ideal)
    BY_INDEX,        // Sequential index mapping
    BY_PYRAMID_GROUP // Use pyramid group entity mapping
};

struct IlocMatchResult {
    IlocMatchStrategy strategy;
    std::map<uint32_t, size_t> itemIDToIlocIndex;  // Map item ID -> iloc index
    
    size_t getIlocIndex(uint32_t itemID, size_t fallbackIndex) const {
        auto it = itemIDToIlocIndex.find(itemID);
        if (it != itemIDToIlocIndex.end()) {
            return it->second;
        }
        return fallbackIndex;
    }
};

IlocMatchResult determineIlocMatchStrategy(
    const MetaBoxInfo& info,
    const std::vector<uint32_t>& targetItemIDs) {
    
    IlocMatchResult result;
    result.strategy = IlocMatchStrategy::BY_INDEX;  // Default fallback
    
    std::cout << "\n=== Determining iloc Matching Strategy ===\n";
    std::cout << "Target items: [";
    for (size_t i = 0; i < targetItemIDs.size(); ++i) {
        if (i > 0) std::cout << ", ";
        std::cout << targetItemIDs[i];
    }
    std::cout << "]\n";
    
    std::cout << "iloc items: [";
    for (size_t i = 0; i < info.itemLocations.size(); ++i) {
        if (i > 0) std::cout << ", ";
        std::cout << info.itemLocations[i].itemID;
    }
    std::cout << "]\n\n";
    
    // Strategy 1: Try direct item ID matching
    int directMatches = 0;
    for (uint32_t targetID : targetItemIDs) {
        for (size_t i = 0; i < info.itemLocations.size(); ++i) {
            if (info.itemLocations[i].itemID == targetID) {
                directMatches++;
                result.itemIDToIlocIndex[targetID] = i;
                break;
            }
        }
    }
    
    if (directMatches == targetItemIDs.size()) {
        std::cout << "✓ Strategy: BY_ITEM_ID\n";
        std::cout << "  All " << directMatches << " items matched by ID\n";
        result.strategy = IlocMatchStrategy::BY_ITEM_ID;
        return result;
    }
    
    std::cout << "✗ Direct ID matching failed (" << directMatches << "/" 
              << targetItemIDs.size() << " matches)\n\n";
    
    // Strategy 2: Try pyramid group mapping
    bool hasPyramidGroup = false;
    for (const auto& group : info.entityGroups) {
        if (group.groupType == "pymd") {
            hasPyramidGroup = true;
            std::cout << "Found pyramid group with " << group.entityIDs.size() 
                      << " entities\n";
            
            // Check if pyramid group makes sense
            if (group.entityIDs.size() == targetItemIDs.size() &&
                group.entityIDs.size() == info.itemLocations.size()) {
                
                std::cout << "✓ Strategy: BY_PYRAMID_GROUP\n";
                std::cout << "  Entity mapping:\n";
                
                // Pyramid group entities are typically in reverse order
                // Entity IDs usually go from highest to lowest (e.g., 9,8,7...1)
                // These map to pyramid levels 0,1,2...
                for (size_t levelIdx = 0; levelIdx < group.entityIDs.size(); ++levelIdx) {
                    uint32_t entityID = group.entityIDs[levelIdx];
                    
                    // Map entity ID to iloc index
                    // Typically: entity 9->iloc[0], entity 8->iloc[1], etc.
                    // But entity IDs might not be sequential, so we need to sort
                    
                    // For now, assume entities are in descending order
                    // and iloc is in ascending pyramid order (largest first)
                    size_t ilocIdx = levelIdx;
                    
                    // We need to map this to an actual item ID
                    // The target items are sorted by size (largest first)
                    // So targetItemIDs[levelIdx] corresponds to level levelIdx
                    if (levelIdx < targetItemIDs.size()) {
                        result.itemIDToIlocIndex[targetItemIDs[levelIdx]] = ilocIdx;
                        std::cout << "    Level " << levelIdx << ": Item " 
                                  << targetItemIDs[levelIdx] << " -> iloc[" << ilocIdx << "]\n";
                    }
                }
                
                result.strategy = IlocMatchStrategy::BY_PYRAMID_GROUP;
                return result;
            }
            break;
        }
    }
    
    if (hasPyramidGroup) {
        std::cout << "✗ Pyramid group size mismatch\n\n";
    }
    
    // Strategy 3: Fall back to index-based mapping
    std::cout << "✓ Strategy: BY_INDEX (fallback)\n";
    std::cout << "  Assuming items are in same order:\n";
    
    if (targetItemIDs.size() == info.itemLocations.size()) {
        for (size_t i = 0; i < targetItemIDs.size(); ++i) {
            result.itemIDToIlocIndex[targetItemIDs[i]] = i;
            std::cout << "    Item " << targetItemIDs[i] << " -> iloc[" << i << "]\n";
        }
        std::cout << "  ✓ Count matches (" << targetItemIDs.size() << " items)\n";
    } else {
        std::cout << "  ⚠ Warning: Item count mismatch (target=" << targetItemIDs.size() 
                  << ", iloc=" << info.itemLocations.size() << ")\n";
        std::cout << "    Mapping first " << std::min(targetItemIDs.size(), info.itemLocations.size()) 
                  << " items by index\n";
        
        size_t matchCount = std::min(targetItemIDs.size(), info.itemLocations.size());
        for (size_t i = 0; i < matchCount; ++i) {
            result.itemIDToIlocIndex[targetItemIDs[i]] = i;
        }
    }
    
    result.strategy = IlocMatchStrategy::BY_INDEX;
    return result;
}

#endif // HELPER_FUNCTIONS_FOR_ID_MAPPING_DEFINED

##### Tiled Pyramid HEIF using Tili

In [ ]:
#ifndef HEIF_META_BOX_PARSER_TILI_PYRAMID_DEFINED
#define HEIF_META_BOX_PARSER_TILI_PYRAMID_DEFINED

// Structure to hold pyramid level information
struct TiliPyramidLevel {
    int level;                          // 0 = full res, 1+ = overviews
    uint32_t itemID;                    // Item ID for this level
    uint32_t imageWidth;
    uint32_t imageHeight;
    uint32_t numCols;
    uint32_t numRows;
    uint32_t tileWidth;
    uint32_t tileHeight;
    
    size_t offsetTablePosition;         // File position of offset table
    size_t offsetTableSize;             // Size of offset table
    uint8_t offsetFieldBytes;
    uint8_t sizeFieldBytes;
    bool sequentialHint;
    
    std::vector<TileInfo> tiles;        // Tile metadata
    
    void print() const {
        std::cout << "Level " << level << " (Item " << itemID << "):\n";
        std::cout << "  Image: " << imageWidth << "×" << imageHeight << "\n";
        std::cout << "  Grid: " << numCols << "×" << numRows << " = " 
                  << (numCols * numRows) << " tiles\n";
        std::cout << "  Tile size: " << tileWidth << "×" << tileHeight << "\n";
        std::cout << "  Offset table: " << offsetTableSize << " bytes @ " 
                  << offsetTablePosition << "\n";
    }
};

// Extract metadata for all pyramid levels
std::vector<TiliPyramidLevel> extractTiliPyramidMetadata(const MetaBoxInfo& info,const std::string& codecHint = "deflate") {
    std::vector<TiliPyramidLevel> pyramid;
    
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Extracting TILI Pyramid Metadata                  ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    // Step 1: Get item IDs from property associations
    std::vector<uint32_t> itemIDs;
    
    std::cout << "Items with tilC properties:\n";
    for (const auto& assoc : info.itemPropertyAssociations) {
        bool hasTilC = false;
        for (const auto& prop : assoc.properties) {
            if (prop.propertyType == "tilC" || prop.tili) {
                hasTilC = true;
                break;
            }
        }
        
        if (hasTilC) {
            itemIDs.push_back(assoc.itemID);
            std::cout << "  Item " << assoc.itemID << "\n";
        }
    }
    
    if (itemIDs.empty()) {
        std::cout << "No items with tilC properties found\n";
        return pyramid;
    }
    
    // Step 2: Sort by image size (largest first = level 0)
    struct ItemWithSize {
        uint32_t itemID;
        uint32_t width;
        uint32_t height;
        uint64_t area;
    };
    
    std::vector<ItemWithSize> itemsWithSize;
    
    for (uint32_t itemID : itemIDs) {
        ItemWithSize item;
        item.itemID = itemID;
        item.width = 0;
        item.height = 0;
        
        for (const auto& assoc : info.itemPropertyAssociations) {
            if (assoc.itemID == itemID) {
                for (const auto& prop : assoc.properties) {
                    if (prop.ispe) {
                        item.width = prop.ispe->width;
                        item.height = prop.ispe->height;
                        item.area = (uint64_t)item.width * item.height;
                        break;
                    }
                }
                break;
            }
        }
        
        if (item.width > 0 && item.height > 0) {
            itemsWithSize.push_back(item);
        }
    }
    
    std::sort(itemsWithSize.begin(), itemsWithSize.end(),
              [](const ItemWithSize& a, const ItemWithSize& b) {
                  return a.area > b.area;
              });
    
    std::cout << "\nPyramid levels sorted by size:\n";
    
    // Extract sorted item IDs
    std::vector<uint32_t> sortedItemIDs;
    for (const auto& item : itemsWithSize) {
        sortedItemIDs.push_back(item.itemID);
    }
    
    // Step 3: Determine iloc matching strategy
    auto ilocMatch = determineIlocMatchStrategy(info, sortedItemIDs);
    
    // Step 4: Create pyramid levels
    for (size_t i = 0; i < itemsWithSize.size(); ++i) {
        const auto& item = itemsWithSize[i];
        
        TiliPyramidLevel level;
        level.level = i;
        level.itemID = item.itemID;
        level.imageWidth = item.width;
        level.imageHeight = item.height;
        level.tileWidth = 1024;
        level.tileHeight = 1024;
        level.offsetFieldBytes = 4;
        level.sizeFieldBytes = 3;
        level.sequentialHint = false;
        
        std::cout << "  Level " << i << ": Item " << item.itemID 
                  << " (" << item.width << "×" << item.height << ")\n";
        
        // Get detailed properties
        for (const auto& assoc : info.itemPropertyAssociations) {
            if (assoc.itemID == item.itemID) {
                for (const auto& prop : assoc.properties) {
                    if (prop.tili) {
                        level.tileWidth = prop.tili->tileWidth;
                        level.tileHeight = prop.tili->tileHeight;
                        level.offsetFieldBytes = prop.tili->offsetFieldBytes;
                        level.sizeFieldBytes = prop.tili->sizeFieldBytes;
                        level.sequentialHint = prop.tili->sequentialHint;
                        
                        std::cout << "    Tile size: " << level.tileWidth << "×" << level.tileHeight 
                                  << ", offset=" << (int)level.offsetFieldBytes << "B"
                                  << ", size=" << (int)level.sizeFieldBytes << "B\n";
                    }
                }
                break;
            }
        }
        
        // Calculate grid
        level.numCols = (level.imageWidth + level.tileWidth - 1) / level.tileWidth;
        level.numRows = (level.imageHeight + level.tileHeight - 1) / level.tileHeight;
        uint32_t numTiles = level.numCols * level.numRows;
        
        std::cout << "    Grid: " << level.numCols << "×" << level.numRows 
                  << " = " << numTiles << " tiles\n";
        
        // Get offset table position using the determined strategy
        size_t ilocIdx = ilocMatch.getIlocIndex(item.itemID, i);
        
        if (ilocIdx < info.itemLocations.size()) {
            const auto& loc = info.itemLocations[ilocIdx];
            level.offsetTablePosition = loc.baseOffset + (loc.extents.empty() ? 0 : loc.extents[0].first);
            size_t entrySize = level.offsetFieldBytes + level.sizeFieldBytes;
            level.offsetTableSize = numTiles * entrySize;
            
            std::cout << "    Offset table: " << level.offsetTableSize << " bytes @ " 
                      << level.offsetTablePosition << " (iloc[" << ilocIdx << "]";
            
            if (ilocMatch.strategy == IlocMatchStrategy::BY_ITEM_ID) {
                std::cout << ", matched by ID " << loc.itemID;
            } else if (ilocMatch.strategy == IlocMatchStrategy::BY_PYRAMID_GROUP) {
                std::cout << ", via pyramid group";
            } else {
                std::cout << ", by index";
            }
            std::cout << ")\n";
        } else {
            std::cout << "    ⚠ No iloc entry found (index " << ilocIdx << " out of range)\n";
            level.offsetTablePosition = 0;
            level.offsetTableSize = 0;
        }
        
        // Create placeholder tiles
        for (uint32_t t = 0; t < numTiles; ++t) {
            TileInfo tile;
            tile.itemID = level.itemID;
            tile.tileIndex = t;
            tile.pyramidLevel = level.level;
            tile.tilingScheme = "tili";
            tile.compression = codecHint;
            
            tile.row = t / level.numCols;
            tile.col = t % level.numCols;
            tile.xOffset = tile.col * level.tileWidth;
            tile.yOffset = tile.row * level.tileHeight;
            tile.width = std::min(level.tileWidth, level.imageWidth - tile.xOffset);
            tile.height = std::min(level.tileHeight, level.imageHeight - tile.yOffset);
            
            tile.byteOffset = 0;
            tile.byteCount = 0;
            
            level.tiles.push_back(tile);
        }
        
        pyramid.push_back(level);
    }
    
    std::cout << "\n✓ Extracted metadata for " << pyramid.size() << " pyramid levels\n";
    
    return pyramid;
}

// Populate tile offsets for all pyramid levels
bool populateTiliPyramidOffsets(
    std::vector<TiliPyramidLevel>& pyramid,
    const std::string& filepath) {
    
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Populating TILI Pyramid Tile Offsets              ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    for (auto& level : pyramid) {
        std::cout << "─────────────────────────────────────────────────────────────\n";
        std::cout << "Level " << level.level << " (Item " << level.itemID << "):\n";
        
        // Read offset table
        std::vector<uint8_t> tableData;
        try {
            tableData = readFileRange(filepath, level.offsetTablePosition, level.offsetTableSize);
        } catch (const std::exception& e) {
            std::cerr << "  ✗ Failed to read offset table: " << e.what() << "\n";
            continue;
        }
        
        std::cout << "  Read " << tableData.size() << " bytes of offset table\n";
        
        // Parse offset table
        bool success = populateTileOffsetsFromTable(
            level.tiles,
            tableData,
            level.offsetTablePosition,
            level.offsetFieldBytes,
            level.sizeFieldBytes,
            level.sequentialHint
        );
        
        if (!success) {
            std::cerr << "  ✗ Failed to parse offset table\n";
            continue;
        }
        
        // Count valid tiles
        int validTiles = 0;
        int refTiles = 0;
        int emptyTiles = 0;
        
        for (const auto& tile : level.tiles) {
            if (tile.byteOffset == 0) {
                emptyTiles++;
            } else if (tile.byteOffset == 1) {
                refTiles++;
            } else if (tile.byteCount > 0) {
                validTiles++;
            } else {
                refTiles++;  // Duplicate offset
            }
        }
        
        std::cout << "  ✓ Level " << level.level << ": " << validTiles << " unique, "
                  << refTiles << " references, " << emptyTiles << " empty\n";
    }
    
    std::cout << "\n✓ Populated offsets for all pyramid levels\n";
    return true;
}

// Complete pyramid extraction
std::vector<TiliPyramidLevel> extractTiliPyramidComplete(
    const MetaBoxInfo& info,
    const std::string& filepath) {
    
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Complete TILI Pyramid Extraction                  ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n";
    
    // Extract metadata
    auto pyramid = extractTiliPyramidMetadata(info);
    
    if (pyramid.empty()) {
        std::cout << "\n✗ No pyramid levels found\n";
        return pyramid;
    }
    
    // Populate offsets
    populateTiliPyramidOffsets(pyramid, filepath);
    
    // Summary
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Pyramid Summary                                   ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    int totalTiles = 0;
    for (const auto& level : pyramid) {
        totalTiles += level.tiles.size();
        level.print();
        std::cout << "\n";
    }
    
    std::cout << "Total: " << pyramid.size() << " levels, " << totalTiles << " tiles\n";
    
    return pyramid;
}

#endif // HEIF_META_BOX_PARSER_TILI_PYRAMID_DEFINED

### HEIF MDAT Box for metadta without special libraries

In [ ]:
#ifndef HEIF_MDAT_PARSER_DEFINED
#define HEIF_MDAT_PARSER_DEFINED

// Function to extract actual tile byte offsets from mdat
// This reads the file and calculates where each tile is stored

struct MdatTileLocations {
    uint64_t mdatStart;
    std::vector<std::pair<uint64_t, uint64_t>> tileOffsets; // (offset, size) pairs
};

MdatTileLocations parseMdatTileLocations(const std::string& filepath, 
                                         const std::vector<LibheifTileInfo>& levels) {
    MdatTileLocations result;
    
    // The mdat box starts at offset 6547 (from our parsing)
    result.mdatStart = 6547;
    
    // Read the mdat box header to get actual data start
    std::vector<uint8_t> mdatHeader = readFileRange(filepath, result.mdatStart, 16);
    
    // Parse mdat box header
    uint64_t mdatBoxSize = readBE32(mdatHeader.data());
    uint32_t mdatType = readBE32(mdatHeader.data() + 4);
    
    size_t mdatHeaderSize = 8;
    if (mdatBoxSize == 1) {
        // 64-bit size
        mdatBoxSize = readBE64(mdatHeader.data() + 8);
        mdatHeaderSize = 16;
    }
    
    uint64_t mdatDataStart = result.mdatStart + mdatHeaderSize;
    
    std::cout << "\nmdat box at offset " << result.mdatStart << "\n";
    std::cout << "mdat header size: " << mdatHeaderSize << " bytes\n";
    std::cout << "mdat data starts at: " << mdatDataStart << "\n";
    std::cout << "mdat total size: " << mdatBoxSize << " bytes\n";
    
    // For UNCI deflate tiles, we need to scan through the mdat to find tile boundaries
    // Deflate compressed data has identifiable headers
    // For now, estimate based on total size and tile count
    
    uint32_t totalTiles = 0;
    for (const auto& level : levels) {
        totalTiles += level.tiles.size();
    }
    
    uint64_t mdatDataSize = mdatBoxSize - mdatHeaderSize;
    uint64_t avgTileSize = mdatDataSize / totalTiles;
    
    std::cout << "Total tiles: " << totalTiles << "\n";
    std::cout << "mdat data size: " << mdatDataSize << " bytes\n";
    std::cout << "Average tile size: " << avgTileSize << " bytes\n";
    
    // PROPER SOLUTION: Parse tile index if present, or use libheif API
    std::cout << "\n=== LIMITATION ===\n";
    std::cout << "Cannot determine exact tile byte offsets without:\n";
    std::cout << "1. Parsing UNCI tile index (if present in file)\n";
    std::cout << "2. Scanning deflate stream boundaries (complex)\n";
    std::cout << "3. Using libheif API to extract tiles\n\n";
    
    std::cout << "RECOMMENDED APPROACH:\n";
    std::cout << "Use libheif API to extract tiles programmatically:\n\n";
    std::cout << "```cpp\n";
    std::cout << "heif_context* ctx = heif_context_alloc();\n";
    std::cout << "heif_context_read_from_file(ctx, \"file.heif\", nullptr);\n";
    std::cout << "heif_image_handle* handle;\n";
    std::cout << "heif_context_get_image_handle(ctx, <itemID>, &handle);\n";
    std::cout << "heif_image_tiling tiling;\n";
    std::cout << "heif_image_handle_get_tiling_info(handle, &tiling);\n";
    std::cout << "// Then extract each tile:\n";
    std::cout << "for (int ty = 0; ty < tiling.num_rows; ty++) {\n";
    std::cout << "  for (int tx = 0; tx < tiling.num_cols; tx++) {\n";
    std::cout << "    heif_image* tile;\n";
    std::cout << "    heif_image_handle_get_tile(handle, tx, ty, &tile);\n";
    std::cout << "    // Process tile...\n";
    std::cout << "  }\n";
    std::cout << "}\n";
    std::cout << "```\n\n";
    
    return result;
}

#endif // HEIF_MDAT_PARSER_DEFINED

###  Test/Example Usage for Meta Box Parser

#### Main

In [ ]:
#ifndef TEST_META_PARSER_DEFINED
#define TEST_META_PARSER_DEFINED

void testMetaParser(const std::string& source, SourceType type, 
                    size_t offset, size_t byteCount) {
    try {
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Testing Meta Box Parser                          ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        // Read meta box data
        std::vector<uint8_t> data;
        
        if (type == SourceType::FILE) {
            std::cout << "Source: FILE\n";
            std::cout << "Path: " << source << "\n";
            data = readFileRange(source, offset, byteCount);
        } else if (type == SourceType::HTTP_RANGE) {
            std::cout << "Source: HTTP/HTTPS\n";
            std::cout << "URL: " << source << "\n";
            data = readHttpRange(source, offset, byteCount);
        }
        
        std::cout << "Offset: " << offset << " bytes\n";
        std::cout << "Byte Count: " << byteCount << " bytes\n";
        std::cout << "Retrieved: " << data.size() << " bytes\n\n";
        
        // Parse meta box
        MetaBoxInfo metaInfo = parseMetaBox(data, offset);
        
        // Print detailed summary
        metaInfo.printSummary();
        
        // Print detailed iloc information
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Detailed Item Location (iloc) Analysis            ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        if (metaInfo.itemLocations.empty()) {
            std::cout << "No item locations found.\n";
        } else {
            std::cout << "Total Items with Location Data: " << metaInfo.itemLocations.size() << "\n\n";
            
            for (const auto& loc : metaInfo.itemLocations) {
                std::cout << "─────────────────────────────────────────────────────────────\n";
                loc.print();
                std::cout << "\n";
            }
        }
        
        // Build and display item data requests
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Formed Item Data Requests (All Tiles/Items)       ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        auto requests = buildItemDataRequests(metaInfo);
        
        if (requests.empty()) {
            std::cout << "No items with location data found.\n";
        } else {
            std::cout << "Total Retrievable Items: " << requests.size() << "\n\n";
            
            // Group by type
            std::map<std::string, std::vector<ItemDataRequest>> itemsByType;
            for (const auto& req : requests) {
                itemsByType[req.itemType].push_back(req);
            }
            
            // Print statistics
            std::cout << "Items by Type:\n";
            for (const auto& [itemType, items] : itemsByType) {
                std::cout << "  " << itemType << ": " << items.size() << " item(s)\n";
            }
            std::cout << "\n";
            
            // Count tiles/pyramids
            int numFullRes = 0;
            int numOverviews = 0;
            int numThumbnails = 0;
            int numAuxiliary = 0;
            
            for (const auto& req : requests) {
                if (req.isThumbnail) numThumbnails++;
                else if (req.isAuxiliary) numAuxiliary++;
                else if (req.pyramidLevel == 0) numFullRes++;
                else numOverviews++;
            }
            
            std::cout << "Item Categories:\n";
            std::cout << "  Full Resolution: " << numFullRes << "\n";
            std::cout << "  Overview/Pyramid Levels: " << numOverviews << "\n";
            std::cout << "  Thumbnails: " << numThumbnails << "\n";
            std::cout << "  Auxiliary Images: " << numAuxiliary << "\n";
            std::cout << "\n";
            
            // Print detailed information for each item
            std::cout << "═══════════════════════════════════════════════════════════\n";
            std::cout << "Detailed Item Information:\n";
            std::cout << "═══════════════════════════════════════════════════════════\n\n";
            
            for (size_t i = 0; i < requests.size(); ++i) {
                std::cout << "┌─── Item " << (i + 1) << " of " << requests.size() << " ───";
                std::cout << std::string(40, '-') << "┐\n";
                requests[i].print();
                std::cout << "└" << std::string(63, '-') << "┘\n\n";
            }
            
            // Show how to retrieve each item
            std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
            std::cout << "║          Retrieval Commands for All Items                  ║\n";
            std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
            
            // Group commands by category
            std::cout << "// ========== FULL RESOLUTION ITEMS ==========\n";
            for (size_t i = 0; i < requests.size(); ++i) {
                const auto& req = requests[i];
                if (!req.isThumbnail && !req.isAuxiliary && req.pyramidLevel == 0) {
                    std::cout << "// Item " << req.itemID << " (" << req.itemType << ")";
                    if (!req.name.empty()) std::cout << " - \"" << req.name << "\"";
                    std::cout << " - " << req.width << "x" << req.height;
                    std::cout << " - " << req.byteCount << " bytes\n";
                    
                    if (type == SourceType::FILE) {
                        std::cout << "auto item" << req.itemID << "_data = readFileRange(\""
                                  << source << "\", " << req.offset << ", " << req.byteCount << ");\n";
                    } else {
                        std::cout << "auto item" << req.itemID << "_data = readHttpRange(\""
                                  << source << "\", " << req.offset << ", " << req.byteCount << ");\n";
                    }
                    std::cout << "\n";
                }
            }
            
            if (numOverviews > 0) {
                std::cout << "\n// ========== OVERVIEW/PYRAMID LEVELS ==========\n";
                for (size_t i = 0; i < requests.size(); ++i) {
                    const auto& req = requests[i];
                    if (!req.isThumbnail && !req.isAuxiliary && req.pyramidLevel > 0) {
                        std::cout << "// Item " << req.itemID << " (" << req.itemType << ")";
                        if (!req.name.empty()) std::cout << " - \"" << req.name << "\"";
                        std::cout << " - Level " << req.pyramidLevel;
                        std::cout << " - " << req.width << "x" << req.height;
                        std::cout << " - " << req.byteCount << " bytes\n";
                        
                        if (type == SourceType::FILE) {
                            std::cout << "auto item" << req.itemID << "_overview" << req.pyramidLevel 
                                      << " = readFileRange(\"" << source << "\", " 
                                      << req.offset << ", " << req.byteCount << ");\n";
                        } else {
                            std::cout << "auto item" << req.itemID << "_overview" << req.pyramidLevel 
                                      << " = readHttpRange(\"" << source << "\", " 
                                      << req.offset << ", " << req.byteCount << ");\n";
                        }
                        std::cout << "\n";
                    }
                }
            }
            
            if (numThumbnails > 0) {
                std::cout << "\n// ========== THUMBNAILS ==========\n";
                for (size_t i = 0; i < requests.size(); ++i) {
                    const auto& req = requests[i];
                    if (req.isThumbnail) {
                        std::cout << "// Item " << req.itemID << " (THUMBNAIL)";
                        if (!req.name.empty()) std::cout << " - \"" << req.name << "\"";
                        std::cout << " - " << req.width << "x" << req.height;
                        std::cout << " - " << req.byteCount << " bytes\n";
                        
                        if (type == SourceType::FILE) {
                            std::cout << "auto item" << req.itemID << "_thumbnail = readFileRange(\""
                                      << source << "\", " << req.offset << ", " << req.byteCount << ");\n";
                        } else {
                            std::cout << "auto item" << req.itemID << "_thumbnail = readHttpRange(\""
                                      << source << "\", " << req.offset << ", " << req.byteCount << ");\n";
                        }
                        std::cout << "\n";
                    }
                }
            }
            
            if (numAuxiliary > 0) {
                std::cout << "\n// ========== AUXILIARY IMAGES ==========\n";
                for (size_t i = 0; i < requests.size(); ++i) {
                    const auto& req = requests[i];
                    if (req.isAuxiliary) {
                        std::cout << "// Item " << req.itemID << " (AUXILIARY: " << req.auxType << ")";
                        if (!req.name.empty()) std::cout << " - \"" << req.name << "\"";
                        std::cout << " - " << req.width << "x" << req.height;
                        std::cout << " - " << req.byteCount << " bytes\n";
                        
                        if (type == SourceType::FILE) {
                            std::cout << "auto item" << req.itemID << "_aux = readFileRange(\""
                                      << source << "\", " << req.offset << ", " << req.byteCount << ");\n";
                        } else {
                            std::cout << "auto item" << req.itemID << "_aux = readHttpRange(\""
                                      << source << "\", " << req.offset << ", " << req.byteCount << ");\n";
                        }
                        std::cout << "\n";
                    }
                }
            }
            
            // Print tile information if tiling is detected
            std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
            std::cout << "║          Tile/Multi-Extent Analysis                        ║\n";
            std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
            
            bool foundMultiExtent = false;
            for (const auto& loc : metaInfo.itemLocations) {
                if (loc.extents.size() > 1) {
                    foundMultiExtent = true;
                    std::cout << "Item ID " << loc.itemID << " has " << loc.extents.size() 
                              << " extents (possibly tiled):\n";
                    
                    for (size_t i = 0; i < loc.extents.size(); ++i) {
                        uint64_t absOffset = loc.baseOffset + loc.extents[i].first;
                        std::cout << "  Tile/Extent " << (i + 1) << ": offset=" << absOffset 
                                  << ", length=" << loc.extents[i].second << " bytes\n";
                    }
                    std::cout << "\n";
                }
            }
            
            if (!foundMultiExtent) {
                std::cout << "No multi-extent (tiled) items detected.\n";
                std::cout << "All items use single contiguous data blocks.\n";
            }
            
            // Print uncC configuration if present for unci items
            std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
            std::cout << "║          Uncompressed Image Configuration (uncC)           ║\n";
            std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
            
            bool foundUnci = false;
            for (const auto& req : requests) {
                if (req.itemType == "unci" && req.uncompressedConfig) {
                    foundUnci = true;
                    std::cout << "Item ID " << req.itemID;
                    if (!req.name.empty()) std::cout << " (\"" << req.name << "\")";
                    std::cout << ":\n";
                    std::cout << "  Dimensions: " << req.width << " x " << req.height << "\n";
                    std::cout << "  Configuration: ";
                    req.uncompressedConfig->print();
                    std::cout << "\n";
                    
                    // Calculate expected data size
                    if (req.uncompressedConfig->componentFormat.size() > 0) {
                        uint64_t totalBits = 0;
                        for (uint8_t bits : req.uncompressedConfig->componentFormat) {
                            totalBits += bits;
                        }
                        uint64_t expectedSize = (static_cast<uint64_t>(req.width) * 
                                                 req.height * totalBits + 7) / 8;
                        std::cout << "  Expected uncompressed size: " << expectedSize << " bytes\n";
                        std::cout << "  Actual data size: " << req.byteCount << " bytes\n";
                        
                        if (req.byteCount < expectedSize) {
                            double ratio = static_cast<double>(expectedSize) / req.byteCount;
                            std::cout << "  Compression ratio: ~" << std::fixed 
                                      << std::setprecision(2) << ratio << ":1\n";
                        }
                    }
                    
                    // Show tile configuration if present
                    if (req.uncompressedConfig->numTileColsMinus1 > 0 || 
                        req.uncompressedConfig->numTileRowsMinus1 > 0) {
                        uint32_t numCols = req.uncompressedConfig->numTileColsMinus1 + 1;
                        uint32_t numRows = req.uncompressedConfig->numTileRowsMinus1 + 1;
                        uint32_t totalTiles = numCols * numRows;
                        
                        std::cout << "  Tile Grid: " << numCols << " x " << numRows 
                                  << " (" << totalTiles << " tiles total)\n";
                        
                        // Estimate tile dimensions
                        uint32_t tileWidth = req.width / numCols;
                        uint32_t tileHeight = req.height / numRows;
                        std::cout << "  Approximate tile size: " << tileWidth << " x " 
                                  << tileHeight << " pixels\n";
                    }
                    
                    std::cout << "\n";
                }
            }
            
            if (!foundUnci) {
                std::cout << "No uncompressed (unci) items found.\n";
            }
            
            // Summary statistics
            std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
            std::cout << "║          Summary Statistics                                ║\n";
            std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
            
            uint64_t totalBytes = 0;
            uint64_t minSize = UINT64_MAX;
            uint64_t maxSize = 0;
            
            for (const auto& req : requests) {
                totalBytes += req.byteCount;
                minSize = safe_min(minSize, req.byteCount);
                maxSize = safe_max(maxSize, req.byteCount);
            }
            
            std::cout << "Total Items: " << requests.size() << "\n";
            std::cout << "Total Data Size: " << totalBytes << " bytes";
            if (totalBytes > 1024 * 1024) {
                std::cout << " (" << (totalBytes / (1024.0 * 1024.0)) << " MB)";
            }
            std::cout << "\n";
            std::cout << "Smallest Item: " << minSize << " bytes\n";
            std::cout << "Largest Item: " << maxSize << " bytes\n";
            std::cout << "Average Item Size: " << (totalBytes / requests.size()) << " bytes\n";
            
            if (metaInfo.hasPrimaryItem) {
                bool foundPrimary = false;
                for (const auto& req : requests) {
                    if (req.itemID == metaInfo.primaryItemID) {
                        std::cout << "\nPrimary Item: ID " << req.itemID;
                        if (!req.name.empty()) std::cout << " (\"" << req.name << "\")";
                        std::cout << "\n";
                        std::cout << "  Type: " << req.itemType << "\n";
                        std::cout << "  Dimensions: " << req.width << " x " << req.height << "\n";
                        std::cout << "  Size: " << req.byteCount << " bytes\n";
                        foundPrimary = true;
                        break;
                    }
                }
            }
        }
        
        std::cout << "\n✓ Meta box parsing completed successfully!\n";
        
    } catch (const std::exception& e) {
        std::cerr << "\n✗ Error: " << e.what() << "\n";
    }
}

// Convenience overload with string type
void testMetaParser(const std::string& source, const std::string& typeStr,
                    size_t offset, size_t byteCount) {
    SourceType type;
    if (typeStr == "file" || typeStr == "FILE") {
        type = SourceType::FILE;
    } else if (typeStr == "http" || typeStr == "HTTP" || 
               typeStr == "http-range" || typeStr == "HTTP-RANGE" ||
               typeStr == "https" || typeStr == "HTTPS") {
        type = SourceType::HTTP_RANGE;
    } else {
        std::cerr << "Error: Invalid type '" << typeStr << "'. Use 'file' or 'http-range'.\n";
        return;
    }
    testMetaParser(source, type, offset, byteCount);
}

// Integrated test: parse ftyp, then automatically parse meta
void testFullHeifParsing(const std::string& source, SourceType type) {
    try {
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Full HEIF Parsing Workflow                        ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        // Step 1: Read initial bytes and parse ftyp
        std::cout << "Step 1: Reading initial bytes and parsing ftyp...\n\n";
        
        std::vector<uint8_t> initialData;
        if (type == SourceType::FILE) {
            initialData = readFileRange(source, 0, 1024);
        } else {
            initialData = readHttpRange(source, 0, 1024);
        }
        
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        if (!metaLoc.found) {
            std::cout << "\n✗ Meta box not found in first 1024 bytes. Try reading more data.\n";
            return;
        }
        
        std::cout << "\n✓ Found meta box at offset " << metaLoc.offset 
                  << ", size " << metaLoc.size << " bytes\n\n";
        
        // Step 2: Read and parse meta box
        std::cout << "Step 2: Reading and parsing meta box...\n\n";
        
        testMetaParser(source, type, metaLoc.offset, metaLoc.size);
        
    } catch (const std::exception& e) {
        std::cerr << "\n✗ Error: " << e.what() << "\n";
    }
}

// Convenience overload for full parsing
void testFullHeifParsing(const std::string& source, const std::string& typeStr) {
    SourceType type;
    if (typeStr == "file" || typeStr == "FILE") {
        type = SourceType::FILE;
    } else if (typeStr == "http" || typeStr == "HTTP" || 
               typeStr == "http-range" || typeStr == "HTTP-RANGE" ||
               typeStr == "https" || typeStr == "HTTPS") {
        type = SourceType::HTTP_RANGE;
    } else {
        std::cerr << "Error: Invalid type '" << typeStr << "'. Use 'file' or 'http-range'.\n";
        return;
    }
    testFullHeifParsing(source, type);
}

// Add to test section

void testTileExtraction(const std::string& source, SourceType type) {
    try {
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Tile-Level Metadata Extraction                    ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        // Parse HEIF file
        std::vector<uint8_t> initialData;
        if (type == SourceType::FILE) {
            initialData = readFileRange(source, 0, 8192); // Read even more initial data
        } else {
            initialData = readHttpRange(source, 0, 8192);
        }
        
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        if (!metaLoc.found) {
            std::cout << "✗ Meta box not found in first bytes, reading more...\n";
            // Try reading more
            if (type == SourceType::FILE) {
                initialData = readFileRange(source, 0, 65536);
            } else {
                initialData = readHttpRange(source, 0, 65536);
            }
            auto result = parseFtypAndLocateMeta(initialData, 0);
            ftyp = result.first;
            metaLoc = result.second;
            
            if (!metaLoc.found) {
                std::cout << "✗ Still no meta box found\n";
                return;
            }
        }
        
        std::cout << "✓ Found meta box at offset " << metaLoc.offset 
                  << ", size " << metaLoc.size << " bytes\n";
        
        std::vector<uint8_t> metaData;
        if (type == SourceType::FILE) {
            metaData = readFileRange(source, metaLoc.offset, metaLoc.size);
        } else {
            metaData = readHttpRange(source, metaLoc.offset, metaLoc.size);
        }
        
        std::cout << "✓ Read " << metaData.size() << " bytes of meta box\n";
        
        MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
        
        // DEBUG: Show item structure
        debugItemStructure(metaInfo);
        
        // Extract all tiles
        std::cout << "\nAttempting standard tile extraction...\n";
        auto tiles = extractAllTiles(metaInfo);
        
        if (tiles.empty()) {
            std::cout << "\nStandard extraction failed. Trying special handler...\n";
            tiles = extractTilesFromUnciMultiLevel(metaInfo);
        }

        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Extracted Tiles Summary                           ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        std::cout << "Total Tiles: " << tiles.size() << "\n\n";
        
        // Group by scheme and level
        std::map<std::string, std::map<int, std::vector<TileInfo>>> tilesBySchemeAndLevel;
        for (const auto& tile : tiles) {
            tilesBySchemeAndLevel[tile.tilingScheme][tile.pyramidLevel].push_back(tile);
        }
        
        // Print pyramid structure
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Pyramid Structure                                 ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        for (const auto& [scheme, levelMap] : tilesBySchemeAndLevel) {
            std::cout << "Tiling Scheme: " << scheme << "\n";
            std::cout << "├─ Pyramid Levels: " << levelMap.size() << "\n";
            
            for (const auto& [level, levelTiles] : levelMap) {
                std::cout << "├─ Level " << level << ": " << levelTiles.size() << " tiles";
                
                if (!levelTiles.empty()) {
                    // Calculate grid dimensions
                    uint32_t maxRow = 0, maxCol = 0;
                    uint64_t totalBytes = 0;
                    for (const auto& tile : levelTiles) {
                        maxRow = safe_max(maxRow, tile.row);
                        maxCol = safe_max(maxCol, tile.col);
                        totalBytes += tile.byteCount;
                    }
                    
                    std::cout << " [" << (maxCol + 1) << "×" << (maxRow + 1) << " grid]";
                    
                    if (levelTiles[0].width > 0) {
                        // Estimate full resolution
                        uint32_t fullWidth = levelTiles[0].width * (maxCol + 1);
                        uint32_t fullHeight = levelTiles[0].height * (maxRow + 1);
                        std::cout << " [~" << fullWidth << "×" << fullHeight << "px]";
                    }
                    
                    std::cout << " [" << (totalBytes / (1024.0 * 1024.0)) << " MB]";
                }
                std::cout << "\n";
            }
            std::cout << "\n";
        }
        
        // Show tile details for each level (first 5 tiles per level)
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Sample Tile Details (first 5 per level)           ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        for (const auto& [scheme, levelMap] : tilesBySchemeAndLevel) {
            for (const auto& [level, levelTiles] : levelMap) {
                std::cout << "─────────────────────────────────────────────────────────────\n";
                std::cout << "Level " << level << " (" << scheme << ") - Showing " 
                          << safe_min(size_t(5), levelTiles.size()) << " of " 
                          << levelTiles.size() << " tiles:\n\n";
                
                for (size_t i = 0; i < safe_min(size_t(5), levelTiles.size()); ++i) {
                    levelTiles[i].print();
                    std::cout << "\n";
                }
                
                if (levelTiles.size() > 5) {
                    std::cout << "... (" << (levelTiles.size() - 5) << " more tiles)\n\n";
                }
            }
        }
        
        // Generate read commands for first tile of each level
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Sample Read Commands (first tile per level)       ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        for (const auto& [scheme, levelMap] : tilesBySchemeAndLevel) {
            for (const auto& [level, levelTiles] : levelMap) {
                if (!levelTiles.empty()) {
                    const auto& tile = levelTiles[0];
                    std::cout << "// Level " << level << ", Tile [0,0] - " 
                              << tile.width << "×" << tile.height << "px, "
                              << tile.byteCount << " bytes\n";
                    
                    if (type == SourceType::FILE) {
                        std::cout << "auto level" << level << "_tile0_data = readFileRange(\""
                                  << source << "\", " << tile.byteOffset << ", " 
                                  << tile.byteCount << ");\n";
                    } else {
                        std::cout << "auto level" << level << "_tile0_data = readHttpRange(\""
                                  << source << "\", " << tile.byteOffset << ", " 
                                  << tile.byteCount << ");\n";
                    }
                    std::cout << "\n";
                }
            }
        }
        
        std::cout << "✓ Tile extraction completed!\n";
        std::cout << "✓ Found " << tiles.size() << " total tiles across " 
                  << tilesBySchemeAndLevel.begin()->second.size() << " pyramid levels\n";
          
    } catch (const std::exception& e) {
        std::cerr << "\n✗ Error: " << e.what() << "\n";
    }
}

// Add this to help visualize the tile structure

void printTileGridASCII(const std::vector<TileInfo>& tiles, int level) {
    // Filter tiles by level
    std::vector<TileInfo> levelTiles;
    for (const auto& tile : tiles) {
        if (tile.pyramidLevel == level) {
            levelTiles.push_back(tile);
        }
    }
    
    if (levelTiles.empty()) {
        std::cout << "No tiles at level " << level << "\n";
        return;
    }
    
    // Find grid dimensions
    uint32_t maxRow = 0, maxCol = 0;
    for (const auto& tile : levelTiles) {
        maxRow = safe_max(maxRow, tile.row);
        maxCol = safe_max(maxCol, tile.col);
    }
    
    uint32_t gridRows = maxRow + 1;
    uint32_t gridCols = maxCol + 1;
    
    std::cout << "Tile Grid for Level " << level << " [" << gridRows << "×" << gridCols << "]:\n";
    std::cout << "┌" << std::string(gridCols * 4 + 1, '-') << "┐\n";
    
    for (uint32_t r = 0; r < gridRows; ++r) {
        std::cout << "│ ";
        for (uint32_t c = 0; c < gridCols; ++c) {
            bool found = false;
            for (const auto& tile : levelTiles) {
                if (tile.row == r && tile.col == c) {
                    std::cout << "■ ";
                    found = true;
                    break;
                }
            }
            if (!found) {
                std::cout << "□ ";
            }
        }
        std::cout << " │\n";
    }
    
    std::cout << "└" << std::string(gridCols * 4 + 1, '-') << "┘\n";
    std::cout << "■ = tile present, □ = tile missing\n\n";
}

// Updated test
void testCompleteExtractionWithIcef(const std::string& filepath) {
    std::cout << "╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Complete Tile Extraction with icef                ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    // Parse metadata
    std::vector<uint8_t> initialData = readFileRange(filepath, 0, 8192);
    auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
    
    std::vector<uint8_t> metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
    MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
    
    // Extract with icef offsets
    auto levels = extractLibheifUnciTilesWithOffsets(metaInfo, filepath, true);
    
    // Summary
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Summary                                           ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    int totalTiles = 0;
    for (const auto& level : levels) {
        std::cout << "Level " << level.pyramidLevel << ": " 
                  << level.tiles.size() << " tiles";
        if (!level.tiles.empty() && level.tiles[0].byteCount > 0) {
            std::cout << " [offsets available]";
            totalTiles += level.tiles.size();
        }
        std::cout << "\n";
    }
    
    std::cout << "\nTotal tiles with offsets: " << totalTiles << "\n";
    
    // Show first tile from each level
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Sample Tiles (first from each level)              ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    for (const auto& level : levels) {
        if (!level.tiles.empty()) {
            const auto& tile = level.tiles[0];
            std::cout << "Level " << level.pyramidLevel << ", Tile [0,0]:\n";
            std::cout << "  Offset: " << tile.byteOffset << " bytes\n";
            std::cout << "  Size: " << tile.byteCount << " bytes\n";
            std::cout << "  Dimensions: " << tile.width << "×" << tile.height << "\n\n";
        }
    }
}

// Enhanced test function
void testLibheifUnciTiles(const std::string& source, SourceType type) {
    try {
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          libheif UNCI Tiled Pyramid Extraction             ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        // Parse file
        std::vector<uint8_t> initialData;
        if (type == SourceType::FILE) {
            initialData = readFileRange(source, 0, 8192);
        } else {
            initialData = readHttpRange(source, 0, 8192);
        }
        
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        if (!metaLoc.found) {
            std::cout << "✗ Meta box not found\n";
            return;
        }
        
        std::vector<uint8_t> metaData;
        if (type == SourceType::FILE) {
            metaData = readFileRange(source, metaLoc.offset, metaLoc.size);
        } else {
            metaData = readHttpRange(source, metaLoc.offset, metaLoc.size);
        }
        
        MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
        
        // Extract tiles
        auto levels = extractLibheifUnciTiles(metaInfo, source, type == SourceType::FILE);
        
        // Display results
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Extraction Summary                                ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        int totalTiles = 0;
        for (const auto& level : levels) {
            totalTiles += level.tiles.size();
            std::cout << "Level " << level.pyramidLevel << " (Item " << level.itemID << "): ";
            std::cout << level.imageWidth << "×" << level.imageHeight << ", ";
            std::cout << level.numCols << "×" << level.numRows << " grid, ";
            std::cout << level.tiles.size() << " tiles\n";
        }
        
        std::cout << "\nTotal: " << levels.size() << " levels, " << totalTiles << " tiles\n";
        
        // At the end of testLibheifUnciTiles, after displaying the summary:

        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Attempting mdat Analysis                          ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n";

        if (type == SourceType::FILE) {
            auto mdatInfo = parseMdatTileLocations(source, levels);
        }

        // Show sample tiles from each level
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Sample Tiles (first 3 per level)                  ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        for (const auto& level : levels) {
            std::cout << "Level " << level.pyramidLevel << ":\n";
            for (size_t i = 0; i < std::min(size_t(3), level.tiles.size()); ++i) {
                const auto& tile = level.tiles[i];
                std::cout << "  Tile [" << tile.row << "," << tile.col << "]: ";
                std::cout << tile.width << "×" << tile.height << "px at (" << tile.xOffset << "," << tile.yOffset << "), ";
                if (tile.byteCount > 0) {
                    std::cout << tile.byteCount << " bytes @ offset " << tile.byteOffset;
                } else {
                    std::cout << "offset unknown";
                }
                std::cout << "\n";
            }
            if (level.tiles.size() > 3) {
                std::cout << "  ... (" << (level.tiles.size() - 3) << " more tiles)\n";
            }
            std::cout << "\n";
        }
        
        std::cout << "✓ Extraction complete!\n";
        
    } catch (const std::exception& e) {
        std::cerr << "✗ Error: " << e.what() << "\n";
    }
}


// Add to test section

void testGridTileExtractionWithOffsets(const std::string& filepath) {
    try {
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Grid Tile Extraction with Byte Offsets           ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        // Parse metadata
        std::vector<uint8_t> initialData = readFileRange(filepath, 0, 8192);
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        std::vector<uint8_t> metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
        MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
        
        // Extract tiles with offsets
        auto levels = extractGridTilesWithOffsets(metaInfo);
        
        // Display results
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Tile Extraction Summary                           ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        int totalTiles = 0;
        int tilesWithOffsets = 0;
        
        for (const auto& level : levels) {
            int levelWithOffsets = 0;
            for (const auto& tile : level.tiles) {
                if (tile.byteCount > 0) {
                    levelWithOffsets++;
                }
            }
            
            totalTiles += level.tiles.size();
            tilesWithOffsets += levelWithOffsets;
            
            std::cout << "Level " << level.pyramidLevel << ": "
                      << levelWithOffsets << "/" << level.tiles.size()
                      << " tiles with offsets\n";
        }
        
        std::cout << "\nTotal: " << tilesWithOffsets << "/" << totalTiles
                  << " tiles have byte offset information\n";
        
        // Show sample tiles
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Sample Tile Information (first 3 per level)       ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        for (const auto& level : levels) {
            std::cout << "Level " << level.pyramidLevel << " (" << level.imageWidth
                      << "×" << level.imageHeight << "):\n";
            
            for (size_t i = 0; i < std::min(size_t(3), level.tiles.size()); ++i) {
                const auto& tile = level.tiles[i];
                std::cout << "  Tile [" << tile.row << "," << tile.col << "]: ";
                std::cout << tile.width << "×" << tile.height << "px at ("
                          << tile.xOffset << "," << tile.yOffset << "), ";
                
                if (tile.byteCount > 0) {
                    std::cout << tile.byteCount << " bytes @ offset " << tile.byteOffset;
                } else {
                    std::cout << "no offset data";
                }
                std::cout << "\n";
            }
            std::cout << "\n";
        }
        
        // Generate sample read commands
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Sample Read Commands                              ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        for (const auto& level : levels) {
            if (!level.tiles.empty() && level.tiles[0].byteCount > 0) {
                const auto& tile = level.tiles[0];
                std::cout << "// Level " << level.pyramidLevel << ", Tile [0,0]\n";
                std::cout << "auto level" << level.pyramidLevel << "_tile0 = readFileRange(\""
                          << filepath << "\", " << tile.byteOffset << ", "
                          << tile.byteCount << ");\n\n";
            }
        }
        
    } catch (const std::exception& e) {
        std::cerr << "✗ Error: " << e.what() << "\n";
    }
}


// Wrapper function for testing
void testGridTileExtraction(const std::string& filepath) {
    try {
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Grid Tile Extraction Test                         ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        // Parse metadata
        std::vector<uint8_t> initialData = readFileRange(filepath, 0, 8192);
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        std::vector<uint8_t> metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
        MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
        
        // Extract grid tiles
        auto tiles = extractGridTiles(metaInfo);
        
        // Summary
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Extraction Summary                                ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        std::cout << "Total tiles extracted: " << tiles.size() << "\n";
        
        if (!tiles.empty()) {
            int tilesWithOffsets = 0;
            uint64_t totalBytes = 0;
            
            for (const auto& tile : tiles) {
                if (tile.byteCount > 0) {
                    tilesWithOffsets++;
                    totalBytes += tile.byteCount;
                }
            }
            
            std::cout << "Tiles with byte offsets: " << tilesWithOffsets << "\n";
            std::cout << "Total data size: " << (totalBytes / (1024.0 * 1024.0)) 
                      << " MB\n\n";
            
            // Calculate grid from tiles
            uint32_t maxRow = 0, maxCol = 0;
            for (const auto& tile : tiles) {
                maxRow = std::max(maxRow, tile.row);
                maxCol = std::max(maxCol, tile.col);
            }
            
            std::cout << "Grid dimensions: " << (maxCol + 1) << "×" << (maxRow + 1) << "\n\n";
        }
        
        // Show sample tiles
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Sample Tiles (first 10)                           ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        for (size_t i = 0; i < std::min(size_t(10), tiles.size()); ++i) {
            const auto& tile = tiles[i];
            std::cout << "Tile " << i << " (Item " << tile.itemID << "):\n";
            std::cout << "  Position: [" << tile.row << "," << tile.col << "] at ("
                      << tile.xOffset << "," << tile.yOffset << ")\n";
            std::cout << "  Size: " << tile.width << "×" << tile.height << " pixels\n";
            
            if (tile.byteCount > 0) {
                std::cout << "  Data: " << tile.byteCount << " bytes @ offset " 
                          << tile.byteOffset << "\n";
            } else {
                std::cout << "  Data: No location info\n";
            }
            std::cout << "\n";
        }
        
        // Generate read commands for first few tiles
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Sample Read Commands (first 5 tiles)              ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        for (size_t i = 0; i < std::min(size_t(5), tiles.size()); ++i) {
            const auto& tile = tiles[i];
            if (tile.byteCount > 0) {
                std::cout << "// Tile " << i << " [" << tile.row << "," << tile.col 
                          << "] - " << tile.width << "×" << tile.height << "px\n";
                std::cout << "auto tile" << i << "_data = readFileRange(\""
                          << filepath << "\", " << tile.byteOffset << ", "
                          << tile.byteCount << ");\n\n";
            }
        }
        
        // Show grid visualization
        if (tiles.size() <= 400) { // Only show for reasonable sizes
            std::cout << "╔════════════════════════════════════════════════════════════╗\n";
            std::cout << "║          Tile Grid Visualization                           ║\n";
            std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
            
            printTileGridASCII(tiles, 0);
        }
        
    } catch (const std::exception& e) {
        std::cerr << "✗ Error: " << e.what() << "\n";
    }
}


// Test function
void testGridTileExtractionWithPyramid(const std::string& filepath) {
    try {
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Grid Tile Extraction Test                         ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        // Parse metadata
        std::vector<uint8_t> initialData = readFileRange(filepath, 0, 8192);
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        std::vector<uint8_t> metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
        MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
        
        // Extract grid tiles with pyramid
        auto levels = extractGridTilesWithPyramid(metaInfo);
        
        // Summary
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Extraction Summary                                ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        int totalTiles = 0;
        int totalWithOffsets = 0;
        uint64_t totalBytes = 0;
        
        for (const auto& level : levels) {
            int withOffsets = 0;
            uint64_t levelBytes = 0;
            
            for (const auto& tile : level.tiles) {
                if (tile.byteCount > 0) {
                    withOffsets++;
                    levelBytes += tile.byteCount;
                }
            }
            
            totalTiles += level.tiles.size();
            totalWithOffsets += withOffsets;
            totalBytes += levelBytes;
            
            std::cout << "Level " << level.pyramidLevel << " (" 
                      << level.imageWidth << "×" << level.imageHeight << "): "
                      << withOffsets << "/" << level.tiles.size() << " tiles, "
                      << (levelBytes / (1024.0 * 1024.0)) << " MB\n";
        }
        
        std::cout << "\nTotal: " << totalWithOffsets << "/" << totalTiles 
                  << " tiles with offsets, "
                  << (totalBytes / (1024.0 * 1024.0)) << " MB\n\n";
        
        // Show sample tiles from each level
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Sample Tiles (first 3 per level)                  ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        for (const auto& level : levels) {
            std::cout << "Level " << level.pyramidLevel << ":\n";
            
            for (size_t i = 0; i < std::min(size_t(3), level.tiles.size()); ++i) {
                const auto& tile = level.tiles[i];
                std::cout << "  Tile " << i << " (Item " << tile.itemID << ") ["
                          << tile.row << "," << tile.col << "]: "
                          << tile.width << "×" << tile.height << "px at ("
                          << tile.xOffset << "," << tile.yOffset << ")";
                
                if (tile.byteCount > 0) {
                    std::cout << ", " << tile.byteCount << " bytes @ " << tile.byteOffset;
                }
                std::cout << "\n";
            }
            std::cout << "\n";
        }
        
        // Sample read commands
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Sample Read Commands                              ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        for (const auto& level : levels) {
            if (!level.tiles.empty() && level.tiles[0].byteCount > 0) {
                const auto& tile = level.tiles[0];
                std::cout << "// Level " << level.pyramidLevel << ", Tile [0,0]\n";
                std::cout << "auto level" << level.pyramidLevel << "_tile0 = readFileRange(\""
                          << filepath << "\", " << tile.byteOffset << ", "
                          << tile.byteCount << ");\n\n";
            }
        }
        
    } catch (const std::exception& e) {
        std::cerr << "✗ Error: " << e.what() << "\n";
    }
}

void testGridPyramidExtraction(const std::string& filepath) {
    std::cout << "╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Grid Pyramid Tile Extraction Test                 ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    // Parse metadata
    std::vector<uint8_t> initialData = readFileRange(filepath, 0, 16384);
    auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
    
    std::vector<uint8_t> metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
    MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
    
    // Extract pyramid tiles
    auto levels = extractGridTilesWithPyramid(metaInfo);
    
    // Show sample tiles from each level
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Sample Tiles from Each Level                      ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    for (const auto& level : levels) {
        std::cout << "Level " << level.pyramidLevel << " samples:\n";
        for (size_t i = 0; i < std::min(size_t(3), level.tiles.size()); ++i) {
            const auto& tile = level.tiles[i];
            std::cout << "  Tile [" << tile.row << "," << tile.col << "]: "
                      << tile.width << "×" << tile.height << "px, "
                      << tile.byteCount << " bytes @ " << tile.byteOffset << "\n";
        }
        std::cout << "\n";
    }
}



// Add this diagnostic function right after the testGridTileExtraction function:

void diagnoseGridFile(const std::string& filepath) {
    std::cout << "╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Grid File Diagnostic                              ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    // Parse metadata
    std::vector<uint8_t> initialData = readFileRange(filepath, 0, 16384); // Read more
    auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
    
    if (!metaLoc.found) {
        std::cout << "✗ Meta box not found\n";
        return;
    }
    
    std::cout << "Meta box at offset: " << metaLoc.offset << ", size: " << metaLoc.size << "\n\n";
    
    std::vector<uint8_t> metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
    MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
    
    std::cout << "═══════════════════════════════════════════════════════════\n";
    std::cout << "PARSED DATA SUMMARY:\n";
    std::cout << "═══════════════════════════════════════════════════════════\n\n";
    
    std::cout << "Items (iinf): " << metaInfo.items.size() << "\n";
    std::cout << "Item Locations (iloc): " << metaInfo.itemLocations.size() << "\n";
    std::cout << "Item Property Associations (ipma): " << metaInfo.itemPropertyAssociations.size() << "\n";
    std::cout << "Property Container (ipco): " << metaInfo.propertyContainer.size() << "\n";
    std::cout << "Item References (iref): " << metaInfo.itemReferences.size() << "\n\n";
    
    // Show sample items
    std::cout << "Sample Items (first 10):\n";
    for (size_t i = 0; i < std::min(size_t(10), metaInfo.items.size()); ++i) {
        std::cout << "  Item " << metaInfo.items[i].itemID << ": " 
                  << metaInfo.items[i].itemType << "\n";
    }
    std::cout << "\n";
    
    // Show sample locations
    std::cout << "Sample Item Locations (first 10):\n";
    for (size_t i = 0; i < std::min(size_t(10), metaInfo.itemLocations.size()); ++i) {
        std::cout << "  Item " << metaInfo.itemLocations[i].itemID 
                  << ": " << metaInfo.itemLocations[i].extents.size() << " extents, "
                  << "base_offset=" << metaInfo.itemLocations[i].baseOffset << "\n";
    }
    std::cout << "\n";
    
    // Show sample property associations
    std::cout << "Sample Property Associations (first 10):\n";
    for (size_t i = 0; i < std::min(size_t(10), metaInfo.itemPropertyAssociations.size()); ++i) {
        std::cout << "  Item " << metaInfo.itemPropertyAssociations[i].itemID 
                  << ": " << metaInfo.itemPropertyAssociations[i].properties.size() 
                  << " properties\n";
    }
    std::cout << "\n";
    
    // Check dimg references
    std::cout << "dimg References:\n";
    for (const auto& ref : metaInfo.itemReferences) {
        if (ref.referenceType == "dimg") {
            int nonZero = 0;
            for (uint32_t id : ref.toItemIDs) {
                if (id != 0) nonZero++;
            }
            std::cout << "  Item " << ref.fromItemID << " -> " << nonZero << " tiles\n";
        }
    }
    std::cout << "\n";
    
    std::cout << "═══════════════════════════════════════════════════════════\n";
    std::cout << "ISSUE DIAGNOSIS:\n";
    std::cout << "═══════════════════════════════════════════════════════════\n\n";
    
    if (metaInfo.itemLocations.size() < 100) {
        std::cout << "⚠ PROBLEM: Only " << metaInfo.itemLocations.size() 
                  << " item locations parsed (expected ~1800+)\n";
        std::cout << "  → The iloc parser is not reading all entries correctly\n\n";
    }
    
    if (metaInfo.itemPropertyAssociations.size() < 100) {
        std::cout << "⚠ PROBLEM: Only " << metaInfo.itemPropertyAssociations.size() 
                  << " property associations parsed (expected ~1800+)\n";
        std::cout << "  → The ipma parser is not reading all entries correctly\n\n";
    }
    
    std::cout << "RECOMMENDATION:\n";
    std::cout << "The iloc and ipma parsers need to be fixed to read all entries.\n";
    std::cout << "This is likely a bug in the parseIlocBox() and parseIpmaBox() functions.\n";
}

// Test function
void testCompleteGridExtraction(const std::string& filepath) {
    std::cout << "╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Complete Grid Tile Extraction Test                ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    // Parse metadata
    std::vector<uint8_t> initialData = readFileRange(filepath, 0, 8192);
    auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
    
    std::vector<uint8_t> metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
    MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
    
    // Extract tiles
    auto tiles = extractGridTilesComplete(metaInfo, metaLoc.offset, metaLoc.size);
    
    // Generate read commands
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Sample Tile Read Commands                         ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    for (size_t i = 0; i < std::min(size_t(5), tiles.size()); ++i) {
        const auto& tile = tiles[i];
        std::cout << "// Tile [" << tile.row << "," << tile.col << "] - " 
                  << tile.width << "×" << tile.height << "px, " << tile.byteCount << " bytes\n";
        std::cout << "auto tile" << i << "_data = readFileRange(\"" << filepath 
                  << "\", " << tile.byteOffset << ", " << tile.byteCount << ");\n\n";
    }
    
    std::cout << "✓ All " << tiles.size() << " tiles have complete byte offset/count information!\n";
}

#endif // TEST_META_PARSER_DEFINED

#### Test unci

##### Metadata extraction test

In [ ]:
#ifndef TEST_META_PARSER_UNCI_TILES_DEFINED
#define TEST_META_PARSER_UNCI_TILES_DEFINED

void testUnciTileExtraction(const std::string& filepath) {
    try {
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Testing UNCI Tile Extraction                      ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        std::cout << "File: " << filepath << "\n\n";
        
        // Step 1: Parse ftyp and locate meta box
        std::cout << "Step 1: Parsing ftyp box...\n";
        std::vector<uint8_t> initialData;
        
        try {
            initialData = readFileRange(filepath, 0, 4096);
        } catch (const std::exception& e) {
            std::cerr << "ERROR: Failed to read file: " << e.what() << "\n";
            return;
        }
        
        if (initialData.empty()) {
            std::cerr << "ERROR: No data read from file\n";
            return;
        }
        
        std::cout << "Read " << initialData.size() << " bytes\n";
        
        // Parse ftyp and locate meta
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        if (!metaLoc.found) {
            std::cerr << "ERROR: Meta box not found in initial data\n";
            return;
        }
        
        std::cout << "\nMeta box found at offset: " << metaLoc.offset 
                  << ", size: " << metaLoc.size << " bytes\n\n";
        
        // Step 2: Read meta box
        std::cout << "Step 2: Reading meta box...\n";
        std::vector<uint8_t> metaData;
        
        try {
            metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
        } catch (const std::exception& e) {
            std::cerr << "ERROR: Failed to read meta box: " << e.what() << "\n";
            return;
        }
        
        if (metaData.empty()) {
            std::cerr << "ERROR: No meta data read\n";
            return;
        }
        
        std::cout << "Read " << metaData.size() << " bytes of meta box\n\n";
        
        // Step 3: Parse meta box
        std::cout << "Step 3: Parsing meta box...\n";
        MetaBoxInfo metaInfo;
        
        try {
            metaInfo = parseMetaBox(metaData, metaLoc.offset);
        } catch (const std::exception& e) {
            std::cerr << "ERROR: Failed to parse meta box: " << e.what() << "\n";
            return;
        }
        
        std::cout << "Meta box parsed successfully\n";
        std::cout << "  Items: " << metaInfo.items.size() << "\n";
        std::cout << "  Locations: " << metaInfo.itemLocations.size() << "\n";
        std::cout << "  Property associations: " << metaInfo.itemPropertyAssociations.size() << "\n\n";
        
        // Safety check
        if (metaInfo.itemPropertyAssociations.empty()) {
            std::cerr << "ERROR: No property associations found\n";
            return;
        }
        
        // Step 4: Debug item structure
        std::cout << "Step 4: Analyzing item structure...\n";
        try {
            debugItemStructure(metaInfo);
        } catch (const std::exception& e) {
            std::cerr << "WARNING: Debug failed: " << e.what() << "\n";
        }
        
        // Step 5: Extract UNCI tiles
        std::cout << "\nStep 5: Extracting UNCI tiles...\n";
        std::vector<TileInfo> tiles;
        
        try {
            tiles = extractUnciTiles(metaInfo);
        } catch (const std::exception& e) {
            std::cerr << "ERROR: Tile extraction failed: " << e.what() << "\n";
            return;
        }
        
        if (tiles.empty()) {
            std::cerr << "WARNING: No tiles extracted\n";
            return;
        }
        
        std::cout << "\n✓ Successfully extracted " << tiles.size() << " tiles\n";
        
        // Step 6: Display tile details
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Tile Details                                      ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        // Show first 10 tiles
        size_t numToShow = std::min(size_t(10), tiles.size());
        for (size_t i = 0; i < numToShow; ++i) {
            const auto& tile = tiles[i];
            std::cout << "Tile " << i << " [" << tile.row << "," << tile.col << "]: "
                      << tile.width << "×" << tile.height << "px at ("
                      << tile.xOffset << "," << tile.yOffset << "), "
                      << tile.byteCount << " bytes @ offset " << tile.byteOffset << "\n";
        }
        
        if (tiles.size() > numToShow) {
            std::cout << "... (" << (tiles.size() - numToShow) << " more tiles)\n";
        }
        
        // Show last tile
        if (tiles.size() > numToShow) {
            const auto& lastTile = tiles.back();
            std::cout << "\nLast tile [" << lastTile.row << "," << lastTile.col << "]: "
                      << lastTile.width << "×" << lastTile.height << "px at ("
                      << lastTile.xOffset << "," << lastTile.yOffset << "), "
                      << lastTile.byteCount << " bytes @ offset " << lastTile.byteOffset << "\n";
        }
        
        std::cout << "\n✓ Test completed successfully!\n";
        
    } catch (const std::exception& e) {
        std::cerr << "\n✗ FATAL ERROR: " << e.what() << "\n";
    } catch (...) {
        std::cerr << "\n✗ FATAL ERROR: Unknown exception\n";
    }
}


// Test function for UNCI pyramid
void testUnciPyramidExtraction(const std::string& filepath) {
    try {
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Testing UNCI Pyramid Tile Extraction              ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        std::cout << "File: " << filepath << "\n\n";
        
        // Step 1: Parse ftyp and locate meta
        std::cout << "Step 1: Parsing ftyp box...\n";
        auto initialData = readFileRange(filepath, 0, 4096);
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        if (!metaLoc.found) {
            std::cerr << "ERROR: Meta box not found\n";
            return;
        }
        
        std::cout << "\nMeta box at offset: " << metaLoc.offset 
                  << ", size: " << metaLoc.size << " bytes\n\n";
        
        // Step 2: Read meta box
        std::cout << "Step 2: Reading meta box...\n";
        auto metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
        std::cout << "Read " << metaData.size() << " bytes\n\n";
        
        // Step 3: Parse meta box
        std::cout << "Step 3: Parsing meta box...\n";
        MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
        std::cout << "Parsed successfully\n\n";
        
        // Step 4: Extract pyramid tiles
        std::cout << "Step 4: Extracting UNCI pyramid tiles...\n";
        auto pyramid = extractUnciPyramidTiles(metaInfo);
        
        // Step 5: Show sample tiles from each level
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Sample Tiles from Each Level                      ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        for (const auto& level : pyramid) {
            std::cout << "Level " << level.level << " samples:\n";
            for (size_t i = 0; i < std::min(size_t(5), level.tiles.size()); ++i) {
                const auto& t = level.tiles[i];
                std::cout << "  Tile [" << t.row << "," << t.col << "]: "
                          << t.width << "×" << t.height << "px at ("
                          << t.xOffset << "," << t.yOffset << "), "
                          << t.byteCount << " bytes @ offset " << t.byteOffset << "\n";
            }
            std::cout << "\n";
        }
        
        std::cout << "✓ Test completed successfully!\n";
        
    } catch (const std::exception& e) {
        std::cerr << "\n✗ Error: " << e.what() << "\n";
    }
}


#endif // TEST_META_PARSER_UNCI_TILES_DEFINED

##### Decompression tests for unci

###### Pyramid-unci

In [ ]:
#ifndef TEST_UNCI_DECOMPRESSION_DEFINED
#define TEST_UNCI_DECOMPRESSION_DEFINED

#include <zlib.h>

// Decompress a single tile and validate
bool testTileDecompression(const std::string& filepath, const TileInfo& tile) {
    try {
        // Read compressed data
        auto compressedData = readFileRange(filepath, tile.byteOffset, tile.byteCount);
        
        // Calculate expected decompressed size
        uint32_t bytesPerPixel = 3; // RGB
        if (tile.uncompressedConfig && !tile.uncompressedConfig->componentFormat.empty()) {
            bytesPerPixel = 0;
            for (uint8_t bits : tile.uncompressedConfig->componentFormat) {
                bytesPerPixel += (bits + 7) / 8;
            }
        }
        
        size_t expectedSize = static_cast<size_t>(tile.width) * tile.height * bytesPerPixel;
        std::vector<uint8_t> decompressedData(expectedSize);
        
        // Decompress using zlib
        z_stream stream = {};
        stream.avail_in = compressedData.size();
        stream.next_in = compressedData.data();
        stream.avail_out = decompressedData.size();
        stream.next_out = decompressedData.data();
        
        int ret = inflateInit2(&stream, -MAX_WBITS); // Raw deflate
        if (ret != Z_OK) {
            std::cerr << "    ✗ inflateInit failed: " << ret << "\n";
            return false;
        }
        
        ret = inflate(&stream, Z_FINISH);
        inflateEnd(&stream);
        
        if (ret != Z_STREAM_END) {
            std::cerr << "    ✗ inflate failed: " << ret << "\n";
            return false;
        }
        
        size_t actualSize = stream.total_out;
        
        std::cout << "    ✓ Decompressed successfully\n";
        std::cout << "      Compressed: " << tile.byteCount << " bytes\n";
        std::cout << "      Decompressed: " << actualSize << " bytes\n";
        std::cout << "      Expected: " << expectedSize << " bytes\n";
        std::cout << "      Ratio: " << (tile.byteCount > 0 ? (double)actualSize / tile.byteCount : 0.0) << ":1\n";
        
        if (actualSize != expectedSize) {
            std::cout << "      ⚠ Size mismatch (difference: " << (int64_t)actualSize - (int64_t)expectedSize << " bytes)\n";
        }
        
        return true;
        
    } catch (const std::exception& e) {
        std::cerr << "    ✗ Exception: " << e.what() << "\n";
        return false;
    }
}

// Test decompression for all tiles in a pyramid
void testUnciPyramidDecompression(const std::string& filepath) {
    try {
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          UNCI Pyramid Decompression Test                   ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        // Parse metadata
        std::vector<uint8_t> initialData = readFileRange(filepath, 0, 4096);
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        if (!metaLoc.found) {
            std::cerr << "✗ Meta box not found\n";
            return;
        }
        
        std::vector<uint8_t> metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
        MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
        
        // Extract pyramid
        auto pyramid = extractUnciPyramidTiles(metaInfo);
        
        if (pyramid.empty()) {
            std::cerr << "✗ No pyramid levels found\n";
            return;
        }
        
        // Test decompression for each level
        for (const auto& level : pyramid) {
            std::cout << "\n─────────────────────────────────────────────────────────────\n";
            std::cout << "Testing Level " << level.level << " (" << level.tiles.size() << " tiles):\n\n";
            
            int successCount = 0;
            int failCount = 0;
            
            // Test first 5 tiles of this level
            size_t tilesToTest = std::min(size_t(5), level.tiles.size());
            
            for (size_t i = 0; i < tilesToTest; ++i) {
                const auto& tile = level.tiles[i];
                std::cout << "  Tile " << i << " [" << tile.row << "," << tile.col << "]: "
                          << tile.width << "×" << tile.height << "px\n";
                
                if (testTileDecompression(filepath, tile)) {
                    successCount++;
                } else {
                    failCount++;
                }
                std::cout << "\n";
            }
            
            std::cout << "  Summary: " << successCount << " succeeded, " << failCount << " failed\n";
        }
        
        std::cout << "\n✓ Decompression test complete\n";
        
    } catch (const std::exception& e) {
        std::cerr << "\n✗ Error: " << e.what() << "\n";
    }
}

#endif // TEST_UNCI_DECOMPRESSION_DEFINED

###### Test decompression Functions

In [ ]:
#ifndef TEST_UNCI_TILE_DECOMPRESSION_DEFINED
#define TEST_UNCI_TILE_DECOMPRESSION_DEFINED

#include <zlib.h>
#include <vector>
#include <string>
#include <iostream>
#include <iomanip>
#include <cstring>

// Structure to hold decompression results
struct TileDecompressionResult {
    bool success;
    size_t compressedSize;
    size_t decompressedSize;
    size_t expectedSize;
    double compressionRatio;
    std::string errorMessage;
    std::vector<uint8_t> decompressedData;
    
    void print() const {
        if (success) {
            std::cout << "    ✓ Decompression successful\n";
            std::cout << "      Compressed size: " << compressedSize << " bytes\n";
            std::cout << "      Decompressed size: " << decompressedSize << " bytes\n";
            std::cout << "      Expected size: " << expectedSize << " bytes\n";
            std::cout << "      Compression ratio: " << std::fixed << std::setprecision(2) 
                      << compressionRatio << ":1\n";
            
            if (decompressedSize != expectedSize) {
                int64_t diff = (int64_t)decompressedSize - (int64_t)expectedSize;
                std::cout << "      ⚠ Size mismatch: " << (diff > 0 ? "+" : "") << diff << " bytes\n";
            }
        } else {
            std::cout << "    ✗ Decompression failed: " << errorMessage << "\n";
        }
    }
};

// Calculate expected uncompressed tile size
size_t calculateExpectedTileSize(const TileInfo& tile) {
    size_t bytesPerPixel = 3; // Default RGB
    
    if (tile.uncompressedConfig && !tile.uncompressedConfig->componentFormat.empty()) {
        bytesPerPixel = 0;
        for (uint8_t bits : tile.uncompressedConfig->componentFormat) {
            bytesPerPixel += (bits + 7) / 8;
        }
    } else if (!tile.bitsPerChannel.empty()) {
        bytesPerPixel = 0;
        for (uint8_t bits : tile.bitsPerChannel) {
            bytesPerPixel += (bits + 7) / 8;
        }
    }
    
    return static_cast<size_t>(tile.width) * tile.height * bytesPerPixel;
}

// Decompress a single tile using zlib (deflate)
TileDecompressionResult decompressTile(const std::string& filepath, const TileInfo& tile) {
    TileDecompressionResult result;
    result.success = false;
    result.compressedSize = tile.byteCount;
    result.decompressedSize = 0;
    result.expectedSize = calculateExpectedTileSize(tile);
    result.compressionRatio = 0.0;
    
    try {
        // Validate tile has location data
        if (tile.byteOffset == 0 || tile.byteCount == 0) {
            result.errorMessage = "No location data for tile";
            return result;
        }
        
        if (tile.byteOffset == 1) {
            result.errorMessage = "Tile is a reference to lower resolution";
            return result;
        }
        
        // Read compressed data
        std::vector<uint8_t> compressedData;
        try {
            compressedData = readFileRange(filepath, tile.byteOffset, tile.byteCount);
        } catch (const std::exception& e) {
            result.errorMessage = std::string("Failed to read tile data: ") + e.what();
            return result;
        }
        
        if (compressedData.empty()) {
            result.errorMessage = "No data read from file";
            return result;
        }
        
        // Allocate decompression buffer
        result.decompressedData.resize(result.expectedSize);
        
        // Initialize zlib stream
        z_stream stream = {};
        stream.avail_in = compressedData.size();
        stream.next_in = compressedData.data();
        stream.avail_out = result.decompressedData.size();
        stream.next_out = result.decompressedData.data();
        
        // Try raw deflate first (no zlib header)
        int ret = inflateInit2(&stream, -MAX_WBITS);
        if (ret != Z_OK) {
            result.errorMessage = std::string("inflateInit2 failed: ") + zError(ret);
            return result;
        }
        
        // Decompress
        ret = inflate(&stream, Z_FINISH);
        
        if (ret == Z_STREAM_END) {
            // Success
            result.success = true;
            result.decompressedSize = stream.total_out;
            result.compressionRatio = (double)result.decompressedSize / result.compressedSize;
            result.decompressedData.resize(result.decompressedSize);
        } else if (ret == Z_BUF_ERROR && stream.total_out > 0) {
            // Partial decompression - might need larger buffer
            result.errorMessage = "Buffer too small, got " + std::to_string(stream.total_out) + " bytes";
            result.decompressedSize = stream.total_out;
        } else {
            result.errorMessage = std::string("inflate failed: ") + zError(ret);
        }
        
        inflateEnd(&stream);
        
    } catch (const std::exception& e) {
        result.errorMessage = std::string("Exception: ") + e.what();
    }
    
    return result;
}

// Verify decompressed data integrity (basic checks)
struct DataIntegrityCheck {
    bool allZeros;
    bool allOnes;
    bool hasVariation;
    uint8_t minValue;
    uint8_t maxValue;
    double meanValue;
    
    void analyze(const std::vector<uint8_t>& data, size_t sampleSize = 10000) {
        if (data.empty()) {
            allZeros = allOnes = hasVariation = false;
            minValue = maxValue = 0;
            meanValue = 0.0;
            return;
        }
        
        minValue = 255;
        maxValue = 0;
        uint64_t sum = 0;
        
        size_t samplesToCheck = std::min(sampleSize, data.size());
        size_t step = data.size() / samplesToCheck;
        if (step == 0) step = 1;
        
        for (size_t i = 0; i < data.size(); i += step) {
            uint8_t val = data[i];
            minValue = std::min(minValue, val);
            maxValue = std::max(maxValue, val);
            sum += val;
        }
        
        size_t actualSamples = (data.size() + step - 1) / step;
        meanValue = (double)sum / actualSamples;
        
        allZeros = (maxValue == 0);
        allOnes = (minValue == 255 && maxValue == 255);
        hasVariation = (maxValue - minValue) > 10;
    }
    
    void print() const {
        std::cout << "      Data integrity:\n";
        std::cout << "        Min value: " << (int)minValue << "\n";
        std::cout << "        Max value: " << (int)maxValue << "\n";
        std::cout << "        Mean value: " << std::fixed << std::setprecision(2) << meanValue << "\n";
        
        if (allZeros) {
            std::cout << "        ⚠ WARNING: All pixels are zero (black)\n";
        } else if (allOnes) {
            std::cout << "        ⚠ WARNING: All pixels are 255 (white)\n";
        } else if (hasVariation) {
            std::cout << "        ✓ Data has variation (likely valid image)\n";
        } else {
            std::cout << "        ⚠ Data has low variation\n";
        }
    }
};

// Test decompression of selected tiles
void testUnciTileDecompression(
    const std::string& filepath,
    const std::vector<TileInfo>& tiles,
    const std::vector<size_t>& tileIndices = {}) {
    
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          UNCI Tile Decompression Test                      ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    std::cout << "File: " << filepath << "\n";
    std::cout << "Total tiles: " << tiles.size() << "\n\n";
    
    // Determine which tiles to test
    std::vector<size_t> indicesToTest;
    
    if (tileIndices.empty()) {
        // Default: test first 5, middle 3, and last 2 tiles
        size_t numTiles = tiles.size();
        
        // First 5
        for (size_t i = 0; i < std::min(size_t(5), numTiles); ++i) {
            indicesToTest.push_back(i);
        }
        
        // Middle 3
        if (numTiles > 10) {
            size_t mid = numTiles / 2;
            for (size_t i = mid - 1; i <= mid + 1 && i < numTiles; ++i) {
                indicesToTest.push_back(i);
            }
        }
        
        // Last 2
        if (numTiles > 7) {
            indicesToTest.push_back(numTiles - 2);
            indicesToTest.push_back(numTiles - 1);
        }
        
        // Remove duplicates and sort
        std::sort(indicesToTest.begin(), indicesToTest.end());
        indicesToTest.erase(std::unique(indicesToTest.begin(), indicesToTest.end()), 
                           indicesToTest.end());
    } else {
        indicesToTest = tileIndices;
    }
    
    std::cout << "Testing " << indicesToTest.size() << " tiles: [";
    for (size_t i = 0; i < indicesToTest.size(); ++i) {
        if (i > 0) std::cout << ", ";
        std::cout << indicesToTest[i];
    }
    std::cout << "]\n\n";
    
    // Statistics
    int successCount = 0;
    int failCount = 0;
    int skipCount = 0;
    uint64_t totalCompressed = 0;
    uint64_t totalDecompressed = 0;
    
    // Test each tile
    for (size_t idx : indicesToTest) {
        if (idx >= tiles.size()) {
            std::cout << "⚠ Skipping index " << idx << " (out of range)\n\n";
            continue;
        }
        
        const auto& tile = tiles[idx];
        
        std::cout << "─────────────────────────────────────────────────────────────\n";
        std::cout << "Testing Tile " << idx << " [row=" << tile.row << ", col=" << tile.col << "]:\n";
        std::cout << "  Dimensions: " << tile.width << "×" << tile.height << " pixels\n";
        std::cout << "  Position: (" << tile.xOffset << ", " << tile.yOffset << ")\n";
        std::cout << "  Pyramid level: " << tile.pyramidLevel << "\n";
        
        if (tile.byteOffset == 0) {
            std::cout << "  ⚠ Skipped: Empty tile\n\n";
            skipCount++;
            continue;
        }
        
        if (tile.byteOffset == 1) {
            std::cout << "  ⚠ Skipped: Reference tile\n\n";
            skipCount++;
            continue;
        }
        
        std::cout << "  File location: offset=" << tile.byteOffset 
                  << ", size=" << tile.byteCount << " bytes\n\n";
        
        // Attempt decompression
        auto result = decompressTile(filepath, tile);
        result.print();
        
        if (result.success) {
            successCount++;
            totalCompressed += result.compressedSize;
            totalDecompressed += result.decompressedSize;
            
            // Check data integrity
            DataIntegrityCheck integrity;
            integrity.analyze(result.decompressedData);
            integrity.print();
            
            // Show sample pixels (first 16 bytes)
            std::cout << "\n      Sample data (first 16 bytes):\n        ";
            for (size_t i = 0; i < std::min(size_t(16), result.decompressedData.size()); ++i) {
                printf("%02X ", result.decompressedData[i]);
            }
            std::cout << "\n";
        } else {
            failCount++;
        }
        
        std::cout << "\n";
    }
    
    // Summary
    std::cout << "╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          Decompression Test Summary                        ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    std::cout << "Total tiles tested: " << indicesToTest.size() << "\n";
    std::cout << "Successful: " << successCount << "\n";
    std::cout << "Failed: " << failCount << "\n";
    std::cout << "Skipped: " << skipCount << "\n\n";
    
    if (successCount > 0) {
        double avgCompression = (double)totalDecompressed / totalCompressed;
        std::cout << "Total compressed data: " << totalCompressed << " bytes\n";
        std::cout << "Total decompressed data: " << totalDecompressed << " bytes\n";
        std::cout << "Average compression ratio: " << std::fixed << std::setprecision(2) 
                  << avgCompression << ":1\n\n";
    }
    
    if (failCount > 0) {
        std::cout << "⚠ Some tiles failed to decompress. Check:\n";
        std::cout << "  - Compression format (deflate vs zlib vs gzip)\n";
        std::cout << "  - Byte offsets are correct\n";
        std::cout << "  - File corruption\n\n";
    }
    
    if (successCount == indicesToTest.size() - skipCount) {
        std::cout << "✓ All tested tiles decompressed successfully!\n";
    }
}

// Convenience function: test single tile by index
void testSingleUnciTile(const std::string& filepath, 
                        const std::vector<TileInfo>& tiles,
                        size_t tileIndex) {
    testUnciTileDecompression(filepath, tiles, {tileIndex});
}

// Convenience function: test tiles from specific pyramid level
void testUnciTilesByLevel(const std::string& filepath,
                          const std::vector<TileInfo>& tiles,
                          int pyramidLevel,
                          size_t maxTiles = 5) {
    std::vector<size_t> indices;
    
    for (size_t i = 0; i < tiles.size() && indices.size() < maxTiles; ++i) {
        if (tiles[i].pyramidLevel == pyramidLevel) {
            indices.push_back(i);
        }
    }
    
    if (indices.empty()) {
        std::cout << "No tiles found at pyramid level " << pyramidLevel << "\n";
        return;
    }
    
    std::cout << "Found " << indices.size() << " tiles at level " << pyramidLevel << "\n";
    testUnciTileDecompression(filepath, tiles, indices);
}

// Full workflow: extract and test decompression
void testUnciFileComplete(const std::string& filepath) {
    try {
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Complete UNCI File Decompression Test             ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        // Step 1: Parse metadata
        std::cout << "Step 1: Parsing file metadata...\n";
        auto initialData = readFileRange(filepath, 0, 8192);
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        if (!metaLoc.found) {
            std::cerr << "✗ Meta box not found\n";
            return;
        }
        
        auto metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
        MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
        
        // Step 2: Extract tiles
        std::cout << "\nStep 2: Extracting tile metadata...\n";
        auto tiles = extractUnciTiles(metaInfo);
        
        if (tiles.empty()) {
            std::cerr << "✗ No tiles extracted\n";
            return;
        }
        
        std::cout << "✓ Extracted " << tiles.size() << " tiles\n";
        
        // Step 3: Test decompression
        std::cout << "\nStep 3: Testing tile decompression...\n";
        testUnciTileDecompression(filepath, tiles);
        
    } catch (const std::exception& e) {
        std::cerr << "✗ Error: " << e.what() << "\n";
    }
}

#endif // TEST_UNCI_TILE_DECOMPRESSION_DEFINED

###### Tile extraction helper - tiles

In [ ]:
#ifndef UNCI_TILE_TEST_HELPERS_DEFINED
#define UNCI_TILE_TEST_HELPERS_DEFINED

#include <vector>
#include <string>
#include <map>
#include <optional>

// Structure to hold extracted tiles ready for testing
struct UnciTileTestData {
    std::string filepath;
    MetaBoxInfo metaInfo;
    std::vector<TileInfo> allTiles;
    std::map<int, std::vector<TileInfo>> tilesByLevel;  // pyramidLevel -> tiles
    
    // Statistics
    size_t totalTiles;
    int numLevels;
    bool isPyramid;
    
    // Full image dimensions (from level 0 or single level)
    uint32_t fullImageWidth;
    uint32_t fullImageHeight;
    uint32_t tileGridCols;
    uint32_t tileGridRows;
    
    UnciTileTestData() : totalTiles(0), numLevels(0), isPyramid(false),
                         fullImageWidth(0), fullImageHeight(0),
                         tileGridCols(0), tileGridRows(0) {}
    
    void printSummary() const {
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          UNCI Tile Test Data Summary                       ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        std::cout << "File: " << filepath << "\n";
        std::cout << "Total tiles: " << totalTiles << "\n";
        std::cout << "Pyramid levels: " << numLevels << "\n";
        std::cout << "Is pyramid: " << (isPyramid ? "Yes" : "No") << "\n\n";
        
        if (fullImageWidth > 0 && fullImageHeight > 0) {
            std::cout << "Full image dimensions: " << fullImageWidth << "×" << fullImageHeight << "\n";
            std::cout << "Tile grid: " << tileGridCols << "×" << tileGridRows << "\n\n";
        }
        
        if (isPyramid) {
            std::cout << "Pyramid structure:\n";
            for (const auto& [level, tiles] : tilesByLevel) {
                if (!tiles.empty()) {
                    uint32_t levelWidth = 0, levelHeight = 0;
                    
                    // Calculate level dimensions from tile positions
                    for (const auto& tile : tiles) {
                        levelWidth = std::max(levelWidth, tile.xOffset + tile.width);
                        levelHeight = std::max(levelHeight, tile.yOffset + tile.height);
                    }
                    
                    std::cout << "  Level " << level << ": " << tiles.size() << " tiles, "
                              << levelWidth << "×" << levelHeight << " pixels\n";
                }
            }
        }
    }
    
    // Get tiles for a specific level
    std::vector<TileInfo> getTilesForLevel(int level) const {
        auto it = tilesByLevel.find(level);
        if (it != tilesByLevel.end()) {
            return it->second;
        }
        return {};
    }
    
    // Get a specific tile by index
    std::optional<TileInfo> getTile(size_t index) const {
        if (index < allTiles.size()) {
            return allTiles[index];
        }
        return std::nullopt;
    }
    
    // Get tiles by indices
    std::vector<TileInfo> getTilesByIndices(const std::vector<size_t>& indices) const {
        std::vector<TileInfo> result;
        for (size_t idx : indices) {
            if (idx < allTiles.size()) {
                result.push_back(allTiles[idx]);
            }
        }
        return result;
    }
    
    // Get tiles in a specific region (bounding box)
    std::vector<TileInfo> getTilesInRegion(uint32_t x, uint32_t y, 
                                           uint32_t width, uint32_t height,
                                           int level = 0) const {
        std::vector<TileInfo> result;
        
        auto levelTiles = getTilesForLevel(level);
        for (const auto& tile : levelTiles) {
            // Check if tile intersects with region
            bool intersects = !(tile.xOffset >= x + width ||
                               tile.xOffset + tile.width <= x ||
                               tile.yOffset >= y + height ||
                               tile.yOffset + tile.height <= y);
            
            if (intersects) {
                result.push_back(tile);
            }
        }
        
        return result;
    }
    
    // Get tiles by row/column indices
    std::vector<TileInfo> getTilesByGrid(const std::vector<std::pair<uint32_t, uint32_t>>& positions,
                                         int level = 0) const {
        std::vector<TileInfo> result;
        
        auto levelTiles = getTilesForLevel(level);
        for (const auto& tile : levelTiles) {
            for (const auto& [row, col] : positions) {
                if (tile.row == row && tile.col == col) {
                    result.push_back(tile);
                    break;
                }
            }
        }
        
        return result;
    }
    
    // Get first N tiles
    std::vector<TileInfo> getFirstNTiles(size_t n, int level = -1) const {
        std::vector<TileInfo> result;
        
        if (level >= 0) {
            auto levelTiles = getTilesForLevel(level);
            size_t count = std::min(n, levelTiles.size());
            result.insert(result.end(), levelTiles.begin(), levelTiles.begin() + count);
        } else {
            size_t count = std::min(n, allTiles.size());
            result.insert(result.end(), allTiles.begin(), allTiles.begin() + count);
        }
        
        return result;
    }
    
    // Get random sample of tiles
    std::vector<TileInfo> getRandomSample(size_t n, int level = -1) const {
        std::vector<TileInfo> source = (level >= 0) ? getTilesForLevel(level) : allTiles;
        
        if (n >= source.size()) {
            return source;
        }
        
        std::vector<TileInfo> result;
        std::vector<size_t> indices;
        for (size_t i = 0; i < source.size(); ++i) {
            indices.push_back(i);
        }
        
        // Simple random sampling without replacement
        std::srand(std::time(nullptr));
        for (size_t i = 0; i < n; ++i) {
            size_t randomIndex = std::rand() % indices.size();
            result.push_back(source[indices[randomIndex]]);
            indices.erase(indices.begin() + randomIndex);
        }
        
        return result;
    }
};

// Main extraction function - handles both single-level and pyramid
UnciTileTestData extractUnciTilesForTesting(const std::string& filepath, bool verbose = true) {
    UnciTileTestData testData;
    testData.filepath = filepath;
    
    try {
        if (verbose) {
            std::cout << "╔════════════════════════════════════════════════════════════╗\n";
            std::cout << "║          Extracting UNCI Tiles for Testing                 ║\n";
            std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
            std::cout << "File: " << filepath << "\n\n";
        }
        
        // Step 1: Parse metadata
        if (verbose) std::cout << "Step 1: Parsing metadata...\n";
        auto initialData = readFileRange(filepath, 0, 8192);
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        if (!metaLoc.found) {
            throw std::runtime_error("Meta box not found");
        }
        
        auto metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
        testData.metaInfo = parseMetaBox(metaData, metaLoc.offset);
        
        if (verbose) {
            std::cout << "  ✓ Metadata parsed\n";
            std::cout << "    Items: " << testData.metaInfo.items.size() << "\n";
            std::cout << "    Property associations: " << testData.metaInfo.itemPropertyAssociations.size() << "\n\n";
        }
        
        // Step 2: Detect if this is a pyramid file
        if (verbose) std::cout << "Step 2: Detecting file structure...\n";
        testData.isPyramid = (testData.metaInfo.itemPropertyAssociations.size() > 1);
        
        if (testData.isPyramid) {
            if (verbose) std::cout << "  ✓ Detected pyramid structure\n\n";
            
            // Step 3a: Extract pyramid
            if (verbose) std::cout << "Step 3: Extracting pyramid tiles...\n";
            auto pyramid = extractUnciPyramidTiles(testData.metaInfo);
            
            testData.numLevels = pyramid.size();
            
            for (const auto& level : pyramid) {
                testData.tilesByLevel[level.level] = level.tiles;
                testData.allTiles.insert(testData.allTiles.end(), 
                                        level.tiles.begin(), 
                                        level.tiles.end());
                
                if (level.level == 0) {
                    testData.fullImageWidth = level.imageWidth;
                    testData.fullImageHeight = level.imageHeight;
                    testData.tileGridCols = level.numCols;
                    testData.tileGridRows = level.numRows;
                }
                
                if (verbose) {
                    std::cout << "  Level " << level.level << ": " << level.tiles.size() << " tiles, "
                              << level.imageWidth << "×" << level.imageHeight << " pixels\n";
                }
            }
            
        } else {
            if (verbose) std::cout << "  ✓ Detected single-level structure\n\n";
            
            // Step 3b: Extract single level
            if (verbose) std::cout << "Step 3: Extracting tiles...\n";
            testData.allTiles = extractUnciTiles(testData.metaInfo);
            testData.tilesByLevel[0] = testData.allTiles;
            testData.numLevels = 1;
            
            // Get dimensions from first tile's uncC
            if (!testData.allTiles.empty() && testData.allTiles[0].uncompressedConfig) {
                auto& uncC = testData.allTiles[0].uncompressedConfig;
                testData.tileGridCols = uncC->numTileColsMinus1 + 1;
                testData.tileGridRows = uncC->numTileRowsMinus1 + 1;
                
                // Calculate full image size from tiles
                for (const auto& tile : testData.allTiles) {
                    testData.fullImageWidth = std::max(testData.fullImageWidth, 
                                                       tile.xOffset + tile.width);
                    testData.fullImageHeight = std::max(testData.fullImageHeight, 
                                                        tile.yOffset + tile.height);
                }
            }
            
            if (verbose) {
                std::cout << "  ✓ Extracted " << testData.allTiles.size() << " tiles\n";
            }
        }
        
        testData.totalTiles = testData.allTiles.size();
        
        if (verbose) {
            std::cout << "\n✓ Extraction complete\n";
            testData.printSummary();
        }
        
    } catch (const std::exception& e) {
        std::cerr << "\n✗ Extraction failed: " << e.what() << "\n";
        throw;
    }
    
    return testData;
}

// Extract all tiles from file
std::vector<TileInfo> extractAllUnciTiles(const std::string& filepath, bool verbose = true) {
    auto testData = extractUnciTilesForTesting(filepath, verbose);
    return testData.allTiles;
}

// Extract tiles from specific pyramid level
std::vector<TileInfo> extractUnciTilesByLevel(const std::string& filepath, int level, bool verbose = true) {
    auto testData = extractUnciTilesForTesting(filepath, verbose);
    return testData.getTilesForLevel(level);
}

// Extract tiles by specific indices
std::vector<TileInfo> extractUnciTilesByIndices(const std::string& filepath, 
                                                 const std::vector<size_t>& indices,
                                                 bool verbose = true) {
    auto testData = extractUnciTilesForTesting(filepath, verbose);
    return testData.getTilesByIndices(indices);
}

// Extract tiles in a region
std::vector<TileInfo> extractUnciTilesInRegion(const std::string& filepath,
                                                uint32_t x, uint32_t y,
                                                uint32_t width, uint32_t height,
                                                int level = 0,
                                                bool verbose = true) {
    auto testData = extractUnciTilesForTesting(filepath, verbose);
    
    if (verbose) {
        std::cout << "\n→ Selecting tiles in region (" << x << "," << y << ") to ("
                  << (x + width) << "," << (y + height) << "), level " << level << "\n";
    }
    
    return testData.getTilesInRegion(x, y, width, height, level);
}

// Extract tiles by grid positions
std::vector<TileInfo> extractUnciTilesByGrid(const std::string& filepath,
                                              const std::vector<std::pair<uint32_t, uint32_t>>& positions,
                                              int level = 0,
                                              bool verbose = true) {
    auto testData = extractUnciTilesForTesting(filepath, verbose);
    
    if (verbose) {
        std::cout << "\n→ Selecting tiles at " << positions.size() << " grid positions, level " << level << "\n";
    }
    
    return testData.getTilesByGrid(positions, level);
}

// Extract first N tiles
std::vector<TileInfo> extractFirstUnciTiles(const std::string& filepath, 
                                             size_t n, 
                                             int level = -1,
                                             bool verbose = true) {
    auto testData = extractUnciTilesForTesting(filepath, verbose);
    
    if (verbose) {
        std::cout << "\n→ Selecting first " << n << " tiles";
        if (level >= 0) std::cout << " from level " << level;
        std::cout << "\n";
    }
    
    return testData.getFirstNTiles(n, level);
}

// Extract random sample of tiles
std::vector<TileInfo> extractRandomUnciTiles(const std::string& filepath,
                                              size_t n,
                                              int level = -1,
                                              bool verbose = true) {
    auto testData = extractUnciTilesForTesting(filepath, verbose);
    
    if (verbose) {
        std::cout << "\n→ Selecting random sample of " << n << " tiles";
        if (level >= 0) std::cout << " from level " << level;
        std::cout << "\n";
    }
    
    return testData.getRandomSample(n, level);
}

// Extract corner tiles (useful for testing edge cases)
std::vector<TileInfo> extractCornerUnciTiles(const std::string& filepath,
                                              int level = 0,
                                              bool verbose = true) {
    auto testData = extractUnciTilesForTesting(filepath, verbose);
    
    if (testData.tileGridCols == 0 || testData.tileGridRows == 0) {
        std::cerr << "Warning: Cannot determine grid dimensions\n";
        return {};
    }
    
    std::vector<std::pair<uint32_t, uint32_t>> corners = {
        {0, 0},                                                      // Top-left
        {0, testData.tileGridCols - 1},                             // Top-right
        {testData.tileGridRows - 1, 0},                             // Bottom-left
        {testData.tileGridRows - 1, testData.tileGridCols - 1}      // Bottom-right
    };
    
    if (verbose) {
        std::cout << "\n→ Selecting corner tiles at level " << level << "\n";
    }
    
    return testData.getTilesByGrid(corners, level);
}

#endif // UNCI_TILE_TEST_HELPERS_DEFINED

###### test decompression usages

In [ ]:
// Example 1: Complete workflow (parse + extract + test)
// testUnciFileComplete("benchmark_output/output_unci_512_deflate.heif");

// Example 2: Test specific tiles by index
// std::vector<size_t> indicesToTest = {0, 10, 20, 50, 100};
// testUnciTileDecompression(filepath, tiles, indicesToTest);

// Example 3: Test single tile
//testSingleUnciTile(filepath, tiles, 42);

// Example 4: Test tiles from specific pyramid level
// testUnciTilesByLevel(filepath, tiles, 0, 10); // Level 0, max 10 tiles

// Example 5: Custom test with extracted tiles
// auto tiles = extractUnciTiles(metaInfo);
// testUnciTileDecompression(filepath, tiles); // Tests default selection

###### Test usages with tiles

In [ ]:
// Example 1: Extract all tiles, then test
// auto tiles = extractAllUnciTiles("benchmark_output/output_unci_512_deflate.heif");
// testUnciTileDecompression("benchmark_output/output_unci_512_deflate.heif", tiles);

// Example 2: Extract and test specific level
// auto level0Tiles = extractUnciTilesByLevel(
//    "benchmark_output/output_tiled_pyramid_5levels_unci_256_deflate.heif", 0);
// testUnciTileDecompression(
//    "benchmark_output/output_tiled_pyramid_5levels_unci_256_deflate.heif", level0Tiles);

// Example 3: Extract and test specific indices
// auto selectedTiles = extractUnciTilesByIndices(
//    "benchmark_output/output_unci_512_deflate.heif", {0, 5, 10, 15, 20});
// testUnciTileDecompression("benchmark_output/output_unci_512_deflate.heif", selectedTiles);

// Example 4: Extract and test tiles in region
// auto regionTiles = extractUnciTilesInRegion(
//    "benchmark_output/output_unci_512_deflate.heif", 
//    0, 0, 1024, 1024, 0);
// testUnciTileDecompression("benchmark_output/output_unci_512_deflate.heif", regionTiles);

// Example 5: Extract and test by grid positions
// std::vector<std::pair<uint32_t, uint32_t>> gridPos = {{0, 0}, {0, 1}, {1, 0}, {1, 1}};
// auto gridTiles = extractUnciTilesByGrid(
//    "benchmark_output/output_unci_512_deflate.heif", gridPos);
// testUnciTileDecompression("benchmark_output/output_unci_512_deflate.heif", gridTiles);

// Example 6: Extract and test first N tiles
// auto firstTiles = extractFirstUnciTiles("benchmark_output/output_unci_512_deflate.heif", 10);
// testUnciTileDecompression("benchmark_output/output_unci_512_deflate.heif", firstTiles);

// Example 7: Extract and test random sample
// auto randomTiles = extractRandomUnciTiles("benchmark_output/output_unci_512_deflate.heif", 5);
// testUnciTileDecompression("benchmark_output/output_unci_512_deflate.heif", randomTiles);

// Example 8: Extract and test corner tiles
// auto cornerTiles = extractCornerUnciTiles("benchmark_output/output_unci_512_deflate.heif");
// testUnciTileDecompression("benchmark_output/output_unci_512_deflate.heif", cornerTiles);

// Example 9: Use test data structure for multiple operations
// std::string filepath = "benchmark_output/output_tiled_pyramid_5levels_unci_256_deflate.heif";
// auto testData = extractUnciTilesForTesting(filepath);

// Test each pyramid level separately
// for (int level = 0; level < testData.numLevels; ++level) {
//     std::cout << "\n=== Testing Level " << level << " ===\n";
//     auto levelTiles = testData.getTilesForLevel(level);
//     testUnciTileDecompression(filepath, levelTiles);
// }

// Example 10: Quiet extraction (no verbose output)
// auto quietTiles = extractAllUnciTiles("benchmark_output/output_unci_512_deflate.heif", false);
// testUnciTileDecompression("benchmark_output/output_unci_512_deflate.heif", quietTiles);

#### Test tili

In [ ]:
#ifndef TEST_META_PARSER_TILI_TILES_DEFINED
#define TEST_META_PARSER_TILI_TILES_DEFINED

// Test function for single-level tili extraction
void testTiliSingleLevelExtraction(const std::string& filepath) {
    try {
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Testing Tili Single-Level Extraction              ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        std::cout << "File: " << filepath << "\n\n";
        
        // Parse metadata
        std::cout << "Parsing metadata...\n";
        std::vector<uint8_t> initialData = readFileRange(filepath, 0, 8192);
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        if (!metaLoc.found) {
            std::cerr << "✗ Meta box not found\n";
            return;
        }
        
        std::cout << "Meta box at offset " << metaLoc.offset 
                  << ", size " << metaLoc.size << " bytes\n\n";
        
        std::vector<uint8_t> metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
        MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
        
        // Calculate mdat data start
        size_t mdatBoxStart = metaLoc.offset + metaLoc.size;
        auto mdatHeader = readFileRange(filepath, mdatBoxStart, 16);
        uint32_t boxSize = readBE32(mdatHeader.data());
        size_t headerSize = (boxSize == 1) ? 16 : 8;
        size_t mdatDataStart = mdatBoxStart + headerSize;
        
        std::cout << "mdat data starts at: " << mdatDataStart << "\n\n";
        
        // Extract tiles
        auto tiles = extractTiliTilesSingleLevel(metaInfo, filepath, mdatDataStart);
        
        if (tiles.empty()) {
            std::cout << "\n✗ No tiles extracted\n";
            return;
        }
        
        // Show results
        std::cout << "\n=== Extraction Results ===\n";
        std::cout << "Total tiles: " << tiles.size() << "\n\n";
        
        std::cout << "First 10 tiles:\n";
        for (size_t i = 0; i < std::min(size_t(10), tiles.size()); ++i) {
            const auto& t = tiles[i];
            std::cout << "  Tile " << i << " [" << t.row << "," << t.col << "]: "
                      << t.width << "×" << t.height << "px, ";
            
            if (t.byteOffset == 0) {
                std::cout << "EMPTY\n";
            } else if (t.byteOffset == 1) {
                std::cout << "REFERENCE to lower layer\n";
            } else {
                std::cout << t.byteCount << " bytes @ offset " << t.byteOffset << "\n";
            }
        }
        
        std::cout << "\n✓ Test completed successfully!\n";
        
    } catch (const std::exception& e) {
        std::cerr << "\n✗ Error: " << e.what() << "\n";
    }
}
    

#endif // TEST_META_PARSER_TILI_TILES_DEFINED

test

In [ ]:
#ifndef TEST_META_PARSER_TILI_PYRAMID_TILES_DEFINED
#define TEST_META_PARSER_TILI_PYRAMID_TILES_DEFINED

// Test function for pyramidal TILI extraction
void testTiliPyramidExtraction(const std::string& filepath) {
    try {
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Testing TILI Pyramid Extraction                   ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        std::cout << "File: " << filepath << "\n\n";
        
        // Parse metadata
        std::cout << "Parsing metadata...\n";
        std::vector<uint8_t> initialData = readFileRange(filepath, 0, 8192);
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        if (!metaLoc.found) {
            std::cerr << "✗ Meta box not found\n";
            return;
        }
        
        std::cout << "Meta box at offset " << metaLoc.offset 
                  << ", size " << metaLoc.size << " bytes\n\n";
        
        std::vector<uint8_t> metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
        MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
        
        // Extract pyramid
        auto pyramid = extractTiliPyramidComplete(metaInfo, filepath);
        
        if (pyramid.empty()) {
            std::cout << "\n✗ No pyramid levels extracted\n";
            return;
        }
        
        // Show sample tiles from each level
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Sample Tiles (first 5 per level)                  ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        for (const auto& level : pyramid) {
            std::cout << "Level " << level.level << " samples:\n";
            for (size_t i = 0; i < std::min(size_t(5), level.tiles.size()); ++i) {
                const auto& t = level.tiles[i];
                std::cout << "  Tile [" << t.row << "," << t.col << "]: ";
                
                if (t.byteOffset == 0) {
                    std::cout << "EMPTY\n";
                } else if (t.byteOffset == 1) {
                    std::cout << "REFERENCE to lower level\n";
                } else if (t.byteCount == 0) {
                    std::cout << "DUPLICATE @ offset " << t.byteOffset << "\n";
                } else {
                    std::cout << t.width << "×" << t.height << "px, "
                              << t.byteCount << " bytes @ " << t.byteOffset << "\n";
                }
            }
            std::cout << "\n";
        }
        
        std::cout << "✓ Test completed successfully!\n";
        
    } catch (const std::exception& e) {
        std::cerr << "\n✗ Error: " << e.what() << "\n";
    }
}

#endif // TEST_META_PARSER_TILI_PYRAMID_TILES_DEFINED

#### Tili Decompression Tests

##### codec detection

In [ ]:
#ifndef TILI_CODEC_DETECTION_DEFINED
#define TILI_CODEC_DETECTION_DEFINED

// Detect codec from actual tile data by reading magic bytes
std::string detectCodecFromTileData(const std::string& filepath, 
                                     uint64_t offset, 
                                     uint64_t size) {
    if (offset == 0 || size == 0) {
        return "unknown";
    }
    
    try {
        // Read first 32 bytes to check magic bytes
        size_t bytesToRead = safe_min(size_t(32), size);
        auto headerBytes = readFileRange(filepath, offset, bytesToRead);
        
        if (headerBytes.size() < 4) {
            return "unknown";
        }
        
        // Check for deflate/zlib header
        // Zlib header: 0x78 followed by compression level byte
        if (headerBytes[0] == 0x78) {
            // Common zlib headers:
            // 0x78 0x01 - no compression
            // 0x78 0x9C - default compression
            // 0x78 0xDA - best compression
            // 0x78 0x5E - fast compression
            if (headerBytes[1] == 0x01 || headerBytes[1] == 0x9C || 
                headerBytes[1] == 0xDA || headerBytes[1] == 0x5E) {
                return "deflate";
            }
        }
        
        // Check for AV1 OBU (Open Bitstream Unit) header
        // AV1 starts with OBU header which has specific bit patterns
        // OBU type is in bits 3-7 of first byte
        uint8_t obuType = (headerBytes[0] >> 3) & 0x0F;
        
        // Common AV1 OBU types:
        // 1 = OBU_SEQUENCE_HEADER
        // 2 = OBU_TEMPORAL_DELIMITER
        // 3 = OBU_FRAME_HEADER
        // 6 = OBU_FRAME
        if (obuType >= 1 && obuType <= 8) {
            // Additional validation: check OBU structure
            bool hasExtension = (headerBytes[0] & 0x04) != 0;
            bool hasSize = (headerBytes[0] & 0x02) != 0;
            
            // Valid AV1 should have reasonable structure
            if (hasSize || obuType == 1 || obuType == 2 || obuType == 6) {
                return "av1";
            }
        }
        
        // Check for HEVC NAL unit start code
        // HEVC uses NAL units starting with 0x00 0x00 0x00 0x01 or 0x00 0x00 0x01
        if (headerBytes.size() >= 4) {
            if (headerBytes[0] == 0x00 && headerBytes[1] == 0x00) {
                if ((headerBytes[2] == 0x00 && headerBytes[3] == 0x01) ||
                    headerBytes[2] == 0x01) {
                    return "hevc";
                }
            }
            
            // Also check for HEVC without start code (length-prefixed)
            // First 4 bytes would be NAL unit size
            uint32_t nalSize = readBE32(headerBytes.data());
            if (nalSize > 0 && nalSize < size && nalSize < 1000000) {
                // Check NAL unit type in next bytes
                if (headerBytes.size() >= 6) {
                    uint8_t nalType = (headerBytes[4] >> 1) & 0x3F;
                    // HEVC NAL types range from 0-40
                    if (nalType <= 40) {
                        return "hevc";
                    }
                }
            }
        }
        
        // Check for JPEG (starts with 0xFF 0xD8)
        if (headerBytes[0] == 0xFF && headerBytes[1] == 0xD8) {
            return "jpeg";
        }
        
        // Check for PNG (starts with 0x89 0x50 0x4E 0x47)
        if (headerBytes.size() >= 4 &&
            headerBytes[0] == 0x89 && headerBytes[1] == 0x50 &&
            headerBytes[2] == 0x4E && headerBytes[3] == 0x47) {
            return "png";
        }
        
    } catch (const std::exception& e) {
        std::cerr << "Warning: Could not read tile data for codec detection: " 
                  << e.what() << "\n";
    }
    
    return "unknown";
}

// Detect codec for all tiles and determine majority
std::string detectCodecForTiles(const std::string& filepath,
                                const std::vector<TileInfo>& tiles,
                                size_t samplesToCheck = 10) {
    if (tiles.empty()) {
        return "unknown";
    }
    
    std::map<std::string, int> codecCounts;
    
    // Sample tiles evenly across the dataset
    size_t step = safe_max(size_t(1), tiles.size() / samplesToCheck);
    size_t checked = 0;
    
    for (size_t i = 0; i < tiles.size() && checked < samplesToCheck; i += step) {
        if (tiles[i].byteOffset > 0 && tiles[i].byteCount > 0 && 
            tiles[i].byteOffset != 1) {  // Skip reference tiles
            
            std::string codec = detectCodecFromTileData(filepath, 
                                                       tiles[i].byteOffset, 
                                                       tiles[i].byteCount);
            codecCounts[codec]++;
            checked++;
        }
    }
    
    // Return the most common codec
    std::string mostCommon = "unknown";
    int maxCount = 0;
    
    for (const auto& [codec, count] : codecCounts) {
        if (count > maxCount) {
            maxCount = count;
            mostCommon = codec;
        }
    }
    
    return mostCommon;
}

// Apply detected codec to all tiles
void applyDetectedCodec(const std::string& filepath, std::vector<TileInfo>& tiles) {
    std::cout << "Detecting codec from tile data...\n";
    
    // Sample first valid tile
    std::string detectedCodec = "unknown";
    for (const auto& tile : tiles) {
        if (tile.byteOffset > 0 && tile.byteCount > 0 && tile.byteOffset != 1) {
            detectedCodec = detectCodecFromTileData(filepath, tile.byteOffset, tile.byteCount);
            if (detectedCodec != "unknown") {
                break;
            }
        }
    }
    
    std::cout << "  Detected codec: " << detectedCodec << "\n";
    
    // Apply to all tiles
    for (auto& tile : tiles) {
        tile.compression = detectedCodec;
    }
}

// Apply detected codec to pyramid levels
void applyDetectedCodecToPyramid(const std::string& filepath, 
                                 std::vector<TiliPyramidLevel>& pyramid) {
    std::cout << "Detecting codec for pyramid levels...\n";
    
    for (auto& level : pyramid) {
        if (level.tiles.empty()) continue;
        
        // Detect from first valid tile in this level
        std::string detectedCodec = "unknown";
        for (const auto& tile : level.tiles) {
            if (tile.byteOffset > 0 && tile.byteCount > 0 && tile.byteOffset != 1) {
                detectedCodec = detectCodecFromTileData(filepath, tile.byteOffset, tile.byteCount);
                if (detectedCodec != "unknown") {
                    break;
                }
            }
        }
        
        std::cout << "  Level " << level.level << ": " << detectedCodec << "\n";
        
        // Apply to all tiles in this level
        for (auto& tile : level.tiles) {
            tile.compression = detectedCodec;
        }
    }
}

#endif // TILI_CODEC_DETECTION_DEFINED

##### Functions

In [ ]:
#ifndef TEST_TILI_TILE_DECOMPRESSION_DEFINED
#define TEST_TILI_TILE_DECOMPRESSION_DEFINED

#include <vector>
#include <string>
#include <iostream>
#include <iomanip>
#include <map>
#include <optional>
#include <algorithm>  // Make sure this is included
#include <ctime>      // For std::time in random sampling

// Structure to hold TILI decompression results
struct TiliDecompressionResult {
    bool success;
    size_t compressedSize;
    size_t decompressedSize;
    size_t expectedSize;
    double compressionRatio;
    std::string errorMessage;
    std::vector<uint8_t> decompressedData;
    std::string codec;  // "av1", "hevc", "deflate", etc.
    
    TiliDecompressionResult() : success(false), compressedSize(0), 
                                decompressedSize(0), expectedSize(0), 
                                compressionRatio(0.0), codec("unknown") {}
    
    void print() const {
        if (success) {
            std::cout << "    ✓ Decompression successful\n";
            std::cout << "      Codec: " << codec << "\n";
            std::cout << "      Compressed size: " << compressedSize << " bytes\n";
            std::cout << "      Decompressed size: " << decompressedSize << " bytes\n";
            std::cout << "      Expected size: " << expectedSize << " bytes\n";
            std::cout << "      Compression ratio: " << std::fixed << std::setprecision(2) 
                      << compressionRatio << ":1\n";
            
            if (decompressedSize != expectedSize) {
                int64_t diff = (int64_t)decompressedSize - (int64_t)expectedSize;
                std::cout << "      ⚠ Size mismatch: " << (diff > 0 ? "+" : "") << diff << " bytes\n";
            }
        } else {
            std::cout << "    ✗ Decompression failed: " << errorMessage << "\n";
            if (!codec.empty() && codec != "unknown") {
                std::cout << "      Codec: " << codec << "\n";
            }
        }
    }
};

// Calculate expected tile size for TILI
size_t calculateExpectedTiliTileSize(const TileInfo& tile) {
    size_t bytesPerPixel = 3; // Default RGB
    
    if (tile.uncompressedConfig && !tile.uncompressedConfig->componentFormat.empty()) {
        bytesPerPixel = 0;
        for (uint8_t bits : tile.uncompressedConfig->componentFormat) {
            bytesPerPixel += (bits + 7) / 8;
        }
    } else if (!tile.bitsPerChannel.empty()) {
        bytesPerPixel = 0;
        for (uint8_t bits : tile.bitsPerChannel) {
            bytesPerPixel += (bits + 7) / 8;
        }
    }
    
    return static_cast<size_t>(tile.width) * tile.height * bytesPerPixel;
}

// Detect codec from tile compression field
std::string detectTiliCodec(const TileInfo& tile) {
    if (!tile.compression.empty()) {
        std::string codec = tile.compression;
        std::transform(codec.begin(), codec.end(), codec.begin(), ::tolower);
        return codec;
    }
    return "unknown";
}

// Decompress TILI tile using deflate/zlib
TiliDecompressionResult decompressTiliDeflate(const std::vector<uint8_t>& compressedData,
                                               size_t expectedSize) {
    TiliDecompressionResult result;
    result.compressedSize = compressedData.size();
    result.expectedSize = expectedSize;
    result.codec = "deflate";
    
    try {
        // Allocate decompression buffer
        result.decompressedData.resize(expectedSize);
        
        // Initialize zlib stream
        z_stream stream = {};
        stream.avail_in = compressedData.size();
        stream.next_in = const_cast<uint8_t*>(compressedData.data());
        stream.avail_out = result.decompressedData.size();
        stream.next_out = result.decompressedData.data();
        
        // Try raw deflate first (no zlib header)
        int ret = inflateInit2(&stream, -MAX_WBITS);
        if (ret != Z_OK) {
            result.errorMessage = std::string("inflateInit2 failed: ") + zError(ret);
            return result;
        }
        
        // Decompress
        ret = inflate(&stream, Z_FINISH);
        
        if (ret == Z_STREAM_END) {
            result.success = true;
            result.decompressedSize = stream.total_out;
            result.compressionRatio = (double)result.decompressedSize / result.compressedSize;
            result.decompressedData.resize(result.decompressedSize);
        } else if (ret == Z_BUF_ERROR && stream.total_out > 0) {
            result.errorMessage = "Buffer too small, got " + std::to_string(stream.total_out) + " bytes";
            result.decompressedSize = stream.total_out;
        } else {
            // Try with zlib header
            inflateEnd(&stream);
            
            stream = {};
            stream.avail_in = compressedData.size();
            stream.next_in = const_cast<uint8_t*>(compressedData.data());
            stream.avail_out = result.decompressedData.size();
            stream.next_out = result.decompressedData.data();
            
            ret = inflateInit(&stream);
            if (ret == Z_OK) {
                ret = inflate(&stream, Z_FINISH);
                if (ret == Z_STREAM_END) {
                    result.success = true;
                    result.decompressedSize = stream.total_out;
                    result.compressionRatio = (double)result.decompressedSize / result.compressedSize;
                    result.decompressedData.resize(result.decompressedSize);
                } else {
                    result.errorMessage = std::string("inflate with zlib header failed: ") + zError(ret);
                }
                inflateEnd(&stream);
            } else {
                result.errorMessage = std::string("inflateInit failed: ") + zError(ret);
            }
            return result;
        }
        
        inflateEnd(&stream);
        
    } catch (const std::exception& e) {
        result.errorMessage = std::string("Exception: ") + e.what();
    }
    
    return result;
}

// Stub for AV1 decompression (requires libaom or dav1d)
TiliDecompressionResult decompressTiliAV1(const std::vector<uint8_t>& compressedData,
                                          size_t expectedSize) {
    TiliDecompressionResult result;
    result.compressedSize = compressedData.size();
    result.expectedSize = expectedSize;
    result.codec = "av1";
    result.errorMessage = "AV1 decompression not implemented (requires libaom or dav1d)";
    
    // TODO: Implement AV1 decompression using libaom or dav1d
    // This would require linking against the appropriate codec library
    
    std::cout << "    ℹ AV1 codec detected but decompression not implemented\n";
    std::cout << "      To enable AV1 support, link against libaom or dav1d\n";
    
    return result;
}

// Stub for HEVC decompression (requires libde265 or libx265)
TiliDecompressionResult decompressTiliHEVC(const std::vector<uint8_t>& compressedData,
                                           size_t expectedSize) {
    TiliDecompressionResult result;
    result.compressedSize = compressedData.size();
    result.expectedSize = expectedSize;
    result.codec = "hevc";
    result.errorMessage = "HEVC decompression not implemented (requires libde265)";
    
    // TODO: Implement HEVC decompression using libde265
    
    std::cout << "    ℹ HEVC codec detected but decompression not implemented\n";
    std::cout << "      To enable HEVC support, link against libde265\n";
    
    return result;
}

// Main TILI tile decompression function


// Enhanced codec detection for TILI tiles
std::string detectTiliCodecFromFile(const std::string& filepath, const TileInfo& tile) {
    // First, check if compression is already set in tile metadata
    if (!tile.compression.empty() && tile.compression != "unknown") {
        std::string codec = tile.compression;
        std::transform(codec.begin(), codec.end(), codec.begin(), ::tolower);
        return codec;
    }
    
    // Try to detect from file data by reading magic bytes
    if (tile.byteOffset > 0 && tile.byteCount >= 4) {
        try {
            size_t bytesToRead = safe_min(size_t(16), tile.byteCount);
            auto headerBytes = readFileRange(filepath, tile.byteOffset, bytesToRead);
            
            if (headerBytes.size() >= 4) {
                // Check for deflate/zlib header
                // Zlib header: 0x78 0x9C (default compression) or 0x78 0x01 (no compression)
                if (headerBytes[0] == 0x78 && 
                    (headerBytes[1] == 0x9C || headerBytes[1] == 0x01 || 
                     headerBytes[1] == 0x5E || headerBytes[1] == 0xDA)) {
                    return "deflate";
                }
                
                // Check for AV1 OBU (Open Bitstream Unit) header
                // AV1 temporal delimiter OBU: starts with 0x0A (sequence header) or 0x12 (temporal delimiter)
                if ((headerBytes[0] & 0xF8) == 0x08 || (headerBytes[0] & 0xF8) == 0x10) {
                    return "av1";
                }
                
                // Check for HEVC NAL unit start code: 0x00 0x00 0x00 0x01 or 0x00 0x00 0x01
                if (headerBytes.size() >= 4 &&
                    headerBytes[0] == 0x00 && headerBytes[1] == 0x00 &&
                    (headerBytes[2] == 0x00 || headerBytes[2] == 0x01)) {
                    return "hevc";
                }
            }
        } catch (const std::exception& e) {
            std::cerr << "Warning: Could not read tile header for codec detection: " << e.what() << "\n";
        }
    }
    
    return "unknown";
}

// Diagnostic function to check TILI compression
void diagnoseTiliCompression(const std::string& filepath) {
    std::cout << "╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          TILI Compression Diagnostic                       ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    try {
        // Parse metadata
        auto initialData = readFileRange(filepath, 0, 8192);
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        auto metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
        MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
        
        // Look at tilC properties
        std::cout << "Property associations:\n";
        for (const auto& assoc : metaInfo.itemPropertyAssociations) {
            std::cout << "Item " << assoc.itemID << " properties:\n";
            for (const auto& prop : assoc.properties) {
                std::cout << "  - " << prop.propertyType;
                if (prop.tili) {
                    std::cout << " (tilC found)";
                }
                std::cout << "\n";
            }
        }
        
        // Extract tiles and check their compression field
        std::cout << "\nExtracting tiles...\n";
        size_t mdatDataStart = metaLoc.offset + metaLoc.size + 8;
        auto tiles = extractTiliTilesSingleLevel(metaInfo, filepath, mdatDataStart);
        
        std::cout << "\nFirst 5 tiles compression field:\n";
        size_t numToCheck = safe_min(size_t(5), tiles.size());
        for (size_t i = 0; i < numToCheck; ++i) {
            std::cout << "  Tile " << i << ": compression='" << tiles[i].compression << "'\n";
            
            // Read first 16 bytes
            if (tiles[i].byteOffset > 0 && tiles[i].byteCount > 0) {
                size_t bytesToRead = safe_min(size_t(16), tiles[i].byteCount);
                auto header = readFileRange(filepath, tiles[i].byteOffset, bytesToRead);
                std::cout << "    Header bytes: ";
                size_t bytesToShow = safe_min(size_t(8), header.size());
                for (size_t j = 0; j < bytesToShow; ++j) {
                    printf("%02X ", header[j]);
                }
                std::cout << "\n";
                
                // Detect codec from header
                std::string detectedCodec = detectTiliCodecFromFile(filepath, tiles[i]);
                std::cout << "    Detected from header: " << detectedCodec << "\n";
            }
        }
    } catch (const std::exception& e) {
        std::cerr << "Error: " << e.what() << "\n";
    }
}

// Updated decompression function with better codec detection
TiliDecompressionResult decompressTiliTile(const std::string& filepath, const TileInfo& tile) {
    TiliDecompressionResult result;
    result.expectedSize = calculateExpectedTiliTileSize(tile);
    
    try {
        // Validate tile has location data
        if (tile.byteOffset == 0 || tile.byteCount == 0) {
            result.errorMessage = "No location data for tile";
            return result;
        }
        
        if (tile.byteOffset == 1) {
            result.errorMessage = "Tile is a reference to lower resolution level";
            return result;
        }
        
        // Read compressed data
        std::vector<uint8_t> compressedData;
        try {
            compressedData = readFileRange(filepath, tile.byteOffset, tile.byteCount);
        } catch (const std::exception& e) {
            result.errorMessage = std::string("Failed to read tile data: ") + e.what();
            return result;
        }
        
        if (compressedData.empty()) {
            result.errorMessage = "No data read from file";
            return result;
        }
        
        // Detect codec with improved detection
        std::string codec = detectTiliCodecFromFile(filepath, tile);
        result.codec = codec;
        
        std::cout << "    Codec from tile metadata: '" << tile.compression << "'\n";
        std::cout << "    Detected codec from data: '" << codec << "'\n";
        
        // If still unknown, try to infer from data
        if (codec == "unknown" && compressedData.size() >= 2) {
            // Check for zlib/deflate header
            if (compressedData[0] == 0x78) {
                codec = "deflate";
                result.codec = "deflate";
                std::cout << "    → Inferred deflate from data header (0x78)\n";
            }
        }
        
        // Route to appropriate decompressor
        if (codec == "deflate" || codec == "zlib") {
            result = decompressTiliDeflate(compressedData, result.expectedSize);
        } else if (codec == "av1") {
            result = decompressTiliAV1(compressedData, result.expectedSize);
        } else if (codec == "hevc" || codec == "hev1" || codec == "hvc1") {
            result = decompressTiliHEVC(compressedData, result.expectedSize);
        } else if (codec == "none" || codec == "uncompressed") {
            // Uncompressed data
            result.decompressedData = compressedData;
            result.decompressedSize = compressedData.size();
            result.compressedSize = compressedData.size();
            result.compressionRatio = 1.0;
            result.success = true;
            result.codec = "none";
        } else {
            // Unknown codec - try deflate as fallback
            std::cout << "    ⚠ Unknown codec '" << codec << "', trying deflate as fallback...\n";
            result = decompressTiliDeflate(compressedData, result.expectedSize);
            if (!result.success) {
                result.errorMessage = "Unknown codec '" + codec + "' and deflate fallback failed";
            } else {
                result.codec = "deflate (fallback)";
            }
        }
        
    } catch (const std::exception& e) {
        result.errorMessage = std::string("Exception: ") + e.what();
    }
    
    return result;
}


// Test decompression of TILI tiles
void testTiliTileDecompression(const std::string& filepath,
                                const std::vector<TileInfo>& tiles,
                                const std::vector<size_t>& tileIndices = {}) {
    
    std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          TILI Tile Decompression Test                      ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    std::cout << "File: " << filepath << "\n";
    std::cout << "Total tiles: " << tiles.size() << "\n\n";
    
    // Determine which tiles to test
    std::vector<size_t> indicesToTest;
    
    if (tileIndices.empty()) {
        // Default: test first 5, middle 3, and last 2 tiles
        size_t numTiles = tiles.size();
        
        // First 5
        size_t firstCount = safe_min(size_t(5), numTiles);
        for (size_t i = 0; i < firstCount; ++i) {
            indicesToTest.push_back(i);
        }
        
        // Middle 3
        if (numTiles > 10) {
            size_t mid = numTiles / 2;
            for (size_t i = mid - 1; i <= mid + 1 && i < numTiles; ++i) {
                indicesToTest.push_back(i);
            }
        }
        
        // Last 2
        if (numTiles > 7) {
            indicesToTest.push_back(numTiles - 2);
            indicesToTest.push_back(numTiles - 1);
        }
        
        // Remove duplicates and sort
        std::sort(indicesToTest.begin(), indicesToTest.end());
        indicesToTest.erase(std::unique(indicesToTest.begin(), indicesToTest.end()), 
                           indicesToTest.end());
    } else {
        indicesToTest = tileIndices;
    }
    
    std::cout << "Testing " << indicesToTest.size() << " tiles: [";
    for (size_t i = 0; i < indicesToTest.size(); ++i) {
        if (i > 0) std::cout << ", ";
        std::cout << indicesToTest[i];
    }
    std::cout << "]\n\n";
    
    // Statistics
    int successCount = 0;
    int failCount = 0;
    int skipCount = 0;
    int notImplementedCount = 0;
    uint64_t totalCompressed = 0;
    uint64_t totalDecompressed = 0;
    std::map<std::string, int> codecCounts;
    
    // Test each tile
    for (size_t idx : indicesToTest) {
        if (idx >= tiles.size()) {
            std::cout << "⚠ Skipping index " << idx << " (out of range)\n\n";
            continue;
        }
        
        const auto& tile = tiles[idx];
        
        std::cout << "─────────────────────────────────────────────────────────────\n";
        std::cout << "Testing Tile " << idx << " [row=" << tile.row << ", col=" << tile.col << "]:\n";
        std::cout << "  Dimensions: " << tile.width << "×" << tile.height << " pixels\n";
        std::cout << "  Position: (" << tile.xOffset << ", " << tile.yOffset << ")\n";
        std::cout << "  Pyramid level: " << tile.pyramidLevel << "\n";
        std::cout << "  Compression: " << tile.compression << "\n";
        
        if (tile.byteOffset == 0) {
            std::cout << "  ⚠ Skipped: Empty tile\n\n";
            skipCount++;
            continue;
        }
        
        if (tile.byteOffset == 1) {
            std::cout << "  ⚠ Skipped: Reference tile\n\n";
            skipCount++;
            continue;
        }
        
        std::cout << "  File location: offset=" << tile.byteOffset 
                  << ", size=" << tile.byteCount << " bytes\n\n";
        
        // Attempt decompression
        auto result = decompressTiliTile(filepath, tile);
        result.print();
        
        codecCounts[result.codec]++;
        
        if (result.success) {
            successCount++;
            totalCompressed += result.compressedSize;
            totalDecompressed += result.decompressedSize;
            
            // Check data integrity
            DataIntegrityCheck integrity;
            integrity.analyze(result.decompressedData);
            integrity.print();
            
            // Show sample pixels (first 16 bytes)
            std::cout << "\n      Sample data (first 16 bytes):\n        ";
            size_t sampleSize = safe_min(size_t(16), result.decompressedData.size());
            for (size_t i = 0; i < sampleSize; ++i) {
                printf("%02X ", result.decompressedData[i]);
            }
            std::cout << "\n";
        } else {
            if (result.errorMessage.find("not implemented") != std::string::npos) {
                notImplementedCount++;
            } else {
                failCount++;
            }
        }
        
        std::cout << "\n";
    }
    
    // Summary
    std::cout << "╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          TILI Decompression Test Summary                   ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    std::cout << "Total tiles tested: " << indicesToTest.size() << "\n";
    std::cout << "Successful: " << successCount << "\n";
    std::cout << "Failed: " << failCount << "\n";
    std::cout << "Skipped: " << skipCount << "\n";
    std::cout << "Not implemented: " << notImplementedCount << "\n\n";
    
    if (!codecCounts.empty()) {
        std::cout << "Codecs encountered:\n";
        for (const auto& [codec, count] : codecCounts) {
            std::cout << "  " << codec << ": " << count << " tile(s)\n";
        }
        std::cout << "\n";
    }
    
    if (successCount > 0) {
        double avgCompression = (double)totalDecompressed / totalCompressed;
        std::cout << "Total compressed data: " << totalCompressed << " bytes\n";
        std::cout << "Total decompressed data: " << totalDecompressed << " bytes\n";
        std::cout << "Average compression ratio: " << std::fixed << std::setprecision(2) 
                  << avgCompression << ":1\n\n";
    }
    
    if (failCount > 0) {
        std::cout << "⚠ Some tiles failed to decompress. Check:\n";
        std::cout << "  - Compression format compatibility\n";
        std::cout << "  - Byte offsets are correct\n";
        std::cout << "  - File corruption\n\n";
    }
    
    if (notImplementedCount > 0) {
        std::cout << "ℹ Some tiles use codecs not yet implemented\n";
        std::cout << "  Consider using libheif API for full codec support\n\n";
    }
    
    if (successCount == indicesToTest.size() - skipCount - notImplementedCount) {
        std::cout << "✓ All testable tiles decompressed successfully!\n";
    }
}

// Convenience: test single TILI tile
void testSingleTiliTile(const std::string& filepath, 
                        const std::vector<TileInfo>& tiles,
                        size_t tileIndex) {
    testTiliTileDecompression(filepath, tiles, {tileIndex});
}

// Convenience: test TILI tiles by pyramid level
void testTiliTilesByLevel(const std::string& filepath,
                          const std::vector<TileInfo>& tiles,
                          int pyramidLevel,
                          size_t maxTiles = 5) {
    std::vector<size_t> indices;
    
    for (size_t i = 0; i < tiles.size() && indices.size() < maxTiles; ++i) {
        if (tiles[i].pyramidLevel == pyramidLevel) {
            indices.push_back(i);
        }
    }
    
    if (indices.empty()) {
        std::cout << "No tiles found at pyramid level " << pyramidLevel << "\n";
        return;
    }
    
    std::cout << "Found " << indices.size() << " tiles at level " << pyramidLevel << "\n";
    testTiliTileDecompression(filepath, tiles, indices);
}

// Complete workflow: extract and test TILI tiles
void testTiliFileComplete(const std::string& filepath) {
    try {
        std::cout << "╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Complete TILI File Decompression Test             ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
        
        // Step 1: Parse metadata
        std::cout << "Step 1: Parsing file metadata...\n";
        auto initialData = readFileRange(filepath, 0, 8192);
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
        
        if (!metaLoc.found) {
            std::cerr << "✗ Meta box not found\n";
            return;
        }
        
        auto metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
        MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
        
        // Step 2: Determine if single-level or pyramid
        std::cout << "\nStep 2: Detecting file structure...\n";
        
        // Check for pyramid by looking at tilC properties
        bool isPyramid = false;
        for (const auto& assoc : metaInfo.itemPropertyAssociations) {
            if (assoc.properties.size() > 1) {
                for (const auto& prop : assoc.properties) {
                    if (prop.propertyType == "tilC" || prop.tili) {
                        isPyramid = true;
                        break;
                    }
                }
            }
        }
        
        std::vector<TileInfo> tiles;
        
        if (isPyramid) {
            std::cout << "✓ Detected pyramid structure\n";
            
            // Step 3a: Extract pyramid tiles
            std::cout << "\nStep 3: Extracting pyramid tiles...\n";
            auto pyramid = extractTiliPyramidComplete(metaInfo, filepath);
            
            if (pyramid.empty()) {
                std::cerr << "✗ No pyramid levels extracted\n";
                return;
            }
            
            // Test each level
            for (const auto& level : pyramid) {
                std::cout << "\n═══════════════════════════════════════════════════════════\n";
                std::cout << "Testing Level " << level.level << " (" << level.tiles.size() << " tiles)\n";
                std::cout << "═══════════════════════════════════════════════════════════\n";
                
                testTiliTileDecompression(filepath, level.tiles);
            }
        } else {
            std::cout << "✓ Detected single-level structure\n";
            
            // Step 3b: Extract single level tiles
            std::cout << "\nStep 3: Extracting tiles...\n";
            
            // Calculate mdat data start
            size_t mdatBoxStart = metaLoc.offset + metaLoc.size;
            auto mdatHeader = readFileRange(filepath, mdatBoxStart, 16);
            uint32_t boxSize = readBE32(mdatHeader.data());
            size_t headerSize = (boxSize == 1) ? 16 : 8;
            size_t mdatDataStart = mdatBoxStart + headerSize;
            
            tiles = extractTiliTilesSingleLevel(metaInfo, filepath, mdatDataStart);
            
            if (tiles.empty()) {
                std::cerr << "✗ No tiles extracted\n";
                return;
            }
            
            std::cout << "✓ Extracted " << tiles.size() << " tiles\n";
            
            // Test tiles
            testTiliTileDecompression(filepath, tiles);
        }
        
        std::cout << "\n✓ Complete test finished\n";
        
    } catch (const std::exception& e) {
        std::cerr << "\n✗ Error: " << e.what() << "\n";
    }
}

#endif // TEST_TILI_TILE_DECOMPRESSION_DEFINED

##### Usage examples

In [ ]:
// First run diagnostic to see what's happening
// diagnoseTiliCompression("benchmark_output/output_tili_512_deflate.heif");

// Then run the full test
// testTiliFileComplete("benchmark_output/output_tili_512_deflate.heif");

// Example 1: Complete test (single-level TILI file)
// testTiliFileComplete("benchmark_output/output_tili_512_deflate.heif");

// Example 2: Complete test (pyramid TILI file)
// testTiliFileComplete("benchmark_output/output_tiled_pyramid_5levels_tili_256_deflate.heif");

// Example 3: Extract and test specific tiles
// {
//     std::string filepath = "benchmark_output/output_tili_512_deflate.heif";
    
//     // Extract tiles
//     auto initialData = readFileRange(filepath, 0, 8192);
//     auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
//     auto metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
//     MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);
    
//     size_t mdatDataStart = metaLoc.offset + metaLoc.size + 8;
//     auto tiles = extractTiliTilesSingleLevel(metaInfo, filepath, mdatDataStart);
    
//     // Test specific indices
//     std::vector<size_t> testIndices = {0, 5, 10, 15, 20};
//     testTiliTileDecompression(filepath, tiles, testIndices);
// }

// Example 4: Test single tile
// {
//     auto tiles = /* extract tiles */;
//     testSingleTiliTile(filepath, tiles, 0);
// }

// Example 5: Test tiles by level (pyramid)
// {
//     auto pyramid = extractTiliPyramidComplete(metaInfo, filepath);
    
//     for (const auto& level : pyramid) {
//         testTiliTilesByLevel(filepath, level.tiles, level.level, 10);
//     }
// }

// Example 6: Test with AV1 codec (will show not implemented message)
// testTiliFileComplete("srcdata/mandelbrot-b-262144-tili-pymd.avif");

### Example Usage

In [ ]:

// Example 1: Parse meta box directly (if you know offset and size)
// testMetaParser("sample.heif", SourceType::FILE, 24, 635);
//testMetaParser("benchmark_output/heif_NONE_t256_p3.heic", SourceType::FILE, 24, 635);

// Example 2: Parse meta box with string type
// testMetaParser("sample.heif", "file", 24, 635);

// Example 3: Full workflow - automatically find and parse meta box
// testFullHeifParsing("sample.heif", SourceType::FILE);
// testFullHeifParsing("benchmark_output/heif_DEFLATE_t256_p3.heic", SourceType::FILE);
//testFullHeifParsing("srcdata/ACT2017.heif", SourceType::FILE);

// Example 4: Full workflow with HTTP source
// testFullHeifParsing("https://example.com/sample.heif", "http");


// Example 5: Parse meta from HTTP with known offset
// testMetaParser("https://example.com/sample.heif", "https", 24, 635);

// Example 6: Parse all from file - tiled(256)/pyramidal (5) image with deflate compression and custom grid
//testFullHeifParsing("benchmark_output/output_tiled_pyramid_5levels_grid_256_deflate.heif", SourceType::FILE);

// Example 7: Parse all from file - tiled(256)/pyramidal (5) image with deflate compression and custom unci
// testFullHeifParsing("benchmark_output/output_tiled_pyramid_5levels_unci_256_deflate.heif", SourceType::FILE);

// Example 6: Parse all from file - tiled(256)/pyramidal (5) image with deflate compression and custom tili
// testFullHeifParsing("benchmark_output/output_tiled_pyramid_5levels_tili_256_deflate.heif", SourceType::FILE);

// Example 7: Extract all times from a complex file
// Test with the unci tiled pyramid file
//testTileExtraction("benchmark_output/output_tiled_pyramid_5levels_unci_256_deflate.heif", SourceType::FILE);
// Optionally visualize specific levels
// auto tiles = extractAllTiles(metaInfo); // after parsing
// printTileGridASCII(tiles, 0); // Level 0 (highest resolution)
// printTileGridASCII(tiles, 4); // Level 4 (lowest resolution)

// Example 8: Use libheif-specific extraction for unci multi-level tiles
// testLibheifUnciTiles("benchmark_output/output_tiled_pyramid_5levels_unci_256_deflate.heif", SourceType::FILE);

// Example 9: icef-specific extraction for unci multi-level tiles with offsets
// testCompleteExtractionWithIcef("benchmark_output/output_tiled_pyramid_5levels_unci_256_deflate.heif");

// Example 10: Extract grid tiles with offsets
// Test grid tile extraction
// testTileExtraction("benchmark_output/output_tiled_pyramid_5levels_grid_256_deflate.heif", SourceType::FILE);
// testGridTileExtractionWithOffsets("benchmark_output/output_tiled_pyramid_5levels_grid_256_deflate.heif");

// This version works! for non-pyramid, tiled with grid method
// testGridTileExtraction("benchmark_output/output_tiled_pyramid_5levels_grid_256_deflate.heif");

//testGridTileExtractionWithPyramid("benchmark_output/output_tiled_pyramid_5levels_grid_256_deflate.heif");

//diagnoseGridFile("benchmark_output/output_tiled_pyramid_5levels_grid_256_deflate.heif");
// diagnoseGridFile("benchmark_output/output_grid_512_deflate.heif");

// Example 11: Complete grid extraction with byte offsets for all tiles - NOT FIXED (used the wrong internal )
// testCompleteGridExtraction("benchmark_output/output_grid_512_deflate.heif");
// This version works!
// testGridTileExtraction("benchmark_output/output_grid_512_deflate.heif");

// Example 12: Complete grid extraction with byte offsets for all tiles in a pyramidal file - fixed, works!
// testGridPyramidExtraction("benchmark_output/output_tiled_pyramid_5levels_grid_256_deflate.heif");

// Example 13: Tile extraction for non-pyramidal, tiled HEIF with unci method
// testUnciTileExtraction("benchmark_output/output_unci_512_deflate.heif");
// auto tiles = extractAllUnciTiles("benchmark_output/output_unci_512_deflate.heif");
// std::vector<size_t> indicesToTest = {0, 10, 20, 50, 100};
// auto filepath = "benchmark_output/output_unci_512_deflate.heif";
// testUnciTileDecompression(filepath, tiles, indicesToTest);

// Example 14: Tile extraction for pyramidal HEIF with unci method
// testUnciPyramidExtraction("benchmark_output/output_tiled_pyramid_5levels_unci_256_deflate.heif");
// NEW: Test decompression of extracted tiles
// testUnciPyramidDecompression("benchmark_output/output_tiled_pyramid_5levels_unci_256_deflate.heif");

// Example 15: Tile extraction for non-pyramidal, tiled HEIF with tili method
// testTiliSingleLevelExtraction("benchmark_output/output_tili_512_deflate.heif");
// testTiliSingleLevelExtraction("benchmark_output/tili_geo.heif");

// Example 16: Tile extraction for pyramidal HEIF with tili method
// testTiliPyramidExtraction("benchmark_output/output_tiled_pyramid_5levels_tili_256_deflate.heif");
// testTiliPyramidExtraction("srcdata/mandelbrot-b-262144-tili-pymd.avif");
// testTiliTileExtraction("benchmark_output/output_tiled_pyramid_5levels_tili_256_deflate.heif");

// Test decompression of extracted TILI tiles (deflate)
{
    auto filepath = "benchmark_output/output_tiled_pyramid_5levels_tili_256_deflate.heif";
    // Extract tiles
    auto initialData = readFileRange(filepath, 0, 8192);
    auto [ftyp, metaLoc] = parseFtypAndLocateMeta(initialData, 0);
    auto metaData = readFileRange(filepath, metaLoc.offset, metaLoc.size);
    MetaBoxInfo metaInfo = parseMetaBox(metaData, metaLoc.offset);

    auto pyramid = extractTiliPyramidComplete(metaInfo, filepath);
    
    for (const auto& level : pyramid) {
        testTiliTilesByLevel(filepath, level.tiles, level.level, 10);
    }
}

## GeoHEIF Inspector

In [ ]:
#ifndef GEEOHEIF_INSPECTOR_H
#define GEEOHEIF_INSPECTOR_H
#include <iostream>
#include <fstream>
#include <vector>
#include <map>
#include <string>
#include <cstdint>
#include <algorithm>
#include <optional>

// Structure to hold bounding box information
struct BoundingBox {
    uint32_t minX;
    uint32_t minY;
    uint32_t maxX;
    uint32_t maxY;
    uint32_t width;
    uint32_t height;
    
    BoundingBox() : minX(UINT32_MAX), minY(UINT32_MAX), 
                    maxX(0), maxY(0), width(0), height(0) {}
    
    void expand(uint32_t x, uint32_t y, uint32_t w, uint32_t h) {
        minX = std::min(minX, x);
        minY = std::min(minY, y);
        maxX = std::max(maxX, x + w);
        maxY = std::max(maxY, y + h);
        width = maxX - minX;
        height = maxY - minY;
    }
    
    void print() const {
        std::cout << "Bounding Box: (" << minX << "," << minY << ") to (" 
                  << maxX << "," << maxY << "), size: " << width << "×" << height << "\n";
    }
};

// Structure for pyramid level information
struct PyramidLevelExtent {
    int level;
    uint32_t itemID;
    BoundingBox bbox;
    uint32_t tileWidth;
    uint32_t tileHeight;
    uint32_t numCols;
    uint32_t numRows;
    uint32_t totalTiles;
    std::string tilingScheme;  // "unci", "tili", "grid"
    
    // Padding information (if available from encoder)
    bool hasPaddingInfo;
    uint32_t unpaddedWidth;
    uint32_t unpaddedHeight;
    
    PyramidLevelExtent() : level(0), itemID(0), tileWidth(0), tileHeight(0),
                           numCols(0), numRows(0), totalTiles(0),
                           hasPaddingInfo(false), unpaddedWidth(0), unpaddedHeight(0) {}
    
    void print() const {
        std::cout << "\n━━━ Level " << level << " ━━━\n";
        std::cout << "  Item ID: " << itemID << "\n";
        std::cout << "  Tiling Scheme: " << tilingScheme << "\n";
        std::cout << "  Tile Grid: " << numCols << "×" << numRows << " = " << totalTiles << " tiles\n";
        std::cout << "  Tile Size: " << tileWidth << "×" << tileHeight << " pixels\n";
        std::cout << "  ";
        bbox.print();
        
        if (hasPaddingInfo) {
            std::cout << "  Unpadded Size: " << unpaddedWidth << "×" << unpaddedHeight << " pixels\n";
            uint32_t padX = bbox.width - unpaddedWidth;
            uint32_t padY = bbox.height - unpaddedHeight;
            if (padX > 0 || padY > 0) {
                std::cout << "  Padding: +" << padX << " pixels (X), +" << padY << " pixels (Y)\n";
            }
        }
    }
};

// GeoHEIF Inspector with Pyramid Extent Calculation
class GeoHEIFInspector {
private:
    std::string filepath;
    std::vector<uint8_t> fileData;
    MetaBoxInfo metaInfo;
    std::vector<PyramidLevelExtent> pyramidExtents;
    
    // Helper: Read big-endian uint32
    uint32_t readBE32(size_t offset) const {
        return (static_cast<uint32_t>(fileData[offset]) << 24) |
               (static_cast<uint32_t>(fileData[offset + 1]) << 16) |
               (static_cast<uint32_t>(fileData[offset + 2]) << 8) |
               static_cast<uint32_t>(fileData[offset + 3]);
    }
    
    // Helper: Read big-endian uint64
    uint64_t readBE64(size_t offset) const {
        return (static_cast<uint64_t>(fileData[offset]) << 56) |
               (static_cast<uint64_t>(fileData[offset + 1]) << 48) |
               (static_cast<uint64_t>(fileData[offset + 2]) << 40) |
               (static_cast<uint64_t>(fileData[offset + 3]) << 32) |
               (static_cast<uint64_t>(fileData[offset + 4]) << 24) |
               (static_cast<uint64_t>(fileData[offset + 5]) << 16) |
               (static_cast<uint64_t>(fileData[offset + 6]) << 8) |
               static_cast<uint64_t>(fileData[offset + 7]);
    }
    
public:
    GeoHEIFInspector(const std::string& path) : filepath(path) {}
    
    bool loadFile() {
        std::ifstream file(filepath, std::ios::binary | std::ios::ate);
        if (!file.is_open()) {
            std::cerr << "Failed to open file: " << filepath << "\n";
            return false;
        }
        
        size_t fileSize = file.tellg();
        file.seekg(0, std::ios::beg);
        
        fileData.resize(fileSize);
        file.read(reinterpret_cast<char*>(fileData.data()), fileSize);
        file.close();
        
        std::cout << "✓ Loaded file: " << fileSize << " bytes\n";
        return true;
    }
    
    bool parseMetadata() {
        // Parse ftyp and locate meta
        auto [ftyp, metaLoc] = parseFtypAndLocateMeta(fileData, 0);
        
        if (!metaLoc.found) {
            std::cerr << "✗ Meta box not found\n";
            return false;
        }
        
        std::cout << "✓ Meta box found at offset " << metaLoc.offset 
                  << ", size " << metaLoc.size << " bytes\n";
        
        // Parse meta box
        std::vector<uint8_t> metaData(fileData.begin() + metaLoc.offset,
                                       fileData.begin() + metaLoc.offset + metaLoc.size);
        metaInfo = parseMetaBox(metaData, metaLoc.offset);
        
        std::cout << "✓ Metadata parsed successfully\n";
        return true;
    }
    
    // Calculate extents for UNCI pyramid
    void calculateUnciPyramidExtents() {
        std::cout << "\n═══ Calculating UNCI Pyramid Extents ═══\n";
        
        auto pyramid = extractUnciPyramidTiles(metaInfo);
        
        for (const auto& level : pyramid) {
            PyramidLevelExtent extent;
            extent.level = level.level;
            extent.itemID = level.itemID;
            extent.tilingScheme = "unci";
            extent.tileWidth = level.tileWidth;
            extent.tileHeight = level.tileHeight;
            extent.numCols = level.numCols;
            extent.numRows = level.numRows;
            extent.totalTiles = level.tiles.size();
            
            // Calculate bounding box from actual tile positions
            for (const auto& tile : level.tiles) {
                extent.bbox.expand(tile.xOffset, tile.yOffset, tile.width, tile.height);
            }
            
            // Check for padding info from uncC or ispe
            if (level.uncC) {
                // Try to get unpadded dimensions from image spatial extents
                for (const auto& assoc : metaInfo.itemPropertyAssociations) {
                    if (assoc.itemID == level.itemID) {
                        for (const auto& prop : assoc.properties) {
                            if (prop.ispe) {
                                extent.hasPaddingInfo = true;
                                extent.unpaddedWidth = prop.ispe->width;
                                extent.unpaddedHeight = prop.ispe->height;
                                break;
                            }
                        }
                        break;
                    }
                }
            }
            
            pyramidExtents.push_back(extent);
        }
    }
    
    // Calculate extents for TILI pyramid
    void calculateTiliPyramidExtents() {
        std::cout << "\n═══ Calculating TILI Pyramid Extents ═══\n";
        
        auto pyramid = extractTiliPyramidComplete(metaInfo, filepath);
        
        for (const auto& level : pyramid) {
            PyramidLevelExtent extent;
            extent.level = level.level;
            extent.itemID = level.itemID;
            extent.tilingScheme = "tili";
            extent.tileWidth = level.tileWidth;
            extent.tileHeight = level.tileHeight;
            extent.numCols = level.numCols;
            extent.numRows = level.numRows;
            extent.totalTiles = level.tiles.size();
            
            // Calculate bounding box from actual tile positions
            for (const auto& tile : level.tiles) {
                if (tile.byteOffset > 1) {  // Skip empty and reference tiles
                    extent.bbox.expand(tile.xOffset, tile.yOffset, tile.width, tile.height);
                }
            }
            
            // Check for unpadded dimensions from ispe
            for (const auto& assoc : metaInfo.itemPropertyAssociations) {
                if (assoc.itemID == level.itemID) {
                    for (const auto& prop : assoc.properties) {
                        if (prop.ispe) {
                            extent.hasPaddingInfo = true;
                            extent.unpaddedWidth = prop.ispe->width;
                            extent.unpaddedHeight = prop.ispe->height;
                            break;
                        }
                    }
                    break;
                }
            }
            
            pyramidExtents.push_back(extent);
        }
    }
    
    // Calculate extents for GRID pyramid
    void calculateGridPyramidExtents() {
        std::cout << "\n═══ Calculating GRID Pyramid Extents ═══\n";
        
        auto levels = extractGridTilesWithPyramid(metaInfo);
        
        for (const auto& level : levels) {
            PyramidLevelExtent extent;
            extent.level = level.pyramidLevel;
            extent.itemID = level.itemID;
            extent.tilingScheme = "grid";
            extent.tileWidth = level.tileWidth;
            extent.tileHeight = level.tileHeight;
            extent.numCols = level.numCols;
            extent.numRows = level.numRows;
            extent.totalTiles = level.tiles.size();
            
            // Calculate bounding box from actual tile positions
            for (const auto& tile : level.tiles) {
                extent.bbox.expand(tile.xOffset, tile.yOffset, tile.width, tile.height);
            }
            
            // For GRID, the container item has the full image dimensions
            // Individual tiles have their actual dimensions
            extent.hasPaddingInfo = true;
            extent.unpaddedWidth = level.imageWidth;
            extent.unpaddedHeight = level.imageHeight;
            
            pyramidExtents.push_back(extent);
        }
    }
    
    // Auto-detect tiling scheme and calculate extents
    bool calculatePyramidExtents() {
        // Detect tiling scheme from items and properties
        std::string scheme = detectTilingScheme();
        
        std::cout << "\n✓ Detected tiling scheme: " << scheme << "\n";
        
        if (scheme == "unci") {
            calculateUnciPyramidExtents();
        } else if (scheme == "tili") {
            calculateTiliPyramidExtents();
        } else if (scheme == "grid") {
            calculateGridPyramidExtents();
        } else {
            std::cerr << "✗ Unknown or unsupported tiling scheme\n";
            return false;
        }
        
        return !pyramidExtents.empty();
    }
    
    // Detect tiling scheme from metadata
    std::string detectTilingScheme() const {
        // Check for grid (multiple items with dimg references)
        for (const auto& ref : metaInfo.itemReferences) {
            if (ref.referenceType == "dimg" && ref.toItemIDs.size() > 1) {
                return "grid";
            }
        }
        
        // Check for tili (tilC property)
        for (const auto& assoc : metaInfo.itemPropertyAssociations) {
            for (const auto& prop : assoc.properties) {
                if (prop.propertyType == "tilC" || prop.tili) {
                    return "tili";
                }
                if (prop.propertyType == "uncC" || prop.uncC) {
                    return "unci";
                }
            }
        }
        
        return "unknown";
    }
    
    void printPyramidExtents() const {
        std::cout << "\n╔════════════════════════════════════════════════════════════╗\n";
        std::cout << "║          Pyramid Level Extents Summary                     ║\n";
        std::cout << "╚════════════════════════════════════════════════════════════╝\n";
        
        std::cout << "\nTotal pyramid levels: " << pyramidExtents.size() << "\n";
        
        for (const auto& extent : pyramidExtents) {
            extent.print();
        }
        
        // Print scale factors between levels
        if (pyramidExtents.size() > 1) {
            std::cout << "\n━━━ Scale Factors ━━━\n";
            for (size_t i = 1; i < pyramidExtents.size(); ++i) {
                double scaleX = static_cast<double>(pyramidExtents[i-1].bbox.width) / 
                                pyramidExtents[i].bbox.width;
                double scaleY = static_cast<double>(pyramidExtents[i-1].bbox.height) / 
                                pyramidExtents[i].bbox.height;
                std::cout << "  Level " << (i-1) << " → Level " << i 
                          << ": " << std::fixed << std::setprecision(2) 
                          << scaleX << "× (X), " << scaleY << "× (Y)\n";
            }
        }
    }
    
    // Export extents to JSON for use in other tools
    std::string exportToJSON() const {
        std::ostringstream json;
        json << "{\n";
        json << "  \"file\": \"" << filepath << "\",\n";
        json << "  \"pyramid_levels\": [\n";
        
        for (size_t i = 0; i < pyramidExtents.size(); ++i) {
            const auto& extent = pyramidExtents[i];
            json << "    {\n";
            json << "      \"level\": " << extent.level << ",\n";
            json << "      \"item_id\": " << extent.itemID << ",\n";
            json << "      \"tiling_scheme\": \"" << extent.tilingScheme << "\",\n";
            json << "      \"bbox\": {\n";
            json << "        \"min_x\": " << extent.bbox.minX << ",\n";
            json << "        \"min_y\": " << extent.bbox.minY << ",\n";
            json << "        \"max_x\": " << extent.bbox.maxX << ",\n";
            json << "        \"max_y\": " << extent.bbox.maxY << ",\n";
            json << "        \"width\": " << extent.bbox.width << ",\n";
            json << "        \"height\": " << extent.bbox.height << "\n";
            json << "      },\n";
            json << "      \"tiling\": {\n";
            json << "        \"tile_width\": " << extent.tileWidth << ",\n";
            json << "        \"tile_height\": " << extent.tileHeight << ",\n";
            json << "        \"num_cols\": " << extent.numCols << ",\n";
            json << "        \"num_rows\": " << extent.numRows << ",\n";
            json << "        \"total_tiles\": " << extent.totalTiles << "\n";
            json << "      }";
            
            if (extent.hasPaddingInfo) {
                json << ",\n      \"unpadded\": {\n";
                json << "        \"width\": " << extent.unpaddedWidth << ",\n";
                json << "        \"height\": " << extent.unpaddedHeight << "\n";
                json << "      }";
            }
            
            json << "\n    }";
            if (i < pyramidExtents.size() - 1) json << ",";
            json << "\n";
        }
        
        json << "  ]\n";
        json << "}\n";
        
        return json.str();
    }
    
    // Get extent for specific level
    std::optional<PyramidLevelExtent> getLevelExtent(int level) const {
        for (const auto& extent : pyramidExtents) {
            if (extent.level == level) {
                return extent;
            }
        }
        return std::nullopt;
    }
};

// Usage example
void inspectGeoHEIFPyramid(const std::string& filepath) {
    std::cout << "╔════════════════════════════════════════════════════════════╗\n";
    std::cout << "║          GeoHEIF Pyramid Inspector                         ║\n";
    std::cout << "╚════════════════════════════════════════════════════════════╝\n\n";
    
    GeoHEIFInspector inspector(filepath);
    
    // Load and parse
    if (!inspector.loadFile()) return;
    if (!inspector.parseMetadata()) return;
    
    // Calculate pyramid extents
    if (!inspector.calculatePyramidExtents()) {
        std::cerr << "✗ Failed to calculate pyramid extents\n";
        return;
    }
    
    // Print results
    inspector.printPyramidExtents();
    
    // Export to JSON
    std::string json = inspector.exportToJSON();
    std::cout << "\n━━━ JSON Export ━━━\n" << json;
    
    // Optionally write to file
    std::ofstream jsonFile(filepath + ".extents.json");
    if (jsonFile.is_open()) {
        jsonFile << json;
        jsonFile.close();
        std::cout << "\n✓ Exported to: " << filepath << ".extents.json\n";
    }
}
#endif // GEEOHEIF_INSPECTOR_H

In [ ]:
{
    // Example usage
    std::string filepath = "output/output_heif_with_crs.heif";
    inspectGeoHEIFPyramid(filepath);
}